In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
UNIFIED MULTI-CITY PINN RUNNER
==============================
Modes:
  fit_check  → Only train_full_and_export (fast, ~5-15 min/city)
  full       → train_full_and_export + run_multi_window_eval (5-10 hrs/city)

Usage:
  python run_cities.py --mode fit_check
  python run_cities.py --mode fit_check --cities "Seattle,London,Rome"
  python run_cities.py --mode full --cities "Seattle,London"
  python run_cities.py --skip-us   # world cities only
  python run_cities.py --skip-world # US cities only
"""

import os, sys, time, math, warnings, random, argparse, copy
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from dataclasses import dataclass
from pathlib import Path
from scipy.signal import butter, filtfilt

# ======================== SEED & THREADS ========================
SEED = 1337; random.seed(SEED); np.random.seed(SEED)
N_THREADS = int(os.environ.get("PINN_NUM_THREADS", "4"))
os.environ.setdefault("OMP_NUM_THREADS", str(N_THREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(N_THREADS))
plt.rcParams.update({"figure.dpi": 300, "font.size": 12})

try:
    import statsmodels.api as sm
    from statsmodels.tools.sm_exceptions import ConvergenceWarning
    HAS_SM = True
except: HAS_SM = False

import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.set_num_threads(N_THREADS)

from sklearn.linear_model import LinearRegression, BayesianRidge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

# ======================== PATHS ========================
BASE_PATH_STR = os.environ.get("PINN_DATA_PATH",
    r"C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\CODES_04032021")
PATH = Path(BASE_PATH_STR)
OUT_PATH_STR = os.environ.get("PINN_OUT_PATH", BASE_PATH_STR)

# ======================== CITY CONFIGS ========================
US_CITIES = [
    # (label, county_name, state_abbr, subregion_1)
    ("SanDiego",  "San Diego County",  "CA", "California"),
    ("Seattle",   "King County",       "WA", "Washington"),
    ("NewYork",   "New York County",   "NY", "New York"),
    ("Chicago",   "Cook County",       "IL", "Illinois"),
    ("Houston",   "Harris County",     "TX", "Texas"),
    ("Phoenix",   "Maricopa County",   "AZ", "Arizona"),
    ("Miami",     "Miami-Dade County", "FL", "Florida"),
    ("Denver",    "Denver County",     "CO", "Colorado"),
    ("LosAngeles",  "Los Angeles County",  "CA", "California"),
    ("SanFrancisco",  "San Francisco County",  "CA", "California"),
]

CITY_SPECS = {
    "London":   {"key": "GB_ENG"},
    "SaoPaulo": {"country": "Brazil",    "match": {"subregion1_name": "São Paulo"}, "prefer_agg": True,
                 "fallback_keys": ["BR_SP", "BR_SP_SAO"]},
    "Rome":     {"key": "IT_62"},
    "Paris":    {"country": "France",    "match": {"subregion2_name": "Paris"},           "prefer_agg": True},
    "Tokyo":    {"country": "Japan",     "match": {"subregion1_name": "Tokyo"},           "prefer_agg": True},
    "Sydney":   {"country": "Australia", "match": {"subregion1_name": "New South Wales"}, "prefer_agg": True},
    "Berlin":   {"country": "Germany",   "match": {"subregion1_name": "Berlin"},          "prefer_agg": True,
                 "fallback_keys": ["DE_BE"]},
    "Moscow":   {"country": "Russia",    "match": {"subregion1_name": "Moscow"},          "prefer_agg": True,
                 "fallback_keys": ["RU_MOW", "RU_MOS"]},
}
CITY_POP = {
    "London": 9e6, "Paris": 2.16e6, "SaoPaulo": 12.3e6, "Tokyo": 14e6,
    "Sydney": 5.3e6, "Berlin": 3.7e6, "Moscow": 12.6e6, "Rome": 2.87e6,
}
WORLD_CITIES = list(CITY_SPECS.keys())

# ======================== HYPERPARAMETERS ========================
CASES_ONLY = False
EPOCHS_FULL = 3500; LR_FULL = 3e-3; SD_LAG_DAYS = 7
EPOCHS_MAX = 7000; VALIDATION_DAYS = 14; PATIENCE_EPOCHS = 500; VAL_CHECK_FREQ = 25

W_CUM_CASES=2.0; W_WEEK_CASES=8.0; W_PHYS=8.0; W_IC_ANCHOR=1.0
W_POP=5.0; W_PRIOR=1.0; W_FSmooth=0.04; W_FOURIER_L2=1e-3; W_F_VAR=1e-3; W_S0=5.0
PR_DUR_INCUB=5.0; PR_DUR_INF=7.0; PR_DUR_WARD=10.0; PR_DUR_ICU=14.0
PR_W_INCUB=2.0; PR_W_INF=3.0; PR_W_WARD=3.0; PR_W_ICU=4.0

BETA0_BOUNDS=(0.05,0.50); SIGMA_BOUNDS=(1/7,1/2); DELTA_BOUNDS=(1/9,1/6)
ZETA_BOUNDS=(1/14,1/3); EPSI_BOUNDS=(1/21,1/5)
M_BOUNDS=(0.60,0.99); C_BOUNDS=(0.20,0.60); F_BOUNDS=(0.35,0.65)
OMEGA_BOUNDS=(1/180,1/30); ETA_BOUNDS=(1/365,1/60)
ALPH_BOUNDS=(0.00,0.003); KSD=1.5; RHO_BOUNDS=(0.10,1.0)

F_TIME_MODE="fourier"; FOURIER_K=4
SD_FUTURE_MODE="arima"; SD_FUTURE_PARAM={"arima_min_len": 30}

# Publication validation: 5 distinct regimes, 1 cut each, 2 horizons = 10 evals per city
# Each tuple: (regime_name, cut_date, lookback_days)
#   FirstWave     — early exponential growth, sparse data
#   Winter20_Peak — largest pre-vaccine wave
#   Delta_Peak    — VOC emergence + partial immunity
#   Omicron_Shock — massive regime shift, biggest stress test
#   BA5_ImmuneEsc — late pandemic, waning immunity + escape
VALIDATION_CONFIG = [
    ("FirstWave",  "2020-05-01",  60),   # early exponential, sparse data
    ("Winter20",   "2021-01-15", 120),   # largest pre-vaccine wave (optimized from 180)
    ("Delta",      "2021-08-15", 300),   # variant displacement — long history helps (optimized from 180)
    ("Omicron",    "2022-01-15", 150),   # immune-escape shock (optimized from 180)
    ("BA5_Waning", "2022-06-15", 150),   # waning immunity + subvariant (optimized from 180)
]
HORIZON_LIST=[7,14]; EPOCHS_TINY=2500; SD_LAG_GRID=[3,7]
ARIMA_W_LO=0.25; ARIMA_W_HI=0.65

VARIANT_EVENTS = [
    ("Delta","2021-06-15",1.6,10), ("Omicron","2021-12-15",3.50,5),
    ("BA.5","2022-06-15",1.30,10),  ("XBB","2022-12-15",1.20,10),
]
VAR_BUMP_BOUNDS=(1.00,8.00); VAR_PRIOR_STRENGTH=1.5

# ======================== UTILITY FUNCTIONS ========================
def butter_lowpass(x, cutoff=0.14, order=4):
    b, a = butter(order, cutoff, btype="low"); return filtfilt(b, a, x)

def weekly_avg_np(x, k=7):
    x = np.asarray(x, float)
    return pd.Series(x).rolling(k, min_periods=1).mean().to_numpy() if x.size else x

def daily_from_cum_np(cum):
    cum = np.asarray(cum, float)
    return np.clip(np.diff(np.r_[cum[0], cum]), 0, None) if cum.size else cum

def make_quarter_ids(dates_like):
    q = pd.PeriodIndex(pd.to_datetime(dates_like), freq="Q")
    uq = pd.unique(q); mapd = {str(p):i for i,p in enumerate(uq)}
    return np.array([mapd[str(p)] for p in q], dtype=int), len(uq)

def safe_mape(y_true, y_pred, floor=1.0):
    denom = np.maximum(np.abs(np.asarray(y_true,float)), floor)
    return float(np.mean(np.abs(np.asarray(y_true,float) - np.asarray(y_pred,float)) / denom)) * 100.0

def _bounded(raw, lb, ub): return lb + (ub - lb) * torch.sigmoid(raw)
def _soft_prior(x, target, width): return ((x - target)/width)**2

def lag_tensor(x, lag):
    if lag <= 0: return x
    return torch.cat([x[:1].repeat(lag,1), x[:-lag]], dim=0)

def get_cut_idx_full(df, cut_date):
    idxs = np.where(pd.to_datetime(df["date"].values) <= pd.Timestamp(cut_date))[0]
    if idxs.size == 0: raise ValueError(f"cut_date precedes data")
    return int(idxs[-1])

# ======================== DATA LOADING: US ========================
def load_us_county_series(county_name, state_abbr, subregion_1):
    pop = pd.read_csv(PATH/"covid_county_population_usafacts.csv")
    row = pop[(pop["County Name"]==county_name) & (pop["State"]==state_abbr)]
    if row.empty: raise ValueError(f"County not found: {county_name}, {state_abbr}")
    fips = int(row.iloc[0]["countyFIPS"]); Npop = float(row.iloc[0]["population"])

    cwide = pd.read_csv(PATH/"covid_confirmed_usafacts.csv")
    crow = cwide[cwide["countyFIPS"]==fips].iloc[0].drop(["countyFIPS","County Name","State","StateFIPS"])
    dfc = pd.DataFrame({"date": pd.to_datetime(crow.index), "cases": crow.values.astype(float)})

    dwide = pd.read_csv(PATH/"covid_deaths_usafacts.csv")
    drow = dwide[dwide["countyFIPS"]==fips].iloc[0].drop(["countyFIPS","County Name","State","StateFIPS"])
    dfd = pd.DataFrame({"date": pd.to_datetime(drow.index), "deaths": drow.values.astype(float)})

    mob_files = [PATH/f for f in ["2020_US_Region_Mobility_Report.csv",
                 "2021_US_Region_Mobility_Report.csv","2022_US_Region_Mobility_Report.csv"]
                 if (PATH/f).exists()]
    if not mob_files:
        df_sd = pd.DataFrame({"date": dfc["date"].copy(), "sd": 0.0})
    else:
        mm = pd.concat([pd.read_csv(f) for f in mob_files], ignore_index=True)
        mm = mm[(mm["sub_region_1"]==subregion_1) & (mm["sub_region_2"]==county_name)].copy()
        mm["date"] = pd.to_datetime(mm["date"])
        cols = ["retail_and_recreation_percent_change_from_baseline",
                "grocery_and_pharmacy_percent_change_from_baseline",
                "parks_percent_change_from_baseline",
                "transit_stations_percent_change_from_baseline",
                "workplaces_percent_change_from_baseline",
                "residential_percent_change_from_baseline"]
        mm[cols] = mm[cols].fillna(0.0)
        sd_raw = -mm[cols].mean(axis=1).to_numpy()
        sd_smooth = butter_lowpass(sd_raw, cutoff=0.14) / 100.0
        df_sd = pd.DataFrame({"date": mm["date"].values, "sd": sd_smooth})

    df = dfc.merge(dfd, on="date").merge(df_sd, on="date", how="left").sort_values("date").reset_index(drop=True)
    df["sd"] = df["sd"].ffill().fillna(0.0).clip(0.0, 1.0)
    df["cases"] = np.maximum.accumulate(df["cases"].values)
    df["deaths"] = np.maximum.accumulate(df["deaths"].values)
    return df, Npop

# ======================== DATA LOADING: WORLD ========================
def _get_key_col(df):
    for c in ["location_key", "key"]:
        if c in df.columns: return c
    raise KeyError("No key column found")

def _ensure_google_data():
    import urllib.request
    BASE = "https://storage.googleapis.com/covid19-open-data/v3"
    for f in ["index.csv", "epidemiology.csv", "mobility.csv"]:
        p = PATH / f
        if not p.exists():
            print(f"  Downloading {f}...")
            urllib.request.urlretrieve(f"{BASE}/{f}", p)

def _resolve_key(spec):
    if "key" in spec: return spec["key"]
    idx = pd.read_csv(PATH/"index.csv"); KEY = _get_key_col(idx)
    df = idx.copy()
    for c in ["country_name","subregion1_name","subregion2_name"]:
        if c in df.columns: df[c] = df[c].astype(str)
    if "country" in spec and "country_name" in df.columns:
        df = df[df["country_name"].str.lower()==spec["country"].lower()]
    for col, val in spec.get("match", {}).items():
        if col in df.columns: df = df[df[col].str.lower()==str(val).lower()]
    if df.empty:
        # Try fallback keys before giving up
        for fk in spec.get("fallback_keys", []):
            check = idx[idx[KEY]==fk]
            if not check.empty:
                print(f"    Using fallback key: {fk}")
                return fk
        raise ValueError(f"No match for {spec} (tried fallbacks too)")
    if spec.get("prefer_agg") and "aggregation_level" in df.columns:
        df["aggregation_level"] = pd.to_numeric(df["aggregation_level"], errors="coerce")
        df = df.sort_values("aggregation_level", ascending=False)
    return df.iloc[0][KEY]

def load_world_city_series(city_name, mob_col="mobility_workplaces"):
    _ensure_google_data()
    spec = CITY_SPECS[city_name]
    loc_key = _resolve_key(spec)
    print(f"    Resolved key: {loc_key}")

    epi = pd.read_csv(PATH/"epidemiology.csv", parse_dates=["date"])
    KEY = _get_key_col(epi); epi_filt = epi[epi[KEY]==loc_key].sort_values("date")

    # If primary key yields no data, try fallback keys
    if epi_filt.empty:
        for fk in spec.get("fallback_keys", []):
            epi_filt = epi[epi[KEY]==fk].sort_values("date")
            if not epi_filt.empty:
                print(f"    Primary key empty, using fallback: {fk}"); loc_key = fk; break
    if epi_filt.empty:
        raise ValueError(f"No epidemiology data for {city_name} (key={loc_key})")

    epi = epi_filt
    cases = epi["cumulative_confirmed"].astype(float).to_numpy() if "cumulative_confirmed" in epi.columns \
        else epi.get("new_confirmed",0).fillna(0).astype(float).cumsum().to_numpy()
    deaths = epi["cumulative_deceased"].astype(float).to_numpy() if "cumulative_deceased" in epi.columns \
        else epi.get("new_deceased",0).fillna(0).astype(float).cumsum().to_numpy()
    dfe = pd.DataFrame({"date": epi["date"].values, "cases": cases, "deaths": deaths})

    mob = pd.read_csv(PATH/"mobility.csv", parse_dates=["date"])
    KEY_m = _get_key_col(mob); mob = mob[mob[KEY_m]==loc_key].sort_values("date")
    col = None
    for try_col in [mob_col, f"mobility_{mob_col}", mob_col.replace("mobility_","")]:
        if try_col in mob.columns: col = try_col; break
    if col is None:
        dfm = pd.DataFrame({"date": dfe["date"].copy(), "sd": 0.0})
    else:
        x = mob[col].astype(float).ffill().fillna(0.0).to_numpy()
        dfm = pd.DataFrame({"date": mob["date"].values, "sd": np.clip(-x/100, 0, 1)})

    df = dfe.merge(dfm, on="date", how="inner").sort_values("date").reset_index(drop=True)
    if df.empty:
        # Fall back to left join if inner yields nothing (mobility may not overlap)
        df = dfe.copy(); df["sd"] = 0.0
    df["sd"] = df["sd"].ffill().fillna(0.0).clip(0.0, 1.0)

    # Enforce monotonicity + clip end-of-series outlier spikes
    # (fixes London-like data correction artifacts)
    cases_raw = df["cases"].values.astype(float)
    cases_mono = np.maximum.accumulate(np.nan_to_num(cases_raw, 0))
    daily = np.diff(np.r_[cases_mono[0], cases_mono])
    if daily.size > 14:
        med_daily = np.median(daily[-90:]) if daily.size >= 90 else np.median(daily)
        outlier_thresh = max(med_daily * 10, 500)
        for i in range(len(daily)-1, max(0, len(daily)-14), -1):
            if daily[i] > outlier_thresh:
                print(f"    Clipping outlier spike at end: day {i}, daily={daily[i]:.0f} > {outlier_thresh:.0f}")
                daily[i] = med_daily
        cases_mono = cases_mono[0] + np.cumsum(daily)
    df["cases"] = cases_mono
    df["deaths"] = np.maximum.accumulate(np.nan_to_num(df["deaths"].values.astype(float), 0))

    print(f"    Loaded {city_name}: {len(df)} days, {df['date'].iloc[0].date()} → {df['date'].iloc[-1].date()}")
    return df, float(CITY_POP[city_name])
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
run_cities_model.py — PINN model + training + multi-window eval + main()
Import this after run_cities.py (or paste below it in a single file).
"""

# ======================== VARIANT HELPERS ========================
def _logistic(x, k=4.0): return 1.0/(1.0+np.exp(-k*x))

def _event_step_series(dates_np, event_date_str, ramp_days):
    dates = pd.to_datetime(dates_np); t0 = pd.Timestamp(event_date_str)
    x = (dates - t0).days.to_numpy(dtype=float) / max(1.0, float(ramp_days))
    return _logistic(x, k=4.0).reshape(-1,1).astype(float)

def make_variant_structs(dates_np, events):
    T = len(dates_np); V = np.ones((T,1), float); steps = []
    for name, d0, m_prior, ramp in events:
        step = _event_step_series(dates_np, d0, ramp)
        V *= (1.0 + (m_prior - 1.0) * step); steps.append(step)
    return V, steps

# ======================== ARIMA HELPERS ========================
def _small_auto_arima_safe(y):
    if not HAS_SM: return None
    best = None
    for p in (0,1):
        for d in (0,1):
            for q in (0,1):
                try:
                    with warnings.catch_warnings():
                        warnings.filterwarnings("ignore")
                        m = sm.tsa.SARIMAX(y, order=(p,d,q), enforce_stationarity=False, enforce_invertibility=False)
                        r = m.fit(disp=False, maxiter=50, method="lbfgs")
                        aic = getattr(r, "aic", np.inf)
                        if best is None or aic < best[0]: best = (aic, r)
                except: pass
    return best[1] if best else None

def forecast_daily_levels_from_cum(obs_cum, steps, recalib_window_days=180):
    y = weekly_avg_np(daily_from_cum_np(obs_cum).astype(float))
    y_fit = y[-recalib_window_days:]; last = float(y[-1]) if y.size else 0.0
    if y_fit.size < 7 or np.nanvar(y_fit) < 1e-9 or not HAS_SM:
        return np.full(steps, last, float)
    model = _small_auto_arima_safe(y_fit)
    if model is None: return np.full(steps, last, float)
    try:
        pm = np.asarray(model.get_forecast(steps=steps).predicted_mean, float).reshape(-1)
        if pm.shape[0] < steps: pm = np.pad(pm, (0, steps-pm.shape[0]), mode="edge")
    except: pm = np.full(steps, last, float)
    pm = np.clip(pm, 0.0, None); pm[~np.isfinite(pm)] = last
    return pm[:steps]

def forecast_sd_future_arima(sd_hist, steps, min_len=30):
    if not HAS_SM or steps<=0 or sd_hist is None or len(sd_hist)<max(5,min_len):
        last = float(sd_hist[-1]) if sd_hist is not None and len(sd_hist) else 0.0
        return np.full(steps, last, float)
    y = np.asarray(sd_hist, float)
    if not np.all(np.isfinite(y)) or np.nanstd(y)<1e-6:
        return np.full(steps, float(y[-1]), float)
    model = _small_auto_arima_safe(y)
    if model is None: return np.full(steps, float(y[-1]), float)
    try:
        fut = np.asarray(model.get_forecast(steps=steps).predicted_mean, float).reshape(-1)
        if fut.shape[0]<steps: fut = np.pad(fut, (0, steps-fut.shape[0]), mode="edge")
    except: fut = np.full(steps, float(y[-1]), float)
    fut = np.clip(fut, 0.0, 1.0)
    if not np.all(np.isfinite(fut)): fut[~np.isfinite(fut)] = float(y[-1])
    return fut

def project_sd_future(sd_history, steps, mode=SD_FUTURE_MODE, param=SD_FUTURE_PARAM):
    if steps <= 0: return np.array([], dtype=float)
    sd_history = np.asarray(sd_history, float)
    if mode == "arima":
        return np.clip(forecast_sd_future_arima(sd_history, steps, int(param.get("arima_min_len",30))), 0, 1)
    return np.full(steps, float(sd_history[-1]) if len(sd_history) else 0.0, float)

# ======================== BAYESIAN ALPHA ========================
def behavior_signal_from_deaths(dates, deaths_cum, k=7):
    dw = weekly_avg_np(daily_from_cum_np(deaths_cum))
    d_dw = np.diff(np.r_[dw[:1], dw])
    s = (d_dw - np.median(d_dw)) / (np.median(np.abs(d_dw - np.median(d_dw))) + 1e-9)
    return np.clip(s, -4.0, 4.0)

def bayesian_alpha_series(dates_hist, deaths_cum, alpha_min=0.0, alpha_max=0.5, kappa=0.12, lam=0.995):
    sig = behavior_signal_from_deaths(dates_hist, deaths_cum)
    a=2.0; b=2.0; s_hist = np.zeros_like(sig, float)
    for t, z in enumerate(sig):
        a, b = lam*a, lam*b
        if z >= 0: a += kappa*float(z)
        else: b += kappa*float(-z)
        a = max(a, 0.1); b = max(b, 0.1); s_hist[t] = a/(a+b)
    return alpha_min + (alpha_max - alpha_min) * s_hist

def project_alpha_future(alpha_hist, steps, mode="exp_decay", tau=7, floor=0.05):
    if steps <= 0: return np.array([], float)
    last = float(alpha_hist[-1]) if len(alpha_hist) else floor
    base = max(floor, np.quantile(alpha_hist[-21:], 0.25) if len(alpha_hist)>=21 else floor)
    t = np.arange(1, steps+1, dtype=float)
    return np.clip(base + (last - base)*np.exp(-t/max(tau,1.0)), 0, 1)

# ======================== REGIME GATING ========================
def gating_from_regime(rt_hist, sd_hist):
    if rt_hist.size < 14: return (ARIMA_W_LO+ARIMA_W_HI)/2
    rt14 = rt_hist[-14:]; rt_cv = np.std(rt14)/(np.mean(rt14)+1e-9)
    stab = np.clip(1.0 - rt_cv/0.5, 0, 1)
    sd_sig = np.clip(sd_hist[-1]/0.5, 0, 1) if sd_hist.size >= 7 else 0.0
    w = np.clip(0.7*stab + 0.3*sd_sig, 0, 1)
    return ARIMA_W_LO + (ARIMA_W_HI - ARIMA_W_LO)*w

def carry_wavg_future_from_cum(obs_cum_hist, fut_cum, k=7):
    obs_cum_hist = np.asarray(obs_cum_hist, float); fut_cum = np.asarray(fut_cum, float)
    hist_daily = np.clip(np.diff(np.r_[obs_cum_hist[0], obs_cum_hist]), 0, None) if obs_cum_hist.size else np.array([])
    if fut_cum.size == 0: return np.array([])
    last = obs_cum_hist[-1] if obs_cum_hist.size else 0.0
    fut_daily = np.clip(np.diff(np.r_[last, fut_cum]), 0, None)
    carry = hist_daily[-(k-1):] if hist_daily.size >= (k-1) else hist_daily
    combo = np.r_[carry, fut_daily]; wk = weekly_avg_np(combo, k=k)
    return wk[len(carry):][:len(fut_cum)]

def forecast_loglinear_7d(series_7d, current_date, history_days=21, horizon=14):
    train = series_7d.loc[current_date-pd.Timedelta(days=history_days):current_date]
    if len(train) < 7: return np.full(horizon, float(train.iloc[-1]) if len(train) else 0.0, float)
    y = train.values.astype(float); X = np.arange(len(y)).reshape(-1,1)
    m = LinearRegression().fit(X, np.log(y+1.0))
    return np.clip(np.exp(m.predict(np.arange(len(y),len(y)+horizon).reshape(-1,1)))-1, 0, None)

def forecast_bayesian_poly_7d(series_7d, current_date, history_days=21, horizon=14, degree=2):
    train = series_7d.loc[current_date-pd.Timedelta(days=history_days):current_date]
    if len(train) < 7: return np.full(horizon, float(train.iloc[-1]) if len(train) else 0.0, float)
    y = train.values.astype(float); X = np.arange(len(y)).reshape(-1,1)
    m = make_pipeline(PolynomialFeatures(degree=degree,include_bias=False),
        BayesianRidge(alpha_1=1e-6,alpha_2=1e-6,lambda_1=1e-6,lambda_2=1e-6,fit_intercept=True)).fit(X, np.log(y+1.0))
    return np.clip(np.exp(m.predict(np.arange(len(y),len(y)+horizon).reshape(-1,1)))-1, 0, None)

# ======================== NN MODULES ========================
class SUE_MLP(nn.Module):
    def __init__(self, hidden=128, depth=5):
        super().__init__()
        dims = [1]+[hidden]*(depth-1)+[7]; layers=[]
        for k in range(len(dims)-2): layers += [nn.Linear(dims[k],dims[k+1]), nn.Tanh()]
        layers += [nn.Linear(dims[-2],dims[-1])]
        self.net = nn.Sequential(*layers); self.soft = nn.Softplus(beta=2.0)
    def forward(self, x):
        y = self.soft(self.net(x)); S = torch.relu(1.0 - y.sum(1, keepdim=True))
        return torch.cat([S, y], dim=1)

class FTFourier(nn.Module):
    def __init__(self, k):
        super().__init__()
        self.bias = nn.Parameter(torch.zeros(1, device=DEVICE))
        self.a = nn.Parameter(torch.zeros(k, device=DEVICE))
        self.b = nn.Parameter(torch.zeros(k, device=DEVICE)); self.k = k
    def forward(self, ts):
        t = ts.view(-1); g = self.bias.expand_as(t)
        for k in range(1, self.k+1):
            g = g + self.a[k-1]*torch.sin(2*math.pi*k*t) + self.b[k-1]*torch.cos(2*math.pi*k*t)
        return _bounded(g.view(-1,1), F_BOUNDS[0], F_BOUNDS[1])

# ======================== PREPARE TENSORS ========================
def _prepare_series(df, sd_lag, device=DEVICE):
    n = len(df); t = torch.arange(n, dtype=torch.float32, device=device).unsqueeze(1)
    t_scale = max(1.0, float(t[-1])); ts = (t/t_scale).requires_grad_(True)
    sd = torch.tensor(df["sd"].values, dtype=torch.float32, device=device).unsqueeze(1)
    sd_l = lag_tensor(sd, sd_lag).clamp(0,1)
    Ccum = torch.tensor(df["cases"].values, dtype=torch.float32, device=device).unsqueeze(1)
    Dcum = torch.tensor(df["deaths"].values, dtype=torch.float32, device=device).unsqueeze(1)
    return ts, t_scale, sd_l, Ccum, Dcum

@dataclass
class TrainCfg:
    max_epochs: int; lr: float; sd_lag: int
    rollout_extra: int = 0; validation_days: int = 14; patience_epochs: int = 500

# ======================== CORE TRAINING ========================
def train_sueihcdr_once(dsub, Npop, cfg, return_all=False, sd_future=None):
    dates = pd.to_datetime(dsub["date"]).to_numpy()
    q_np, n_q = make_quarter_ids(dates)
    q_ids_full = torch.tensor(q_np, dtype=torch.long, device=DEVICE)
    ts_full, t_scale, sd_l_full, Ccum_full, Dcum_full = _prepare_series(dsub, cfg.sd_lag)
    Npop_t = torch.tensor([float(Npop)], dtype=torch.float32, device=DEVICE)
    net = SUE_MLP(128, 5).to(DEVICE)

    # Alpha
    Dcum_np = Dcum_full.squeeze().detach().cpu().numpy()
    alpha_np = bayesian_alpha_series(dates, Dcum_np, ALPH_BOUNDS[0], ALPH_BOUNDS[1], 0.08, 0.995)
    alpha_t_full = torch.tensor(alpha_np, dtype=torch.float32, device=DEVICE).unsqueeze(1)
    s_hist_full = ((alpha_t_full - ALPH_BOUNDS[0])/(ALPH_BOUNDS[1]-ALPH_BOUNDS[0]+1e-9)).clamp(0,1)

    # Variants
    V_prior_np, steps_np = make_variant_structs(dates, VARIANT_EVENTS)
    V_prior_full = torch.tensor(V_prior_np, dtype=torch.float32, device=DEVICE)
    steps_full = [torch.tensor(s, dtype=torch.float32, device=DEVICE) for s in steps_np]
    var_params = nn.ParameterList([nn.Parameter(torch.zeros(1, device=DEVICE)) for _ in VARIANT_EVENTS])

    params = {k: nn.Parameter(torch.tensor([0.0], device=DEVICE))
              for k in ["beta0","sigma","delta","zeta","epsi","m","c","omega","eta","rho"]}
    fmod = FTFourier(FOURIER_K).to(DEVICE)
    adam_params = list(net.parameters()) + list(params.values()) + list(fmod.parameters()) + list(var_params)
    opt = optim.Adam(adam_params, lr=cfg.lr)

    def unpack(ts_s, sd_s, q_s, s_s, V_s, steps_s):
        b0 = _bounded(params["beta0"],BETA0_BOUNDS[0],BETA0_BOUNDS[1])
        sg = _bounded(params["sigma"],SIGMA_BOUNDS[0],SIGMA_BOUNDS[1])
        de = _bounded(params["delta"],DELTA_BOUNDS[0],DELTA_BOUNDS[1])
        ze = _bounded(params["zeta"],ZETA_BOUNDS[0],ZETA_BOUNDS[1])
        ep = _bounded(params["epsi"],EPSI_BOUNDS[0],EPSI_BOUNDS[1])
        m  = _bounded(params["m"],M_BOUNDS[0],M_BOUNDS[1])
        c  = _bounded(params["c"],C_BOUNDS[0],C_BOUNDS[1])
        om = _bounded(params["omega"],OMEGA_BOUNDS[0],OMEGA_BOUNDS[1])
        et = _bounded(params["eta"],ETA_BOUNDS[0],ETA_BOUNDS[1])
        rho= _bounded(params["rho"],RHO_BOUNDS[0],RHO_BOUNDS[1])
        om_t = om * (1.0 - s_s)
        f_vec = fmod(ts_s)
        l2 = (fmod.a**2).sum() + (fmod.b**2).sum() + 0.01*(fmod.bias**2)
        df_d = torch.diff(f_vec.view(-1), prepend=f_vec.view(-1)[:1])
        f_reg = W_FOURIER_L2*l2 + W_F_VAR*torch.mean(df_d**2)
        lo, hi = VAR_BUMP_BOUNDS; V_ex = torch.ones_like(V_s)
        for (nm,_,mp,_), pr, st in zip(VARIANT_EVENTS, var_params, steps_s):
            ml = lo + (hi-lo)*torch.sigmoid(pr); V_ex = V_ex*(1.0 + (ml/max(1e-6,mp) - 1.0)*st)
        return b0,sg,de,ze,ep,m,c,om,et,om_t,f_vec,f_reg,V_s*V_ex,rho

    n = len(dsub); run_val = n >= cfg.validation_days + 21
    vi = n - cfg.validation_days if run_val else n

    ts_tr, sd_tr = ts_full[:vi], sd_l_full[:vi]
    Cc_tr, Dc_tr = Ccum_full[:vi], Dcum_full[:vi]
    s_tr, q_tr = s_hist_full[:vi], q_ids_full[:vi]
    V_tr = V_prior_full[:vi]; st_tr = [s[:vi] for s in steps_full]
    al_tr = alpha_t_full[:vi]

    ts_va, sd_va = ts_full[vi:], sd_l_full[vi:]
    Cc_va = Ccum_full[vi:]
    s_va, q_va = s_hist_full[vi:], q_ids_full[vi:]
    V_va = V_prior_full[vi:]; st_va = [s[vi:] for s in steps_full]

    best_vl = np.inf; best_ep = 0; pat = 0; best_sd = None; history = []

    for ep_i in range(1, cfg.max_epochs+1):
        net.train(); opt.zero_grad()
        y = net(ts_tr); S,U,E,I,H,C,D,R = [y[:,k:k+1] for k in range(8)]
        ones = torch.ones_like(ts_tr); sc = 1.0/t_scale
        dS,dU,dE,dI,dH,dC,dD,dR = [torch.autograd.grad(c, ts_tr, ones, create_graph=True)[0]*sc
                                     for c in [S,U,E,I,H,C,D,R]]
        b0,sg,de,ze,ep_p,m,c,om,et,om_t,fv,fr,Vt,rho = unpack(ts_tr,sd_tr,q_tr,s_tr,V_tr,st_tr)
        be = b0*Vt*torch.exp(-KSD*sd_tr); inf = be*S*I
        if ep_i <= 500: de = de.detach() + 0*de

        rS = dS + inf + al_tr*S - om_t*U - et*R
        rU = dU - al_tr*S + om_t*U
        rE = dE - inf + sg*E
        rI = dI - sg*E + de*I
        rH = dH - (1-m)*de*I + ze*H
        rC = dC - c*ze*H + ep_p*C
        rD = dD - fv*ep_p*C
        rR = dR - m*de*I - (1-c)*ze*H - (1-fv)*ep_p*C + et*R
        L_phys = sum(torch.mean(r**2) for r in [rS,rU,rE,rI,rH,rC,rD,rR])

        S_,U_,E_,I_,H_,C_,D_,R_ = [v*Npop_t for v in (S,U,E,I,H,C,D,R)]
        Cpd = rho*sg*E_; Cc_m = Cc_tr[:1] + torch.cumsum(Cpd, dim=0)

        def dp(cum):
            d = torch.clamp(cum[1:]-cum[:-1], min=0.0); return torch.cat([cum[:1]*0, d], dim=0)
        def wa(x, k=7):
            xp = torch.cat([x[:1].repeat(k-1,1), x], dim=0)
            w = torch.ones(1,1,k,device=x.device)/k
            return F.conv1d(xp.T.unsqueeze(0), w).squeeze().unsqueeze(1)

        Cw_d = wa(dp(Cc_tr)); Cw_m = wa(Cpd)
        L_wk = W_WEEK_CASES*F.smooth_l1_loss(torch.log1p(Cw_m), torch.log1p(Cw_d), beta=0.25)
        L_cm = W_CUM_CASES*F.mse_loss(torch.log1p(Cc_m), torch.log1p(Cc_tr))
        L_ic = W_IC_ANCHOR*F.mse_loss(torch.log1p(Cc_m[:1]), torch.log1p(Cc_tr[:1]))
        S0t = torch.clamp(1-Cc_tr[:1]/(Npop_t+1e-9), 0, 1)
        L_s0 = W_S0*F.mse_loss(S[:1], S0t)
        L_pop = W_POP*torch.mean(((S+U+E+I+H+C+D+R)-1)**2)
        L_pr = W_PRIOR*(0.05*_soft_prior(1/(sg+1e-9),PR_DUR_INCUB,PR_W_INCUB) +
                        0.05*_soft_prior(1/(de+1e-9),PR_DUR_INF,PR_W_INF) +
                        0.05*_soft_prior(1/(ze+1e-9),PR_DUR_WARD,PR_W_WARD) +
                        0.05*_soft_prior(1/(ep_p+1e-9),PR_DUR_ICU,PR_W_ICU)).squeeze()
        Rt_e = (b0*Vt*torch.exp(-KSD*sd_tr))/(de+1e-9) * S  # effective Rt includes S
        L_rt = 0.5*torch.mean(F.relu(Rt_e-4)**2 + F.relu(0.6-Rt_e)**2)
        # Soft S-floor: gently penalize S dropping below 0.3 in first year (~365 days)
        n_floor = min(365, S.shape[0])
        L_sfloor = 5.0*torch.mean(F.relu(0.25 - S[:n_floor]))
        L_end = 10.0*F.mse_loss(torch.log1p(Cc_m[-1:]), torch.log1p(Cc_tr[-1:]))
        rho_pr = 0.01*((rho-0.4)**2).mean()
        bp_loss = 0.0
        if VAR_PRIOR_STRENGTH > 0:
            lo,hi = VAR_BUMP_BOUNDS
            for (_,_,mp,_),pr in zip(VARIANT_EVENTS, var_params):
                ml = lo+(hi-lo)*torch.sigmoid(pr)
                bp_loss += (torch.log(ml+1e-9)-math.log(mp+1e-9))**2
            bp_loss *= VAR_PRIOR_STRENGTH
        wp = 0.0 if ep_i <= 300 else W_PHYS
        loss = L_wk+L_cm+L_end+wp*L_phys+L_ic+L_pr+fr.squeeze()+L_s0+L_pop+L_rt+bp_loss+rho_pr+L_sfloor
        loss.backward(); opt.step()

        # Validation
        if run_val and ep_i % VAL_CHECK_FREQ == 0:
            net.eval()
            with torch.no_grad():
                yv = net(ts_va); Ev = yv[:,2:3]  # E is index 2 (S=0,U=1,E=2,I=3,...)
                sv = _bounded(params["sigma"],SIGMA_BOUNDS[0],SIGMA_BOUNDS[1])
                rv = _bounded(params["rho"],RHO_BOUNDS[0],RHO_BOUNDS[1])
                Cpd_v = rv*sv*(Ev*Npop_t)
                # Anchor validation cumulative to the END of the training period
                Cpd_t = (rho*sg*(E*Npop_t)).detach()
                Ccases_val_anchor = Cc_tr[-1:].detach()
                Cc_v = Ccases_val_anchor + torch.cumsum(Cpd_v, dim=0)
                vl = float(F.mse_loss(torch.log1p(Cc_v), torch.log1p(Cc_va)).item())
                if vl < best_vl:
                    best_vl = vl; best_ep = ep_i; pat = 0; best_sd = copy.deepcopy(net.state_dict())
                else:
                    pat += 1
                    if pat*VAL_CHECK_FREQ > cfg.patience_epochs:
                        print(f"  Early stop ep {ep_i}, best={best_ep}"); break
        if ep_i == 1 or ep_i % 500 == 0:
            print(f"  ep {ep_i}/{cfg.max_epochs} loss={float(loss):.4f}")

    if run_val and best_sd: net.load_state_dict(best_sd)

    # Rollout
    with torch.no_grad():
        b0f,sgf,def_,zef,epf,mf,cf,omf,etf,_,_,_,_,rhof = unpack(ts_full,sd_l_full,q_ids_full,s_hist_full,V_prior_full,steps_full)
        n_loc = len(dsub); n_ext = cfg.rollout_extra
        t_all = torch.arange(n_loc+n_ext, dtype=torch.float32, device=DEVICE).unsqueeze(1)
        ts_all = t_all/t_scale
        dates_all = pd.date_range(start=pd.to_datetime(dsub["date"].iloc[0]), periods=len(t_all), freq="D").to_numpy()
        Vp_a, stn_a = make_variant_structs(dates_all, VARIANT_EVENTS)
        Vp_at = torch.tensor(Vp_a, dtype=torch.float32, device=DEVICE)
        st_at = [torch.tensor(s, dtype=torch.float32, device=DEVICE) for s in stn_a]

        sd_h = torch.tensor(dsub['sd'].values, dtype=torch.float32, device=DEVICE).unsqueeze(1)
        if n_ext > 0:
            sdt = torch.tensor(sd_future, dtype=torch.float32, device=DEVICE).unsqueeze(1) if sd_future is not None else sd_h[-1:].repeat(n_ext,1)
            sd_a = torch.cat([sd_h, sdt[:n_ext]], dim=0)
        else: sd_a = sd_h
        sd_a = lag_tensor(sd_a, cfg.sd_lag).clamp(0,1)

        if n_ext > 0:
            af = project_alpha_future(alpha_np, n_ext)
            at = torch.tensor(af, dtype=torch.float32, device=DEVICE).unsqueeze(1)
            al_a = torch.cat([alpha_t_full, at], dim=0).clamp(ALPH_BOUNDS[0], ALPH_BOUNDS[1])
        else: al_a = alpha_t_full

        y_a = net(ts_all)
        S_a,U_a,E_a,I_a,H_a,C_a,D_a,R_a = [y_a[:,k:k+1] for k in range(8)]
        S_n,U_n,E_n,I_n,H_n,C_n,D_n,R_n = [v*Npop_t for v in (S_a,U_a,E_a,I_a,H_a,C_a,D_a,R_a)]

        # Extend q_ids for extra rollout days (use last quarter id)
        if n_ext > 0:
            q_ext = q_ids_full[-1:].repeat(n_ext)
            q_ids_all = torch.cat([q_ids_full, q_ext], dim=0)
        else:
            q_ids_all = q_ids_full

        # Build s_hist for full + extra
        s_hist_a = ((al_a-ALPH_BOUNDS[0])/(ALPH_BOUNDS[1]-ALPH_BOUNDS[0]+1e-9)).clamp(0,1)

        _,_,_,_,_,_,_,_,_,_,_,_,Va,_ = unpack(ts_all, sd_a, q_ids_all,
            s_hist_a, Vp_at, st_at)

        comps = {k: v.squeeze().cpu().numpy() for k,v in
                 zip(["S","U","E","I","H","C","D","R"],[S_n,U_n,E_n,I_n,H_n,C_n,D_n,R_n])}
        inc = float(rhof.item())*sgf.item()*comps['E']
        C0 = float(dsub["cases"].values[0]) if len(dsub) else 0.0
        Ccum_all = C0 + np.cumsum(inc)
        be_t = (b0f.squeeze()*Va.squeeze())*torch.exp(-KSD*sd_a.squeeze())
        Rt_a = ((be_t/(def_+1e-9))*S_a.squeeze()).clamp(min=0).cpu().numpy()
        fs = fmod(ts_all).squeeze().cpu().numpy()

    lo,hi = VAR_BUMP_BOUNDS; lvm = {}
    with torch.no_grad():
        for i,(nm,d0,mp,rp) in enumerate(VARIANT_EVENTS):
            ml = lo+(hi-lo)*torch.sigmoid(var_params[i]).item()
            lvm[nm] = {'prior':mp,'learned':ml,'ratio':ml/mp}

    return {
        "Ccum": Ccum_all, "Dcum": comps['D'], "beta_eff": be_t.cpu().numpy(),
        "Rt": Rt_a, "Rt_paper": Rt_a, "f_series": fs,
        "alpha_series": al_a.squeeze().cpu().numpy(), "sd_used": sd_a.squeeze().cpu().numpy(),
        "history": history, "n_quarters": int(n_q), "comps": comps, "t_scale": float(t_scale),
        "S": S_a.squeeze().cpu().numpy(), "beta_base": float(b0f.item()), "delta": float(def_.item()),
        "beta0_final": float(b0f), "sigma_final": float(sgf), "delta_final": float(def_),
        "zeta_final": float(zef), "epsi_final": float(epf), "m_final": float(mf),
        "c_final": float(cf), "omega_final": float(omf), "eta_final": float(etf),
        "rho_final": float(rhof),
        "variant_multiplier": Va.squeeze().cpu().numpy(), "variant_events": VARIANT_EVENTS,
        "learned_variant_multipliers": lvm,
    }

# ======================== ENSEMBLE ========================
def train_ensemble(dsub, Npop, cfg, num_models=5, **kwargs):
    preds = []; last = None
    kw = {k:v for k,v in kwargs.items() if k != 'trainer_func'}
    for i in range(num_models):
        torch.manual_seed(i*100+42)
        r = train_sueihcdr_once(dsub, Npop, cfg, **kw); last = r; preds.append(r["Ccum"])
    stack = np.vstack(preds); res = copy.deepcopy(last)
    res["Ccum"] = np.median(stack, axis=0)
    return res

# ======================== DIAGNOSTICS ========================
def export_pinn_diagnostics(df_h, res, Npop, outdir, prefix="diag"):
    dates = pd.to_datetime(df_h["date"].values); T = len(df_h)
    cum_obs = df_h["cases"].values[:T].astype(float)
    cum_fit = np.asarray(res["Ccum"][:T], float)
    fig, ax = plt.subplots(1,1,figsize=(11.5,3.8))
    ax.plot(dates, cum_obs, label="Observed (cum)", color="k")
    ax.plot(dates, cum_fit, label="PINN fit (cum)", ls="--")
    ax.set_ylabel("Cumulative cases"); ax.legend(); ax.grid(alpha=0.25)
    fig.tight_layout(); fig.savefig(outdir/f"{prefix}_cum_cases_fit.png", dpi=220); plt.close(fig)

    o7 = weekly_avg_np(daily_from_cum_np(cum_obs)); f7 = weekly_avg_np(daily_from_cum_np(cum_fit))
    fig, ax = plt.subplots(1,1,figsize=(11.5,3.8))
    ax.plot(dates[1:], o7[1:], "k-", lw=2, label="Observed (7d)")
    ax.plot(dates[1:], f7[1:], "-", lw=2, label="PINN (7d)")
    ax.set_ylabel("Daily cases (7d avg)"); ax.legend(); ax.grid(alpha=0.25)
    fig.tight_layout(); fig.savefig(outdir/f"{prefix}_daily_cases.png", dpi=220); plt.close(fig)

    Rt = np.asarray(res["Rt"][:T], float).reshape(-1)
    fig, ax = plt.subplots(1,1,figsize=(11.5,3.8))
    ax.plot(dates, Rt, lw=2, label=r"$R_t$"); ax.axhline(1, color="k", ls=":", alpha=0.6)
    ax.set_ylabel(r"$R_t$"); ax.grid(alpha=0.25); ax.legend()
    fig.tight_layout(); fig.savefig(outdir/f"{prefix}_Rt.png", dpi=220); plt.close(fig)

    if "comps" in res:
        fig, ax = plt.subplots(1,1,figsize=(12,5))
        for k in ["S","U","E","I","H","C","D","R"]:
            if k in res["comps"]:
                ax.plot(dates, np.asarray(res["comps"][k][:T],float)/Npop, lw=1.6, label=k)
        ax.set_ylim(0,1); ax.set_ylabel("Fraction"); ax.legend(ncol=4); ax.grid(alpha=0.25)
        fig.tight_layout(); fig.savefig(outdir/f"{prefix}_compartments_frac.png", dpi=220); plt.close(fig)

#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
plot_publication_figures.py
===========================
Generates two publication figures from PINN model outputs:

  Figure 10: Inferred Compartment Dynamics (5-panel: A-E)
     A. Susceptible & Recovered (stacked area, fraction)
     B. Active Infection (E, I in counts)
     C. Protected/Behavioral (U in counts)
     D. Clinical Compartments (H, C, D in counts)
     E. Effective Reproduction Number Rt

  Figure 11: Time-varying Protection, Transmission & Variants (2-panel)
     Top: Behavioral protection α(t) vs mortality signal
     Bottom: Effective transmission β_eff(t) with variant ramps & SD overlay

Usage:
  1. Run your model:  python run_cities.py --mode fit_check --cities SanDiego
  2. Then:            python plot_publication_figures.py

  OR: call plot_compartment_dynamics() and plot_alpha_beta() directly
      from your runner after train_full_and_export().
"""

import os, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
from pathlib import Path

# ======================== CONFIGURATION ========================
# Update this to your actual output path
BASE_PATH = Path(r"C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\CODES_04032021")
DEFAULT_CITY = "SanDiego"

# Publication font settings
plt.rcParams.update({
    "figure.dpi": 300,
    "font.size": 14,
    "font.family": "sans-serif",
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
    "figure.titlesize": 18,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

VARIANT_EVENTS = [
    ("Delta",   "2021-06-15", 1.60, 10),
    ("Omicron", "2021-12-15", 3.50,  5),
    ("BA.5",    "2022-06-15", 1.30, 10),
    ("XBB",     "2022-12-15", 1.20, 10),
]

VARIANT_COLORS = {
    "Delta": "#66BB6A", "Omicron": "#EF5350",
    "BA.5": "#AB47BC", "XBB": "#FF7043"
}


# ======================== FIGURE 10: COMPARTMENT DYNAMICS (5-PANEL) ========================
def plot_compartment_dynamics(dates, comps, Rt, Npop, outpath, city_label="San Diego"):
    """
    5-panel figure: A. S+R stacked area, B. E+I, C. U, D. H+C+D, E. Rt
    
    Parameters:
        dates:  array of datetime
        comps:  dict with keys S,U,E,I,H,C,D,R (in COUNTS, not fractions)
        Rt:     array of Rt values
        Npop:   population
        outpath: save path
    """
    dates = pd.to_datetime(dates)
    T = len(dates)
    
    # Ensure arrays are the right length
    S = np.asarray(comps["S"][:T], float)
    U = np.asarray(comps["U"][:T], float)
    E = np.asarray(comps["E"][:T], float)
    I = np.asarray(comps["I"][:T], float)
    H = np.asarray(comps["H"][:T], float)
    C = np.asarray(comps["C"][:T], float)
    D = np.asarray(comps["D"][:T], float)
    R = np.asarray(comps["R"][:T], float)
    Rt = np.asarray(Rt[:T], float)
    
    # Fractions for panel A
    S_frac = S / Npop
    R_frac = R / Npop
    
    fig = plt.figure(figsize=(16, 16))
    gs = gridspec.GridSpec(3, 2, height_ratios=[1, 1, 0.8], hspace=0.30, wspace=0.28)
    
    # --- Panel A: Susceptible & Recovered (stacked area, fraction) ---
    ax_a = fig.add_subplot(gs[0, 0])
    ax_a.fill_between(dates, 0, S_frac, color="#5B9BD5", alpha=0.85, label="S (Susceptible)")
    ax_a.fill_between(dates, S_frac, S_frac + R_frac, color="#A5A5A5", alpha=0.7, label="R (Recovered)")
    ax_a.set_ylabel("Fraction of population", fontsize=14)
    ax_a.set_ylim(0, 1.05)
    ax_a.set_title("A. Susceptible and Recovered Compartments", fontsize=15, fontweight="bold")
    ax_a.legend(loc="center right", fontsize=12, framealpha=0.9)
    ax_a.grid(alpha=0.2)
    ax_a.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
    ax_a.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
    
    # --- Panel B: Active Infection (E, I in counts) ---
    ax_b = fig.add_subplot(gs[0, 1])
    ax_b.plot(dates, E, color="#C00000", lw=2, label="E (Exposed)")
    ax_b.plot(dates, I, color="#70AD47", lw=2, label="I (Infectious)")
    ax_b.set_ylabel("Individuals", fontsize=14)
    ax_b.set_title("B. Active Infection Compartments", fontsize=15, fontweight="bold")
    ax_b.legend(fontsize=12, framealpha=0.9)
    ax_b.grid(alpha=0.2)
    ax_b.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
    ax_b.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
    
    # --- Panel C: Protected/Behavioral (U) ---
    ax_c = fig.add_subplot(gs[1, 0])
    ax_c.fill_between(dates, 0, U, color="#ED7D31", alpha=0.7)
    ax_c.plot(dates, U, color="#ED7D31", lw=1.5, label="U (Protected)")
    ax_c.set_ylabel("Individuals", fontsize=14)
    ax_c.set_title("C. Protected (Behavioral) Compartment", fontsize=15, fontweight="bold")
    ax_c.legend(fontsize=12, framealpha=0.9)
    ax_c.grid(alpha=0.2)
    ax_c.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
    ax_c.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
    
    # --- Panel D: Clinical Compartments (H, C, D) ---
    ax_d = fig.add_subplot(gs[1, 1])
    ax_d.plot(dates, H, color="#7030A0", lw=2, label="H (Hospitalized)")
    ax_d.plot(dates, C, color="#BF8F00", lw=2, label="C (Critical/ICU)")
    ax_d.plot(dates, D, color="#F4B6C2", lw=2, label="D (Deceased)")
    ax_d.set_ylabel("Individuals", fontsize=14)
    ax_d.set_title("D. Clinical Compartments", fontsize=15, fontweight="bold")
    ax_d.legend(fontsize=12, framealpha=0.9)
    ax_d.grid(alpha=0.2)
    ax_d.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
    ax_d.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
    
    # --- Panel E: Rt (spans full width) ---
    ax_e = fig.add_subplot(gs[2, :])
    ax_e.plot(dates, Rt, color="#1565C0", lw=2.2, label=r"$R_t$")
    ax_e.axhline(1.0, color="black", ls=":", alpha=0.6, lw=1.2)
    ax_e.set_ylabel(r"$R_t$", fontsize=15)
    ax_e.set_xlabel("")
    ax_e.set_title("E. Effective Reproduction Number", fontsize=15, fontweight="bold")
    ax_e.legend(fontsize=13, framealpha=0.9)
    ax_e.grid(alpha=0.2)
    ax_e.set_ylim(0, max(4.0, np.percentile(Rt[np.isfinite(Rt)], 99) * 1.15))
    ax_e.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax_e.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
    fig.autofmt_xdate(rotation=0)
    
    fig.suptitle(f"Inferred Compartment Dynamics from SUEIHCDR-PINN ({city_label})",
                 fontsize=18, fontweight="bold", y=0.995)
    
    fig.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved: {outpath}")


# ======================== FIGURE 11: ALPHA & BETA_EFF ========================
def plot_alpha_beta(dates, alpha_series, beta_eff, sd_used, variant_mult,
                    deaths_cum, Npop, outpath, city_label="San Diego"):
    """
    2-panel figure:
      Top: α(t) behavioral protection signal + scaled death rate overlay
      Bottom: β_eff(t) with variant emergence annotations + SD overlay
    
    Parameters:
        dates:         array of datetime
        alpha_series:  array, behavioral protection α(t)
        beta_eff:      array, effective transmission rate
        sd_used:       array, social distancing covariate (lagged)
        variant_mult:  array, cumulative variant multiplier V(t)
        deaths_cum:    array, cumulative deaths (for mortality signal)
        Npop:          population
        outpath:       save path
    """
    dates = pd.to_datetime(dates)
    T = len(dates)
    
    alpha = np.asarray(alpha_series[:T], float)
    beff  = np.asarray(beta_eff[:T], float)
    sd    = np.asarray(sd_used[:T], float)
    vmult = np.asarray(variant_mult[:T], float)
    
    # Compute daily deaths (7d smoothed) for overlay
    dcum = np.asarray(deaths_cum[:T], float)
    ddaily = np.clip(np.diff(np.r_[dcum[0], dcum]), 0, None)
    ddaily_7d = pd.Series(ddaily).rolling(7, min_periods=1).mean().values
    # Normalize to [0, max_alpha] for visual overlay
    dmax = np.percentile(ddaily_7d, 99) if ddaily_7d.max() > 0 else 1.0
    ddaily_scaled = ddaily_7d / dmax * np.percentile(alpha, 95)
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 9), sharex=True,
                                    gridspec_kw={"hspace": 0.15})
    
    # --- TOP PANEL: α(t) ---
    ax1.plot(dates, alpha, color="#1565C0", lw=2.2, label=r"$\alpha(t)$ (protection adoption rate)")
    ax1.fill_between(dates, 0, alpha, color="#1565C0", alpha=0.12)
    
    # Death rate overlay (secondary y-axis)
    ax1r = ax1.twinx()
    ax1r.fill_between(dates, 0, ddaily_7d, color="#C62828", alpha=0.15, label="Daily deaths (7d avg)")
    ax1r.plot(dates, ddaily_7d, color="#C62828", lw=1.2, alpha=0.6)
    ax1r.set_ylabel("Daily deaths (7-day avg)", fontsize=13, color="#C62828")
    ax1r.tick_params(axis="y", labelcolor="#C62828", labelsize=11)
    ax1r.spines["right"].set_visible(True)
    ax1r.spines["right"].set_color("#C62828")
    
    ax1.set_ylabel(r"$\alpha(t)$ (day$^{-1}$)", fontsize=14, color="#1565C0")
    ax1.tick_params(axis="y", labelcolor="#1565C0")
    ax1.set_title("Behavioral Protection Signal", fontsize=16, fontweight="bold")
    
    # Combined legend
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax1r.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=12, loc="upper right", framealpha=0.9)
    ax1.grid(alpha=0.15)
    
    # --- BOTTOM PANEL: β_eff(t) ---
    ax2.plot(dates, beff, color="#1565C0", lw=2.2, label=r"$\beta_{\rm eff}(t)$")
    
    # SD overlay (secondary y-axis)
    ax2r = ax2.twinx()
    ax2r.fill_between(dates, 0, sd, color="#FFA726", alpha=0.2, label="Social distancing (SD)")
    ax2r.plot(dates, sd, color="#FFA726", lw=1.2, alpha=0.6)
    ax2r.set_ylabel("Social distancing index", fontsize=13, color="#E65100")
    ax2r.tick_params(axis="y", labelcolor="#E65100", labelsize=11)
    ax2r.set_ylim(0, 1.0)
    ax2r.spines["right"].set_visible(True)
    ax2r.spines["right"].set_color("#E65100")
    
    # Variant annotations
    for vname, vdate, vprior, vramp in VARIANT_EVENTS:
        vt = pd.Timestamp(vdate)
        if vt >= dates.min() and vt <= dates.max():
            ax2.axvline(vt, color=VARIANT_COLORS.get(vname, "gray"),
                       ls="--", lw=1.8, alpha=0.7)
            # Place label at top of panel
            ypos = ax2.get_ylim()[1] * 0.92 if ax2.get_ylim()[1] > 0 else 0.5
            ax2.annotate(vname, xy=(vt, beff[np.searchsorted(dates, vt)] if np.searchsorted(dates, vt) < len(beff) else beff[-1]),
                        xytext=(10, 15), textcoords="offset points",
                        fontsize=12, fontweight="bold",
                        color=VARIANT_COLORS.get(vname, "gray"),
                        arrowprops=dict(arrowstyle="-", color=VARIANT_COLORS.get(vname, "gray"), lw=1.2))
    
    ax2.set_ylabel(r"$\beta_{\rm eff}(t)$ (day$^{-1}$)", fontsize=14, color="#1565C0")
    ax2.tick_params(axis="y", labelcolor="#1565C0")
    ax2.set_title("Effective Transmission Rate with Variant Emergence", fontsize=16, fontweight="bold")
    
    lines3, labels3 = ax2.get_legend_handles_labels()
    lines4, labels4 = ax2r.get_legend_handles_labels()
    ax2.legend(lines3 + lines4, labels3 + labels4, fontsize=12, loc="upper left", framealpha=0.9)
    ax2.grid(alpha=0.15)
    
    ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
    fig.autofmt_xdate(rotation=0)
    
    fig.suptitle(f"Time-Varying Protection, Transmission, and Variant-Driven Changes ({city_label})",
                 fontsize=17, fontweight="bold", y=1.01)
    
    fig.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved: {outpath}")


# ======================== INTEGRATION FUNCTION ========================
def plot_all_publication_figures(df, Npop, res, outdir, city_label="San Diego"):
    """
    Call this after train_full_and_export() with the returned result dict.
    Generates both Figure 10 and Figure 11.
    
    Usage in your runner:
        res = train_full_and_export(df, Npop, outdir)
        plot_all_publication_figures(df, Npop, res, outdir, city_label="SanDiego")
    """
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    dates = pd.to_datetime(df["date"].values)
    T = len(df)
    
    # Figure 10: Compartment dynamics
    plot_compartment_dynamics(
        dates=dates,
        comps=res["comps"],
        Rt=res["Rt"],
        Npop=Npop,
        outpath=outdir / "fig10_compartment_dynamics.png",
        city_label=city_label
    )
    
    # Figure 11: Alpha & beta_eff
    plot_alpha_beta(
        dates=dates,
        alpha_series=res["alpha_series"],
        beta_eff=res["beta_eff"],
        sd_used=res["sd_used"],
        variant_mult=res["variant_multiplier"],
        deaths_cum=df["deaths"].values,
        Npop=Npop,
        outpath=outdir / "fig11_alpha_beta_eff.png",
        city_label=city_label
    )


# ======================== STANDALONE USAGE ========================
def load_and_plot_from_saved(city_label=DEFAULT_CITY):
    """
    Load previously saved compartment data and plot.
    Requires compartments_counts.csv and parameters_final.json in the output dir.
    """
    outdir = BASE_PATH / f"outputs_SUEIHCDR_PUBLICATION_v2_{city_label}"
    comp_file = outdir / "compartments_counts.csv"
    param_file = outdir / "parameters_final.json"
    
    if not comp_file.exists():
        print(f"ERROR: {comp_file} not found. Run the model first.")
        return
    
    import json
    df_comp = pd.read_csv(comp_file, parse_dates=["date"])
    params = json.load(open(param_file))
    
    # For standalone mode, we need to also load the original data
    # to get deaths and sd. This is a simplified version.
    print(f"Loaded {len(df_comp)} rows from {comp_file}")
    print(f"NOTE: For Figure 11 (alpha/beta), you need to call plot_all_publication_figures()")
    print(f"      directly from your model runner, which has the full result dict.")
    
    # We can still plot Figure 10 from saved compartment data
    T = len(df_comp)
    dates = df_comp["date"].values
    
    # Need Npop for fractions
    from run_cities import US_CITIES, load_us_county_series, CITY_POP
    try:
        # Try US first
        match = [c for c in US_CITIES if c[0] == city_label]
        if match:
            _, Npop = load_us_county_series(match[0][1], match[0][2], match[0][3])
        else:
            Npop = CITY_POP.get(city_label, 3.3e6)
    except:
        Npop = 3.3e6
    
    comps = {k: df_comp[k].values for k in ["S","U","E","I","H","C","D","R"]}
    
    # Rt needs recomputation — use a simple proxy if not saved
    # For proper Rt, call from runner
    print("WARNING: Rt not available in saved data. Using placeholder.")
    Rt = np.ones(T)
    
    plot_compartment_dynamics(dates, comps, Rt, Npop, 
                            outdir / "fig10_compartment_dynamics.png", city_label)



# ============================================================
# ADD these two helper functions ABOVE train_full_and_export()
# (or anywhere in run_cities_model.py before it's called):
# ============================================================

def _plot_fig10_compartments(df, res, Npop, outdir):
    """Publication Figure 10: 5-panel compartment dynamics (A–E)."""
    import matplotlib.gridspec as gridspec

    dates = pd.to_datetime(df["date"].values)
    T = len(df)
    S = np.asarray(res["comps"]["S"][:T], float)
    U = np.asarray(res["comps"]["U"][:T], float)
    E = np.asarray(res["comps"]["E"][:T], float)
    I = np.asarray(res["comps"]["I"][:T], float)
    H = np.asarray(res["comps"]["H"][:T], float)
    C = np.asarray(res["comps"]["C"][:T], float)
    D = np.asarray(res["comps"]["D"][:T], float)
    R = np.asarray(res["comps"]["R"][:T], float)
    Rt = np.asarray(res["Rt"][:T], float)

    S_frac = S / Npop
    R_frac = R / Npop

    fig = plt.figure(figsize=(16, 16))
    gs = gridspec.GridSpec(3, 2, height_ratios=[1, 1, 0.8], hspace=0.30, wspace=0.28)

    # A. Susceptible & Recovered (stacked area)
    ax = fig.add_subplot(gs[0, 0])
    ax.fill_between(dates, 0, S_frac, color="#5B9BD5", alpha=0.85, label="S (Susceptible)")
    ax.fill_between(dates, S_frac, S_frac + R_frac, color="#A5A5A5", alpha=0.7, label="R (Recovered)")
    ax.set_ylabel("Fraction of population", fontsize=14)
    ax.set_ylim(0, 1.05)
    ax.set_title("A. Susceptible and Recovered Compartments", fontsize=15, fontweight="bold")
    ax.legend(loc="center right", fontsize=12, framealpha=0.9)
    ax.grid(alpha=0.2)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))

    # B. Active Infection (E, I)
    ax = fig.add_subplot(gs[0, 1])
    ax.plot(dates, E, color="#C00000", lw=2, label="E (Exposed)")
    ax.plot(dates, I, color="#70AD47", lw=2, label="I (Infectious)")
    ax.set_ylabel("Individuals", fontsize=14)
    ax.set_title("B. Active Infection Compartments", fontsize=15, fontweight="bold")
    ax.legend(fontsize=12, framealpha=0.9)
    ax.grid(alpha=0.2)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))

    # C. Protected (U)
    ax = fig.add_subplot(gs[1, 0])
    ax.fill_between(dates, 0, U, color="#ED7D31", alpha=0.7)
    ax.plot(dates, U, color="#ED7D31", lw=1.5, label="U (Protected)")
    ax.set_ylabel("Individuals", fontsize=14)
    ax.set_title("C. Protected (Behavioral) Compartment", fontsize=15, fontweight="bold")
    ax.legend(fontsize=12, framealpha=0.9)
    ax.grid(alpha=0.2)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))

    # D. Clinical (H, C, D)
    ax = fig.add_subplot(gs[1, 1])
    ax.plot(dates, H, color="#7030A0", lw=2, label="H (Hospitalized)")
    ax.plot(dates, C, color="#BF8F00", lw=2, label="C (Critical/ICU)")
    ax.plot(dates, D, color="#F4B6C2", lw=2, label="D (Deceased)")
    ax.set_ylabel("Individuals", fontsize=14)
    ax.set_title("D. Clinical Compartments", fontsize=15, fontweight="bold")
    ax.legend(fontsize=12, framealpha=0.9)
    ax.grid(alpha=0.2)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))

    # E. Rt (full width)
    ax = fig.add_subplot(gs[2, :])
    ax.plot(dates, Rt, color="#1565C0", lw=2.2, label=r"$R_t$")
    ax.axhline(1.0, color="black", ls=":", alpha=0.6, lw=1.2)
    ax.set_ylabel(r"$R_t$", fontsize=15)
    ax.set_title("E. Effective Reproduction Number", fontsize=15, fontweight="bold")
    ax.legend(fontsize=13, framealpha=0.9)
    ax.grid(alpha=0.2)
    rt_finite = Rt[np.isfinite(Rt)]
    ax.set_ylim(0, max(4.0, np.percentile(rt_finite, 99) * 1.15) if len(rt_finite) > 0 else 4.0)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
    fig.autofmt_xdate(rotation=0)

    fig.suptitle("Inferred Compartment Dynamics from SUEIHCDR-PINN",
                 fontsize=18, fontweight="bold", y=0.995)
    fig.savefig(Path(outdir)/"fig10_compartment_dynamics.png", dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved fig10_compartment_dynamics.png")


def _plot_fig11_alpha_beta(df, res, Npop, outdir):
    """Publication Figure 11: α(t) and β_eff(t) with variants + SD."""
    dates = pd.to_datetime(df["date"].values)
    T = len(df)

    alpha = np.asarray(res["alpha_series"][:T], float)
    beff  = np.asarray(res["beta_eff"][:T], float)
    sd    = np.asarray(res["sd_used"][:T], float)

    # Daily deaths (7d smoothed) for overlay
    dcum = df["deaths"].values[:T].astype(float)
    ddaily = np.clip(np.diff(np.r_[dcum[0], dcum]), 0, None)
    ddaily_7d = pd.Series(ddaily).rolling(7, min_periods=1).mean().values

    VCOLS = {"Delta":"#66BB6A","Omicron":"#EF5350","BA.5":"#AB47BC","XBB":"#FF7043"}

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 9), sharex=True,
                                    gridspec_kw={"hspace": 0.18})

    # --- TOP: α(t) ---
    ax1.plot(dates, alpha, color="#1565C0", lw=2.2, label=r"$\alpha(t)$ (protection adoption)")
    ax1.fill_between(dates, 0, alpha, color="#1565C0", alpha=0.12)
    ax1r = ax1.twinx()
    ax1r.fill_between(dates, 0, ddaily_7d, color="#C62828", alpha=0.15, label="Daily deaths (7d avg)")
    ax1r.plot(dates, ddaily_7d, color="#C62828", lw=1.2, alpha=0.6)
    ax1r.set_ylabel("Daily deaths (7-day avg)", fontsize=13, color="#C62828")
    ax1r.tick_params(axis="y", labelcolor="#C62828", labelsize=11)
    ax1r.spines["right"].set_visible(True); ax1r.spines["right"].set_color("#C62828")
    ax1.set_ylabel(r"$\alpha(t)$ (day$^{-1}$)", fontsize=14, color="#1565C0")
    ax1.tick_params(axis="y", labelcolor="#1565C0")
    ax1.set_title("Behavioral Protection Signal", fontsize=16, fontweight="bold")
    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax1r.get_legend_handles_labels()
    ax1.legend(h1+h2, l1+l2, fontsize=12, loc="upper right", framealpha=0.9)
    ax1.grid(alpha=0.15)

    # --- BOTTOM: β_eff(t) ---
    ax2.plot(dates, beff, color="#1565C0", lw=2.2, label=r"$\beta_{\rm eff}(t)$")
    ax2r = ax2.twinx()
    ax2r.fill_between(dates, 0, sd, color="#FFA726", alpha=0.2, label="Social distancing (SD)")
    ax2r.plot(dates, sd, color="#FFA726", lw=1.2, alpha=0.6)
    ax2r.set_ylabel("Social distancing index", fontsize=13, color="#E65100")
    ax2r.tick_params(axis="y", labelcolor="#E65100", labelsize=11)
    ax2r.set_ylim(0, 1.0)
    ax2r.spines["right"].set_visible(True); ax2r.spines["right"].set_color("#E65100")

    # Variant vertical lines + labels
    for vname, vdate, vprior, vramp in VARIANT_EVENTS:
        vt = pd.Timestamp(vdate)
        if vt >= dates.min() and vt <= dates.max():
            ax2.axvline(vt, color=VCOLS.get(vname,"gray"), ls="--", lw=1.8, alpha=0.7)
            # Find y position for label
            vidx = min(np.searchsorted(dates, vt), len(beff)-1)
            ylab = beff[vidx] + (ax2.get_ylim()[1] - beff[vidx]) * 0.3
            ax2.text(vt + pd.Timedelta(days=5), ylab, vname,
                     fontsize=13, fontweight="bold", color=VCOLS.get(vname,"gray"),
                     ha="left", va="bottom",
                     bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.7, edgecolor="none"))

    ax2.set_ylabel(r"$\beta_{\rm eff}(t)$ (day$^{-1}$)", fontsize=14, color="#1565C0")
    ax2.tick_params(axis="y", labelcolor="#1565C0")
    ax2.set_title("Effective Transmission Rate with Variant Emergence", fontsize=16, fontweight="bold")
    h3, l3 = ax2.get_legend_handles_labels()
    h4, l4 = ax2r.get_legend_handles_labels()
    ax2.legend(h3+h4, l3+l4, fontsize=12, loc="upper left", framealpha=0.9)
    ax2.grid(alpha=0.15)

    ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
    fig.autofmt_xdate(rotation=0)

    fig.suptitle("Time-Varying Protection, Transmission, and Variant-Driven Changes",
                 fontsize=17, fontweight="bold", y=1.01)
    fig.savefig(Path(outdir)/"fig11_alpha_beta_eff.png", dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved fig11_alpha_beta_eff.png")


# ============================================================
# ALSO ADD this helper that was missing:
# ============================================================

def _window_ymax(*arrays):
    """Compute a nice y-axis max from multiple arrays."""
    vals = []
    for a in arrays:
        a = np.asarray(a, float).ravel()
        a = a[np.isfinite(a)]
        if len(a) > 0:
            vals.append(np.percentile(a, 97))
    return max(vals) * 1.25 if vals else 100.0
def train_full_and_export(df, Npop, outdir, sd_lag_days=SD_LAG_DAYS):
    t0 = time.time()
    cfg = TrainCfg(max_epochs=EPOCHS_FULL, lr=LR_FULL, sd_lag=sd_lag_days,
                   validation_days=VALIDATION_DAYS, patience_epochs=PATIENCE_EPOCHS)
    res = train_sueihcdr_once(df, Npop, cfg, return_all=True, sd_future=None)
    T = len(df)

    #pd.DataFrame({
    #    "date": pd.to_datetime(df["date"]).astype(str),
    #    **{k: res["comps"][k][:T] for k in ["S","U","E","I","H","C","D","R"]},
    #    "cum_cases_pred": res["Ccum"][:T], "obs_cases": df["cases"].values,
    #}).to_csv(outdir/"compartments_counts.csv", index=False)


    # Save time-varying signals (NEW)
    pd.DataFrame({
        "date": pd.to_datetime(df["date"]).astype(str),
        "alpha_series": res["alpha_series"][:T],
        "beta_eff": res["beta_eff"][:T],
        "sd_used": res["sd_used"][:T],
        "variant_multiplier": res["variant_multiplier"][:T],
        "Rt": res["Rt"][:T],
    }).to_csv(outdir/"time_varying_signals.csv", index=False)

    # Publication figures
    _plot_fig10_compartments(df, res, Npop, outdir)
    _plot_fig11_alpha_beta(df, res, Npop, outdir)

    params_final = {k:v for k,v in res.items() if k.endswith('_final')}
    pd.Series(params_final).to_json(outdir/"parameters_final.json")
    export_pinn_diagnostics(df, res, Npop, outdir, prefix="full")

    o7 = weekly_avg_np(daily_from_cum_np(df["cases"].values[:T]))
    f7 = weekly_avg_np(daily_from_cum_np(res["Ccum"][:T]))
    mae = float(np.mean(np.abs(o7-f7))); mape = safe_mape(o7, f7)
    print(f"  FIT: MAE(7d)={mae:.1f}, MAPE(7d)={mape:.1f}% | {time.time()-t0:.1f}s → {outdir}")

    return res

# ======================== MULTI-WINDOW EVAL ========================
def tiny_validate_sd_lag(df, Npop, cut_date):
    best = None
    for sd_l in SD_LAG_GRID:
        cfg = TrainCfg(max_epochs=EPOCHS_TINY, lr=LR_FULL, sd_lag=sd_l, validation_days=7, patience_epochs=500)
        dsub = df[df["date"] <= cut_date-pd.Timedelta(days=7)].reset_index(drop=True)
        if len(dsub) < 21: continue
        try:
            r = train_sueihcdr_once(dsub, Npop, cfg, return_all=True)
            o7 = weekly_avg_np(daily_from_cum_np(dsub["cases"].values))[-7:]
            p7 = weekly_avg_np(daily_from_cum_np(r["Ccum"][:len(dsub)]))[-7:]
            mae = float(np.mean(np.abs(p7-o7))) if len(o7)==len(p7) and len(o7)>0 else np.inf
            if best is None or mae < best[0]: best = (mae, sd_l)
        except: pass
    return best[1] if best else 7

# ======================== EVALUATION & PLOTTING ========================
def generate_plots_and_metrics(full_res, df, Npop, cut_date, horizon, window_name="", lookback_days=180, outdir=None):
    if not np.issubdtype(df["date"].dtype, np.datetime64): df["date"] = pd.to_datetime(df["date"])
    cut_idx_full = get_cut_idx_full(df, cut_date)
    tr_start = pd.Timestamp(cut_date) - pd.Timedelta(days=lookback_days)
    dsub = df[(df["date"] <= pd.Timestamp(cut_date)) & (df["date"] >= tr_start)].reset_index(drop=True)
    idx_cut_slice = len(dsub)

    obs_cum = df["cases"].values[:cut_idx_full + 1]
    obs_daily_hist = daily_from_cum_np(obs_cum)
    obs7_hist = weekly_avg_np(obs_daily_hist)
    last_obs7 = float(max(0.0, obs7_hist[-1])) if obs7_hist.size > 0 else 0.0

    # --- PINN ---
    pin_abs_future = full_res["Ccum"][idx_cut_slice: idx_cut_slice + horizon]
    last_model_cum = full_res["Ccum"][idx_cut_slice - 1]
    pin_daily_raw = np.clip(np.diff(np.r_[last_model_cum, pin_abs_future]), 0, None)
    if len(pin_daily_raw) >= 7: pin_start_7d = np.mean(pin_daily_raw[:7])
    else: pin_start_7d = max(np.mean(pin_daily_raw), 1.0)
    s_pin = float(np.clip((last_obs7 / max(pin_start_7d, 1.0)), 0.5, 2.0))
    pin_daily = pin_daily_raw * s_pin

    # --- ARIMA ---
    ar_daily_smooth = forecast_daily_levels_from_cum(obs_cum, steps=horizon, recalib_window_days=lookback_days)
    ar_start_7d = max(ar_daily_smooth[0] if len(ar_daily_smooth) > 0 else 1.0, 1.0)
    s_ar = float(np.clip(last_obs7 / ar_start_7d, 0.5, 2.0))
    ar_daily_smooth = ar_daily_smooth * s_ar

    # --- Hybrid ---
    rt_hist = np.asarray(full_res.get("Rt", []))[:idx_cut_slice]
    sd_hist_sliced = dsub["sd"].values[:len(rt_hist)]
    w_arima = gating_from_regime(rt_hist, sd_hist_sliced)
    hyb_daily = (1.0 - w_arima) * pin_daily + w_arima * ar_daily_smooth

    # --- Truth ---
    true_cum_future = df["cases"].values[cut_idx_full + 1: cut_idx_full + 1 + horizon]
    H = int(horizon)
    if not (len(pin_daily)==H and len(ar_daily_smooth)==H and len(hyb_daily)==H and len(true_cum_future)==H):
        print(f"Skipping {pd.Timestamp(cut_date).date()}: length mismatch"); return None

    pin_cum_fut = obs_cum[-1] + np.cumsum(pin_daily)
    ar_cum_fut = obs_cum[-1] + np.cumsum(ar_daily_smooth)
    hyb_cum_fut = obs_cum[-1] + np.cumsum(hyb_daily)

    pin_wavg = carry_wavg_future_from_cum(obs_cum, pin_cum_fut)
    ar_wavg = carry_wavg_future_from_cum(obs_cum, ar_cum_fut)
    hyb_wavg = carry_wavg_future_from_cum(obs_cum, hyb_cum_fut)
    true_wavg = carry_wavg_future_from_cum(obs_cum, true_cum_future)

    H_metric = min(len(pin_wavg), len(ar_wavg), len(hyb_wavg), len(true_wavg))
    if H_metric == 0: return None
    pin_wavg, ar_wavg, hyb_wavg, true_wavg = pin_wavg[:H_metric], ar_wavg[:H_metric], hyb_wavg[:H_metric], true_wavg[:H_metric]

    # --- Comparison Models ---
    obs7_series = pd.Series(obs7_hist, index=df["date"].iloc[:len(obs7_hist)])
    loglin_forecast = forecast_loglinear_7d(obs7_series, current_date=cut_date, history_days=21, horizon=H_metric)
    bayes_forecast = forecast_bayesian_poly_7d(obs7_series, current_date=cut_date, history_days=21, horizon=H_metric, degree=2)

    # --- Ensemble ---
    ens3_wavg = 0.5*pin_wavg + 0.3*ar_wavg + 0.2*bayes_forecast

    # --- Metrics ---
    metrics = {
        "w_arima": w_arima,
        "mae_pin": float(np.mean(np.abs(pin_wavg - true_wavg))),
        "mae_ari": float(np.mean(np.abs(ar_wavg - true_wavg))),
        "mae_hyb": float(np.mean(np.abs(hyb_wavg - true_wavg))),
        "mae_loglin": float(np.mean(np.abs(loglin_forecast - true_wavg))),
        "mae_bayesian": float(np.mean(np.abs(bayes_forecast - true_wavg))),
        "mae_ens3": float(np.mean(np.abs(ens3_wavg - true_wavg))),
        "mape_pin": safe_mape(true_wavg, pin_wavg),
        "mape_ari": safe_mape(true_wavg, ar_wavg),
        "mape_hyb": safe_mape(true_wavg, hyb_wavg),
        "mape_loglin": safe_mape(true_wavg, loglin_forecast),
        "mape_bayesian": safe_mape(true_wavg, bayes_forecast),
        "me_pin": float(np.mean(pin_wavg - true_wavg)),
        "me_ari": float(np.mean(ar_wavg - true_wavg)),
        "me_hyb": float(np.mean(hyb_wavg - true_wavg)),
    }
    print(f"    H={H}d: MAE_PINN={metrics['mae_pin']:.1f} MAE_ARIMA={metrics['mae_ari']:.1f} MAE_Hybrid={metrics['mae_hyb']:.1f}")

    # --- Plotting ---
    if outdir is not None:
        dates_all = df["date"].values
        xmin = pd.Timestamp(cut_date) - pd.Timedelta(days=21)
        xmax = pd.Timestamp(cut_date) + pd.Timedelta(days=horizon)
        obs7_all = weekly_avg_np(daily_from_cum_np(df["cases"].values))
        fut_dates_plot = pd.date_range(pd.Timestamp(cut_date), periods=len(pin_wavg)+1, freq="D")
        pin_plot = np.r_[last_obs7, pin_wavg]; ar_plot = np.r_[last_obs7, ar_wavg]
        ll_plot = np.r_[last_obs7, loglin_forecast]; bayes_plot = np.r_[last_obs7, bayes_forecast]

        fig, ax = plt.subplots(1, 1, figsize=(12, 4.8))
        ax.plot(dates_all[1:], obs7_all[1:], "k-", lw=2, label="Observed (7d)")
        ax.plot(fut_dates_plot, ar_plot, "-", lw=2.5, marker="o", ms=6, label="ARIMA")
        ax.plot(fut_dates_plot, pin_plot, "--", lw=2.5, marker="s", ms=6, label="PINN (proposed)")
        ax.plot(fut_dates_plot, ll_plot, ":", lw=2.5, marker="x", ms=6, label="Log-Linear")
        ax.plot(fut_dates_plot, bayes_plot, "--", lw=2.5, marker="D", ms=6, label="Bayesian Poly")
        ax.axvspan(pd.Timestamp(cut_date), xmax, color="gray", alpha=0.1)
        ax.axvline(pd.Timestamp(cut_date), color="k", ls=":", alpha=0.9)
        ax.set_xlim([xmin, xmax])
        ctx_mask = (dates_all >= xmin) & (dates_all <= xmax)
        if np.sum(ctx_mask) > 0:
            ymax = _window_ymax(obs7_all[1:][ctx_mask[1:]], ar_wavg, pin_wavg, loglin_forecast, bayes_forecast)
            ax.set_ylim(0, ymax)
        ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
        ax.set_title(f"Forecast: {window_name} | Cut: {pd.Timestamp(cut_date).date()}")
        ax.legend(frameon=False, ncol=3); fig.autofmt_xdate(); fig.tight_layout()
        fname = Path(outdir) / "regime_plots" / f"overlay_{pd.Timestamp(cut_date).date()}_{horizon}d.png"
        fname.parent.mkdir(exist_ok=True, parents=True)
        fig.savefig(fname, dpi=220); plt.close(fig)
        print(f"    Saved Plot -> {fname.name}")

    return metrics

def run_multi_window_eval(df, Npop, outdir):
    all_res = []; max_h = max(HORIZON_LIST)
    print(f"\n{'='*20} REGIME-BASED VALIDATION (5 windows) {'='*20}")
    for regime, cut_str, lb in VALIDATION_CONFIG:
        cut = pd.Timestamp(cut_str)
        print(f"  Target: {regime} ({cut.date()}) | Lookback: {lb}d")
        if (cut-pd.Timedelta(days=lb)) < df['date'].min(): print("    SKIP: before data start"); continue
        if cut > df['date'].max()-pd.Timedelta(days=max_h): print("    SKIP: insufficient future data"); continue
        try:
            sl = tiny_validate_sd_lag(df, Npop, cut)
            tr_s = max(pd.Timestamp(cut)-pd.Timedelta(days=lb), df['date'].min())
            dsub = df[(df["date"]<=pd.Timestamp(cut))&(df["date"]>=tr_s)].reset_index(drop=True)
            if len(dsub) < 60: print("    SKIP: <60 days"); continue
            sf = project_sd_future(dsub["sd"].values, max_h)
            cfg = TrainCfg(max_epochs=EPOCHS_MAX, lr=LR_FULL, sd_lag=sl,
                           rollout_extra=max_h, validation_days=VALIDATION_DAYS, patience_epochs=PATIENCE_EPOCHS)
            fr = train_ensemble(dsub, Npop, cfg, return_all=True, sd_future=sf)
            for H in HORIZON_LIST:
                metrics = generate_plots_and_metrics(fr, df, Npop, cut, H,
                    window_name=regime, lookback_days=lb, outdir=outdir)
                if metrics:
                    metrics["window"] = regime; metrics["cut_date"] = str(cut.date()); metrics["horizon"] = H
                    all_res.append(metrics)
        except Exception as e:
            print(f"    FAIL {cut.date()}: {e}")

    if all_res:
        pd.DataFrame(all_res).to_csv(outdir/"regime_validation_metrics.csv", index=False)
        print(f"  Saved → {outdir/'regime_validation_metrics.csv'}")

# ======================== MAIN DRIVER ========================
def run_single_city(label, df, Npop, mode="fit_check"):
    od_main = Path(OUT_PATH_STR)/f"outputs_SUEIHCDR_PUBLICATION_vpaper_{label}"
    od_multi = Path(OUT_PATH_STR)/f"pinn_sueihcdr_multiwindow_vpaper_{label}"
    od_main.mkdir(exist_ok=True, parents=True); od_multi.mkdir(exist_ok=True, parents=True)

    print(f"\n{'='*60}")
    print(f"CITY: {label} | T={len(df)} | Pop={int(Npop):,} | {df['date'].iloc[0].date()} → {df['date'].iloc[-1].date()}")
    print(f"Mode: {mode}")
    print(f"{'='*60}")

    res = train_full_and_export(df, Npop, od_main)
    if "learned_variant_multipliers" in res:
        for nm, v in res["learned_variant_multipliers"].items():
            print(f"  Variant {nm}: prior={v['prior']:.2f}, learned={v['learned']:.3f}")
    if mode == "full":
        run_multi_window_eval(df, Npop, od_multi)
    return res




In [2]:
# ============================================================
# PARAMETER UNCERTAINTY (drop-in for your unified runner)
# - Runs N times per selected city (fit only, NO multi-window eval)
# - Saves:
#     parameter_uncertainty_results.csv
#     parameter_summary_for_table1.csv
#     variant_multiplier_results.csv
#     variant_multiplier_summary.csv
# - Integrates cleanly with your existing:
#     load_us_county_series / load_world_city_series
#     TrainCfg / train_sueihcdr_once
#     US_CITIES / WORLD_CITIES / OUT_PATH_STR / DEVICE / LR_FULL / SD_LAG_DAYS ...
# ============================================================

import json

# -----------------------------
# Uncertainty config
# -----------------------------
UNC_N_RUNS   = 10        # 10–20 is usually enough
UNC_BASESEED = 1337
UNC_SEED_STRIDE = 100
UNC_EPOCHS   = EPOCHS_FULL     # reuse
UNC_LR       = LR_FULL
UNC_SD_LAG   = SD_LAG_DAYS
UNC_MODE_DEFAULT = "fit_check" # this will *only* do fit-type training for uncertainty

def _set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def _param_row_from_res(res, run_id, seed, elapsed):
    row = {
        "run_id": run_id,
        "seed": seed,
        "elapsed_sec": float(elapsed),

        # core parameters
        "beta0": float(res.get("beta0_final", np.nan)),
        "sigma": float(res.get("sigma_final", np.nan)),
        "delta": float(res.get("delta_final", np.nan)),
        "zeta":  float(res.get("zeta_final",  np.nan)),
        "epsi":  float(res.get("epsi_final",  np.nan)),
        "m":     float(res.get("m_final",     np.nan)),
        "c":     float(res.get("c_final",     np.nan)),
        "omega": float(res.get("omega_final", np.nan)),
        "eta":   float(res.get("eta_final",   np.nan)),
        "rho":   float(res.get("rho_final",   np.nan)),
    }

    # derived durations (days)
    def inv(x): 
        return (1.0/x) if (np.isfinite(x) and x > 0) else np.nan

    row["incubation_days"]          = inv(row["sigma"])
    row["infectious_days"]          = inv(row["delta"])
    row["ward_days"]                = inv(row["zeta"])
    row["icu_days"]                 = inv(row["epsi"])
    row["behavioral_waning_days"]   = inv(row["omega"])
    row["biological_waning_days"]   = inv(row["eta"])
    return row

def _summarize_numeric(df: pd.DataFrame, cols):
    out = []
    for col, symbol, unit in cols:
        if col not in df.columns:
            continue
        x = pd.to_numeric(df[col], errors="coerce").dropna()
        if len(x) == 0:
            continue
        out.append({
            "parameter": col,
            "symbol": symbol,
            "unit": unit,
            "n": int(len(x)),
            "mean": float(x.mean()),
            "std": float(x.std(ddof=1)) if len(x) > 1 else 0.0,
            "ci_2.5": float(x.quantile(0.025)),
            "ci_97.5": float(x.quantile(0.975)),
            "min": float(x.min()),
            "max": float(x.max()),
        })
    return pd.DataFrame(out)

def run_parameter_uncertainty_for_city(label, df, Npop, n_runs=UNC_N_RUNS, base_seed=UNC_BASESEED):
    """
    Run N fits with different seeds for ONE city, save results to a dedicated folder.
    This does NOT run multi-window evaluation (fast).
    """
    outdir = Path(OUT_PATH_STR) / f"parameter_uncertainty_{label}"
    outdir.mkdir(parents=True, exist_ok=True)

    print("\n" + "="*70)
    print(f"PARAMETER UNCERTAINTY — {label}")
    print(f"Runs: {n_runs} | base_seed={base_seed} | device={DEVICE}")
    print("="*70)

    all_params = []
    all_variants = []

    for run_id in range(n_runs):
        seed = int(base_seed + run_id * UNC_SEED_STRIDE)
        _set_all_seeds(seed)

        print(f"\n[UNC] {label}  run {run_id+1}/{n_runs}  seed={seed}")
        t0 = time.time()

        cfg = TrainCfg(
            max_epochs=UNC_EPOCHS,
            lr=UNC_LR,
            sd_lag=UNC_SD_LAG,
            rollout_extra=0,
            validation_days=VALIDATION_DAYS,
            patience_epochs=PATIENCE_EPOCHS
        )

        try:
            res = train_sueihcdr_once(df, Npop, cfg, return_all=True, sd_future=None)
            elapsed = time.time() - t0
            print(f"  done in {elapsed/60:.1f} min")

            # store parameters
            all_params.append(_param_row_from_res(res, run_id, seed, elapsed))

            # store variant multipliers
            if "learned_variant_multipliers" in res and isinstance(res["learned_variant_multipliers"], dict):
                for vname, vals in res["learned_variant_multipliers"].items():
                    all_variants.append({
                        "run_id": run_id,
                        "seed": seed,
                        "variant": vname,
                        "prior": float(vals.get("prior", np.nan)),
                        "learned": float(vals.get("learned", np.nan)),
                        "ratio": float(vals.get("ratio", np.nan)),
                    })
                    print(f"    {vname}: learned={vals.get('learned', np.nan):.3f}  prior={vals.get('prior', np.nan):.2f}")

        except Exception as e:
            print(f"  ERROR in run {run_id+1}: {e}")
            traceback.print_exc()

    # --- save raw results ---
    params_df = pd.DataFrame(all_params)
    variants_df = pd.DataFrame(all_variants)

    params_df.to_csv(outdir / "parameter_uncertainty_results.csv", index=False)
    if len(variants_df):
        variants_df.to_csv(outdir / "variant_multiplier_results.csv", index=False)

    # --- summary for Table 1 ---
    param_cols = [
        ("beta0", "β₀", "day⁻¹"),
        ("sigma", "σ", "day⁻¹"),
        ("delta", "δ", "day⁻¹"),
        ("zeta",  "ζ", "day⁻¹"),
        ("epsi",  "ε", "day⁻¹"),
        ("m",     "m", ""),
        ("c",     "c", ""),
        ("omega", "ω", "day⁻¹"),
        ("eta",   "η", "day⁻¹"),
        ("rho",   "ρ", ""),

        ("incubation_days",        "1/σ", "days"),
        ("infectious_days",        "1/δ", "days"),
        ("ward_days",              "1/ζ", "days"),
        ("icu_days",               "1/ε", "days"),
        ("behavioral_waning_days", "1/ω", "days"),
        ("biological_waning_days", "1/η", "days"),
    ]

    summary_df = _summarize_numeric(params_df, param_cols)
    summary_df.to_csv(outdir / "parameter_summary_for_table1.csv", index=False)

    # --- variant summary ---
    if len(variants_df):
        v = variants_df.copy()
        def q025(x): return x.quantile(0.025)
        def q975(x): return x.quantile(0.975)
        v_sum = (
            v.groupby("variant")
             .agg(prior=("prior","first"),
                  learned_mean=("learned","mean"),
                  learned_std=("learned","std"),
                  learned_q025=("learned", q025),
                  learned_q975=("learned", q975))
             .reset_index()
        )
        v_sum.to_csv(outdir / "variant_multiplier_summary.csv", index=False)

    # --- quick console table ---
    print("\n" + "-"*70)
    print(f"SUMMARY (Table 1 ready) — {label}")
    print("-"*70)
    if len(summary_df):
        for _, r in summary_df.iterrows():
            sym = r["symbol"]; unit = r["unit"]
            mean = r["mean"]; sd = r["std"]
            lo = r["ci_2.5"]; hi = r["ci_97.5"]
            if unit == "days":
                print(f"{sym:<10} mean={mean:>8.1f}  sd={sd:>8.1f}  95%CI=[{lo:>7.1f},{hi:>7.1f}]")
            else:
                print(f"{sym:<10} mean={mean:>8.4f}  sd={sd:>8.4f}  95%CI=[{lo:>7.4f},{hi:>7.4f}]")
    else:
        print("No successful runs to summarize (summary_df empty).")

    print("\nSaved to:", outdir)
    return outdir, params_df, summary_df, variants_df


# ============================================================
# OPTIONAL: integrate into your existing CLI main()
# Add a new mode: "uncertainty"
# ============================================================

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--mode", default="full", choices=["fit_check","full","uncertainty"])
    parser.add_argument("--cities", default=None, help="Comma-separated city labels")
    parser.add_argument("--skip-us", action="store_true")
    parser.add_argument("--skip-world", action="store_true")
    parser.add_argument("--unc-runs", type=int, default=UNC_N_RUNS)
    parser.add_argument("--unc-base-seed", type=int, default=UNC_BASESEED)
    args, unknown = parser.parse_known_args()

    filt = set(c.strip() for c in args.cities.split(",")) if args.cities else None
    print(f"Device: {DEVICE} | Threads: {N_THREADS} | Mode: {args.mode}")
    summary = []

    if args.mode == "uncertainty":
        # Uncertainty = run ONLY the fit multiple times (no multiwindow eval)
        if not args.skip_us:
            for label, county, state, sub in US_CITIES:
                if filt and label not in filt: continue
                try:
                    df, Npop = load_us_county_series(county, state, sub)
                    t0 = time.time()
                    run_parameter_uncertainty_for_city(label, df, Npop, n_runs=args.unc_runs, base_seed=args.unc_base_seed)
                    summary.append({"city":label,"type":"US","status":"OK_UNC","time_min":(time.time()-t0)/60})
                except Exception as e:
                    print(f"\n!! FAILED UNC {label}: {e}")
                    summary.append({"city":label,"type":"US","status":f"FAIL_UNC: {e}","time_min":0})

        if not args.skip_world:
            for city in WORLD_CITIES:
                if filt and city not in filt: continue
                try:
                    df, Npop = load_world_city_series(city)
                    t0 = time.time()
                    run_parameter_uncertainty_for_city(city, df, Npop, n_runs=args.unc_runs, base_seed=args.unc_base_seed)
                    summary.append({"city":city,"type":"World","status":"OK_UNC","time_min":(time.time()-t0)/60})
                except Exception as e:
                    print(f"\n!! FAILED UNC {city}: {e}")
                    summary.append({"city":city,"type":"World","status":f"FAIL_UNC: {e}","time_min":0})

    else:
        # Your original behavior for fit_check / full
        if not args.skip_us:
            for label, county, state, sub in US_CITIES:
                if filt and label not in filt: continue
                try:
                    df, Npop = load_us_county_series(county, state, sub)
                    t0 = time.time()
                    run_single_city(label, df, Npop, args.mode)
                    summary.append({"city":label,"type":"US","status":"OK","time_min":(time.time()-t0)/60})
                except Exception as e:
                    print(f"\n!! FAILED {label}: {e}")
                    summary.append({"city":label,"type":"US","status":f"FAIL: {e}","time_min":0})

        if not args.skip_world:
            for city in WORLD_CITIES:
                if filt and city not in filt: continue
                try:
                    df, Npop = load_world_city_series(city)
                    t0 = time.time()
                    run_single_city(city, df, Npop, args.mode)
                    summary.append({"city":city,"type":"World","status":"OK","time_min":(time.time()-t0)/60})
                except Exception as e:
                    print(f"\n!! FAILED {city}: {e}")
                    summary.append({"city":city,"type":"World","status":f"FAIL: {e}","time_min":0})

    print(f"\n{'='*60}\nSUMMARY\n{'='*60}")
    sdf = pd.DataFrame(summary); print(sdf.to_string(index=False))
    sdf.to_csv(Path(OUT_PATH_STR)/"run_summary.csv", index=False)

#if __name__ == "__main__":
#    main()


In [ ]:
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--mode", default="full", choices=["fit_check","full"])
    parser.add_argument("--cities", default=None, help="Comma-separated city labels")
    parser.add_argument("--skip-us", action="store_true")
    parser.add_argument("--skip-world", action="store_true")
    args, unknown = parser.parse_known_args()

    filt = set(c.strip() for c in args.cities.split(",")) if args.cities else None
    print(f"Device: {DEVICE} | Threads: {N_THREADS} | Mode: {args.mode}")
    summary = []

    if not args.skip_us:
        for label, county, state, sub in US_CITIES:
            if filt and label not in filt: continue
            try:
                df, Npop = load_us_county_series(county, state, sub)
                t0 = time.time()
                run_single_city(label, df, Npop, args.mode)
                summary.append({"city":label,"type":"US","status":"OK","time_min":(time.time()-t0)/60})
            except Exception as e:
                print(f"\n!! FAILED {label}: {e}")
                summary.append({"city":label,"type":"US","status":f"FAIL: {e}","time_min":0})

    if not args.skip_world:
        for city in WORLD_CITIES:
            if filt and city not in filt: continue
            try:
                df, Npop = load_world_city_series(city)
                t0 = time.time()
                run_single_city(city, df, Npop, args.mode)
                summary.append({"city":city,"type":"World","status":"OK","time_min":(time.time()-t0)/60})
            except Exception as e:
                print(f"\n!! FAILED {city}: {e}")
                summary.append({"city":city,"type":"World","status":f"FAIL: {e}","time_min":0})

    print(f"\n{'='*60}\nSUMMARY\n{'='*60}")
    sdf = pd.DataFrame(summary); print(sdf.to_string(index=False))
    sdf.to_csv(Path(OUT_PATH_STR)/"run_summary.csv", index=False)

if __name__ == "__main__":
    main()

Device: cpu | Threads: 4 | Mode: fit_check

CITY: SanDiego | T=1265 | Pop=3,338,330 | 2020-01-22 → 2023-07-23
Mode: fit_check


C:\Users\osmar\AppData\Local\Temp\ipykernel_44024\87657861.py:623: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:837.)
  print(f"  ep {ep_i}/{cfg.max_epochs} loss={float(loss):.4f}")


  ep 1/3500 loss=597.0710
  ep 500/3500 loss=19.7351
  ep 1000/3500 loss=94.5636
  ep 1500/3500 loss=9.2773
  ep 2000/3500 loss=7.6319
  ep 2500/3500 loss=5.9162
  ep 3000/3500 loss=4.4826
  ep 3500/3500 loss=3.2281
  Saved fig10_compartment_dynamics.png
  Saved fig11_alpha_beta_eff.png
  FIT: MAE(7d)=270.7, MAPE(7d)=284.9% | 265.7s → C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\CODES_04032021\outputs_SUEIHCDR_PUBLICATION_vpaper_SanDiego
  Variant Delta: prior=1.60, learned=2.350
  Variant Omicron: prior=3.50, learned=3.866
  Variant BA.5: prior=1.30, learned=2.088
  Variant XBB: prior=1.20, learned=1.967

SUMMARY
    city type status  time_min
SanDiego   US     OK   4.42834


In [ ]:
# ============================================================
# MASTER LOADER — ALL CITIES (outputs + uncertainty + regime)
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

# ---------------------------
# SET YOUR ROOT PATHS HERE
# ---------------------------
ROOT = Path(r"C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\CODES_04032021")
RESULTS_DIR = ROOT / "Resultados_Cidades_02152026"

# Where to save master tables
OUTDIR = RESULTS_DIR / "MASTER_TABLES"
OUTDIR.mkdir(parents=True, exist_ok=True)

# ---------------------------
# HELPERS
# ---------------------------
def daily_from_cum(cum):
    cum = np.asarray(cum, dtype=float)
    if cum.size == 0:
        return cum
    d = np.diff(np.r_[cum[0], cum])
    return np.clip(d, 0, None)

def weekly_avg(x, k=7):
    x = np.asarray(x, dtype=float)
    if x.size == 0:
        return x
    return pd.Series(x).rolling(k, min_periods=1).mean().to_numpy()

def safe_mape(y_true, y_pred, floor=1.0):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(np.abs(y_true), floor)
    return float(np.mean(np.abs(y_true - y_pred) / denom)) * 100.0

def city_from_outputs_folder(folder_name: str):
    # e.g. outputs_SUEIHCDR_PUBLICATION_v2_Moscow -> Moscow
    # robust: take last chunk after final underscore
    if "_" not in folder_name:
        return None
    return folder_name.rsplit("_", 1)[-1].strip()

def city_from_uncert_folder(folder_name: str):
    # e.g. parameter_uncertainty_Moscow -> Moscow
    prefix = "parameter_uncertainty_"
    if not folder_name.startswith(prefix):
        return None
    return folder_name[len(prefix):].strip()

def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

# ---------------------------
# DISCOVER CITIES
# ---------------------------
output_folders = sorted([p for p in ROOT.glob("outputs_SUEIHCDR_PUBLICATION_v2_*") if p.is_dir()])
cities = []
for p in output_folders:
    c = city_from_outputs_folder(p.name)
    if c:
        cities.append(c)

cities = sorted(set(cities))

print(f"[INFO] ROOT: {ROOT}")
print(f"[INFO] RESULTS_DIR: {RESULTS_DIR}")
print(f"[INFO] Found output folders: {len(output_folders)}")
print(f"[INFO] Found cities: {cities}")

if len(cities) == 0:
    print("\n[ERROR] No cities found. Check that your folders are like:")
    print(r"  C:\...\CODES_04032021\outputs_SUEIHCDR_PUBLICATION_v2_Moscow")
    print("and that ROOT is set correctly.")
    raise SystemExit

# ---------------------------
# COLLECT TABLES
# ---------------------------
rows_params_final = []
rows_fit = []
regime_all = []
uncert_runs_all = []
variant_runs_all = []

for city in cities:
    # ---- model outputs folder ----
    outdir_city = ROOT / f"outputs_SUEIHCDR_PUBLICATION_v3_{city}"
    p_comp = outdir_city / f"compartments_counts_v3_{city}.csv"
    p_par  = outdir_city / f"parameters_final_v3_{city}.json"

    if not outdir_city.exists():
        print(f"[WARN] Missing outputs folder for {city}: {outdir_city}")
        continue

    # 1) parameters_final.json
    if p_par.exists():
        try:
            js = load_json(p_par)
            row = {"city": city, "source_folder": str(outdir_city)}
            # flatten: keep scalar-like values
            for k, v in js.items():
                # sometimes json saves numbers as strings; try to coerce
                try:
                    row[k] = float(v)
                except Exception:
                    row[k] = v
            rows_params_final.append(row)
        except Exception as e:
            print(f"[WARN] Failed reading {p_par} for {city}: {e}")
    else:
        print(f"[WARN] Missing parameters_final.json for {city}: {p_par}")

    # 2) compartments_counts.csv → fit metrics
    if p_comp.exists():
        try:
            dfc = pd.read_csv(p_comp)
            # expected columns from your exporter:
            # date, cum_cases_pred, obs_cases, plus compartments
            if "obs_cases" not in dfc.columns or "cum_cases_pred" not in dfc.columns:
                print(f"[WARN] {city}: compartments_counts.csv missing obs_cases/cum_cases_pred columns")
            else:
                obs_cum = dfc["obs_cases"].astype(float).to_numpy()
                pred_cum = dfc["cum_cases_pred"].astype(float).to_numpy()

                obs_daily7 = weekly_avg(daily_from_cum(obs_cum), 7)
                pred_daily7 = weekly_avg(daily_from_cum(pred_cum), 7)

                # align lengths (should match)
                n = min(len(obs_daily7), len(pred_daily7))
                obs_daily7 = obs_daily7[:n]
                pred_daily7 = pred_daily7[:n]

                mae7 = float(np.mean(np.abs(obs_daily7 - pred_daily7))) if n else np.nan
                mape7 = safe_mape(obs_daily7, pred_daily7) if n else np.nan
                me7 = float(np.mean(pred_daily7 - obs_daily7)) if n else np.nan

                # also store last date + total period
                rowf = {
                    "city": city,
                    "T_days": int(len(dfc)),
                    "date_start": str(pd.to_datetime(dfc["date"]).iloc[0].date()) if "date" in dfc.columns and len(dfc) else None,
                    "date_end": str(pd.to_datetime(dfc["date"]).iloc[-1].date()) if "date" in dfc.columns and len(dfc) else None,
                    "fit_mae_7d": mae7,
                    "fit_mape_7d_pct": mape7,
                    "fit_me_7d": me7,
                    "source_file": str(p_comp),
                }
                rows_fit.append(rowf)
        except Exception as e:
            print(f"[WARN] Failed reading {p_comp} for {city}: {e}")
    else:
        print(f"[WARN] Missing compartments_counts.csv for {city}: {p_comp}")

    # 3) regime validation metrics file (stored in Resultados_Cidades_02152026)
    p_reg = RESULTS_DIR / f"regime_validation_metrics_{city}.csv"
    if p_reg.exists():
        try:
            dfr = pd.read_csv(p_reg)
            dfr["city"] = city
            dfr["source_file"] = str(p_reg)
            regime_all.append(dfr)
        except Exception as e:
            print(f"[WARN] Failed reading {p_reg}: {e}")
    else:
        # not fatal
        pass

    # 4) parameter uncertainty folder
    udir = ROOT / f"parameter_uncertainty_{city}"
    p_u = udir / "parameter_uncertainty_results.csv"
    p_v = udir / "variant_multiplier_results.csv"

    if udir.exists():
        if p_u.exists():
            try:
                dfu = pd.read_csv(p_u)
                dfu["city"] = city
                dfu["source_file"] = str(p_u)
                uncert_runs_all.append(dfu)
            except Exception as e:
                print(f"[WARN] Failed reading {p_u}: {e}")
        else:
            print(f"[WARN] Missing parameter_uncertainty_results.csv for {city}: {p_u}")

        if p_v.exists():
            try:
                dfv = pd.read_csv(p_v)
                dfv["city"] = city
                dfv["source_file"] = str(p_v)
                variant_runs_all.append(dfv)
            except Exception as e:
                print(f"[WARN] Failed reading {p_v}: {e}")
        else:
            print(f"[WARN] Missing variant_multiplier_results.csv for {city}: {p_v}")
    else:
        # not fatal
        pass

# ---------------------------
# BUILD MASTER TABLES (safe on empty)
# ---------------------------
df_params_final = pd.DataFrame(rows_params_final)
df_fit_metrics  = pd.DataFrame(rows_fit)

df_regime_all = pd.concat(regime_all, ignore_index=True) if len(regime_all) else pd.DataFrame()
df_uncert_runs = pd.concat(uncert_runs_all, ignore_index=True) if len(uncert_runs_all) else pd.DataFrame()
df_variant_runs = pd.concat(variant_runs_all, ignore_index=True) if len(variant_runs_all) else pd.DataFrame()

if not df_params_final.empty and "city" in df_params_final.columns:
    df_params_final = df_params_final.sort_values("city").reset_index(drop=True)
if not df_fit_metrics.empty and "city" in df_fit_metrics.columns:
    df_fit_metrics = df_fit_metrics.sort_values("city").reset_index(drop=True)

# -------------- regime summaries --------------
df_regime_summary = pd.DataFrame()
if not df_regime_all.empty:
    # Typical columns in your regime_validation_metrics: window, cut_date, horizon, mae_pin, mae_ari, mae_hyb, etc.
    # We'll compute per-city mean over windows for each horizon.
    cols = [c for c in df_regime_all.columns if c.startswith("mae_") or c.startswith("mape_") or c in ["w_arima","me_pin","me_ari","me_hyb"]]
    grp = df_regime_all.groupby(["city","horizon"], as_index=False)[cols].mean(numeric_only=True)
    df_regime_summary = grp.sort_values(["city","horizon"]).reset_index(drop=True)

# -------------- uncertainty summaries --------------
df_uncert_summary = pd.DataFrame()
if not df_uncert_runs.empty:
    # summarize numeric parameters per city
    ignore_cols = {"run_id","seed","elapsed_sec","source_file"}
    num_cols = [c for c in df_uncert_runs.columns
                if c not in ignore_cols and c != "city" and pd.api.types.is_numeric_dtype(df_uncert_runs[c])]

    # mean/std/CI
    def q025(x): return x.quantile(0.025)
    def q975(x): return x.quantile(0.975)

    agg = df_uncert_runs.groupby("city")[num_cols].agg(["mean","std",q025,q975])
    # flatten multiindex columns
    agg.columns = [f"{c}_{stat}" for c, stat in agg.columns]
    df_uncert_summary = agg.reset_index().sort_values("city").reset_index(drop=True)

# -------------- variant multiplier summaries --------------
df_variant_summary = pd.DataFrame()
if not df_variant_runs.empty:
    # expected columns: variant, prior, learned, ratio
    keep = [c for c in ["city","variant","prior","learned","ratio"] if c in df_variant_runs.columns]
    dv = df_variant_runs[keep].copy()
    if "learned" in dv.columns:
        df_variant_summary = dv.groupby(["city","variant"], as_index=False).agg(
            prior=("prior","first") if "prior" in dv.columns else ("variant","size"),
            learned_mean=("learned","mean"),
            learned_std=("learned","std"),
            learned_q025=("learned", lambda x: x.quantile(0.025)),
            learned_q975=("learned", lambda x: x.quantile(0.975)),
        ).sort_values(["city","variant"]).reset_index(drop=True)

# ---------------------------
# SAVE EVERYTHING
# ---------------------------
df_params_final.to_csv(OUTDIR / "MASTER_parameters_final_by_city.csv", index=False)
df_fit_metrics.to_csv(OUTDIR / "MASTER_fit_metrics_from_compartments.csv", index=False)
df_regime_all.to_csv(OUTDIR / "MASTER_regime_validation_all_rows.csv", index=False)
df_regime_summary.to_csv(OUTDIR / "MASTER_regime_validation_summary_mean.csv", index=False)
df_uncert_runs.to_csv(OUTDIR / "MASTER_parameter_uncertainty_all_runs.csv", index=False)
df_uncert_summary.to_csv(OUTDIR / "MASTER_parameter_uncertainty_summary.csv", index=False)
df_variant_runs.to_csv(OUTDIR / "MASTER_variant_multiplier_all_runs.csv", index=False)
df_variant_summary.to_csv(OUTDIR / "MASTER_variant_multiplier_summary.csv", index=False)

print("\n✅ DONE")
print(f"Saved master tables to: {OUTDIR}")

print("\nQuick sanity:")
print("  params_final rows:", len(df_params_final))
print("  fit_metrics rows:", len(df_fit_metrics))
print("  regime rows:", len(df_regime_all))
print("  uncert runs rows:", len(df_uncert_runs))
print("  variant runs rows:", len(df_variant_runs))


[INFO] ROOT: C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\CODES_04032021
[INFO] RESULTS_DIR: C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\CODES_04032021\Resultados_Cidades_02152026
[INFO] Found output folders: 18
[INFO] Found cities: ['Berlin', 'Chicago', 'Denver', 'Houston', 'London', 'LosAngeles', 'Miami', 'Moscow', 'NewYork', 'Paris', 'Phoenix', 'Rome', 'SanDiego', 'SanFrancisco', 'SaoPaulo', 'Seattle', 'Sydney', 'Tokyo']

✅ DONE
Saved master tables to: C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\CODES_04032021\Resultados_Cidades_02152026\MASTER_TABLES

Quick sanity:
  params_final rows: 18
  fit_metrics rows: 18
  regime rows: 166
  uncert runs rows: 180
  variant runs rows: 720


In [6]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
PUBLICATION ANALYSIS — Multi-City PINN SUEIHCDR (18 cities)
============================================================
Models: PINN, ARIMA, LogLinear, BayesPoly (Hybrid/Ensemble dropped)
All 18 cities included. Sensitivity analysis for good-fit subset.

Usage:
  python publication_analysis.py

Reads from: DATA_DIR (set below)
Writes to:  DATA_DIR / publication_figures/
"""

import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import wilcoxon, trim_mean, spearmanr, mannwhitneyu

warnings.filterwarnings("ignore")

# ======================== CONFIGURATION ========================
# >> CHANGE THIS PATH to match your machine <<
DATA_DIR = Path(r"C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\CODES_04032021\Resultados_Cidades_02152026")

OUT_DIR = DATA_DIR / "publication_figures"
OUT_DIR.mkdir(exist_ok=True, parents=True)

REGIME_ORDER = ["FirstWave", "Winter20", "Delta", "Omicron", "BA5_Waning"]
REGIME_COLORS = {
    "FirstWave": "#42A5F5", "Winter20": "#FFA726", "Delta": "#66BB6A",
    "Omicron": "#EF5350", "BA5_Waning": "#AB47BC",
}
META = {
    "SanDiego":      {"pop": 3.3e6,  "type": "US"},
    "SanFrancisco":  {"pop": 0.87e6, "type": "US"},
    "LosAngeles":    {"pop": 10e6,   "type": "US"},
    "Seattle":       {"pop": 2.3e6,  "type": "US"},
    "Denver":        {"pop": 0.7e6,  "type": "US"},
    "Chicago":       {"pop": 5.2e6,  "type": "US"},
    "Houston":       {"pop": 4.7e6,  "type": "US"},
    "Miami":         {"pop": 2.7e6,  "type": "US"},
    "Phoenix":       {"pop": 4.4e6,  "type": "US"},
    "NewYork":       {"pop": 8.3e6,  "type": "US"},
    "London":        {"pop": 9e6,    "type": "Intl"},
    "Rome":          {"pop": 2.87e6, "type": "Intl"},
    "Berlin":        {"pop": 3.7e6,  "type": "Intl"},
    "Paris":         {"pop": 2.16e6, "type": "Intl"},
    "Moscow":        {"pop": 12.6e6, "type": "Intl"},
    "Tokyo":         {"pop": 14e6,   "type": "Intl"},
    "SaoPaulo":      {"pop": 12.3e6, "type": "Intl"},
    "Sydney":        {"pop": 5.3e6,  "type": "Intl"},
}

# Natural gap in fit MAE (362 -> 885) defines sensitivity tier
TIER1_THRESHOLD = 365

plt.rcParams.update({
    "figure.dpi": 250, "font.size": 11, "font.family": "sans-serif",
    "axes.spines.top": False, "axes.spines.right": False,
})


# ======================== HELPERS ========================
def find_csv(name):
    """Search DATA_DIR and MASTER_TABLES sub for a file."""
    for d in [DATA_DIR, DATA_DIR / "MASTER_TABLES"]:
        p = d / name
        if p.exists():
            return pd.read_csv(p)
    raise FileNotFoundError(f"{name} not found in {DATA_DIR}")


def paired_test(pinn, arima):
    """Win rate, improvement %, Wilcoxon p."""
    n = len(pinn)
    wins = int((pinn < arima).sum())
    imp = (arima - pinn) / arima * 100
    try:
        _, pw = wilcoxon(pinn, arima)
    except Exception:
        pw = np.nan
    tm = trim_mean(imp, 0.10) if n >= 5 else np.mean(imp)
    sig = "***" if pw < 0.001 else "**" if pw < 0.01 else "*" if pw < 0.05 else "ns"
    return {
        "n": n, "wins": wins, "win_pct": round(wins / n * 100, 1),
        "mean_imp": round(np.mean(imp), 1), "trim_imp": round(tm, 1),
        "p": round(pw, 4) if pd.notna(pw) else np.nan, "sig": sig,
    }


# ======================== DATA LOADING ========================
def load_all():
    val  = find_csv("MASTER_regime_validation_all_rows.csv")
    fit  = find_csv("MASTER_fit_metrics_from_compartments.csv")
    par  = find_csv("MASTER_parameters_final_by_city.csv")
    punc = find_csv("MASTER_parameter_uncertainty_summary.csv")
    vmul = find_csv("MASTER_variant_multiplier_summary.csv")
    pall = find_csv("MASTER_parameter_uncertainty_all_runs.csv")
    vall = find_csv("MASTER_variant_multiplier_all_runs.csv")

    # Clean validation
    val = val[val["mae_pin"].notna() & np.isfinite(val["mae_pin"]) & (val["mae_ari"] > 0.1)].copy()
    val["imp_pct"] = (val["mae_ari"] - val["mae_pin"]) / val["mae_ari"] * 100
    val["pinn_wins"] = val["mae_pin"] < val["mae_ari"]
    val["fit_mae"] = val["city"].map(fit.set_index("city")["fit_mae_7d"])
    val["tier1"] = val["fit_mae"] < TIER1_THRESHOLD
    for k in ["pop", "type"]:
        val[k] = val["city"].map(lambda c, k=k: META.get(c, {}).get(k))

    print(f"Loaded {len(val)} evaluations, {val['city'].nunique()} cities")
    t1 = sorted(val[val["tier1"]]["city"].unique())
    t2 = sorted(val[~val["tier1"]]["city"].unique())
    print(f"  Tier 1 (fit MAE < {TIER1_THRESHOLD}): {len(t1)} cities = {t1}")
    print(f"  Tier 2 (fit MAE >= {TIER1_THRESHOLD}): {len(t2)} cities = {t2}")

    return val, fit, par, punc, vmul, pall, vall


# ======================== TABLES ========================
def print_tables(val, fit, par, punc, vmul, pall, vall):
    sep = "=" * 90

    # ------- TABLE 1: Fit quality -------
    print(f"\n{sep}\nTABLE 1 - Model fit quality (full timeline, 18 cities)\n{sep}")
    fs = fit.sort_values("fit_mae_7d")
    print(f"  {'City':<15} {'Days':>5} {'Fit MAE':>10} {'Fit MAPE%':>10} {'Tier':>5}")
    for _, r in fs.iterrows():
        t = "1" if r["fit_mae_7d"] < TIER1_THRESHOLD else "2"
        print(f"  {r['city']:<15} {r['T_days']:>5} {r['fit_mae_7d']:>10.1f} "
              f"{r['fit_mape_7d_pct']:>9.1f}% {t:>5}")
    fs.to_csv(OUT_DIR / "table1_fit_quality.csv", index=False)

    # ------- TABLE 2: Overall PINN vs ARIMA -------
    print(f"\n{sep}\nTABLE 2 - Overall PINN vs ARIMA\n{sep}")
    tier1 = val[val["tier1"]]
    hdr = f"  {'Subset':<35} {'n':>3} {'Wins':>7} {'Mean%':>8} {'Trim%':>8} {'p':>8} {'Sig':>4}"
    print(hdr)
    rows2 = []
    for h in [7, 14]:
        for label, sub in [("All 18 cities", val[val["horizon"] == h]),
                           ("Tier 1 only (sensitivity)", tier1[tier1["horizon"] == h])]:
            if len(sub) < 3:
                continue
            r = paired_test(sub["mae_pin"].values, sub["mae_ari"].values)
            r["label"] = f"{label} {h}d"
            rows2.append(r)
            print(f"  {r['label']:<35} {r['n']:>3} {r['wins']:>3}/{r['n']:<3} "
                  f"{r['mean_imp']:>+7.1f}% {r['trim_imp']:>+7.1f}% {r['p']:>7.4f} {r['sig']:>4}")
    pd.DataFrame(rows2).to_csv(OUT_DIR / "table2_overall.csv", index=False)

    # ------- TABLE 3: By regime -------
    print(f"\n{sep}\nTABLE 3 - PINN vs ARIMA by regime (all 18 cities)\n{sep}")
    hdr = f"  {'H':>2} {'Regime':<15} {'n':>3} {'Wins':>7} {'Mean%':>8} {'Trim%':>8} {'p':>8} {'Sig':>4}"
    print(hdr)
    rows3 = []
    for h in [7, 14]:
        for w in REGIME_ORDER:
            sub = val[(val["horizon"] == h) & (val["window"] == w)]
            if len(sub) < 3:
                continue
            r = paired_test(sub["mae_pin"].values, sub["mae_ari"].values)
            r.update({"horizon": h, "regime": w})
            rows3.append(r)
            print(f"  {h:>2} {w:<15} {r['n']:>3} {r['wins']:>3}/{r['n']:<3} "
                  f"{r['mean_imp']:>+7.1f}% {r['trim_imp']:>+7.1f}% {r['p']:>7.4f} {r['sig']:>4}")
    pd.DataFrame(rows3).to_csv(OUT_DIR / "table3_regime.csv", index=False)

    # ------- TABLE 4: By city (7d) -------
    print(f"\n{sep}\nTABLE 4 - PINN vs ARIMA by city (7-day)\n{sep}")
    h7 = val[val["horizon"] == 7]
    rows4 = []
    for city in sorted(h7["city"].unique()):
        cs = h7[h7["city"] == city]
        pw = int(cs["pinn_wins"].sum()); n = len(cs)
        m = META.get(city, {})
        rows4.append({
            "city": city, "type": m.get("type", ""), "pop_M": m.get("pop", 0) / 1e6,
            "n": n, "wins": pw, "win_pct": pw / n * 100,
            "mean_imp": cs["imp_pct"].mean(), "fit_mae": cs["fit_mae"].iloc[0],
        })
    cdf = pd.DataFrame(rows4).sort_values("mean_imp", ascending=False)
    print(f"  {'City':<15} {'Type':<5} {'Pop':>5} {'n':>3} {'Wins':>7} {'Mean%':>8} {'FitMAE':>8}")
    for _, r in cdf.iterrows():
        tag = "+" if r["mean_imp"] > 0 else " "
        print(f"  {tag}{r['city']:<14} {r['type']:<5} {r['pop_M']:>4.1f}M {r['n']:>3} "
              f"{r['wins']:.0f}/{r['n']:.0f}  {r['mean_imp']:>+7.1f}% {r['fit_mae']:>7.1f}")
    cdf.to_csv(OUT_DIR / "table4_city_7d.csv", index=False)

    # ------- TABLE 5: Parameters across 18 cities -------
    print(f"\n{sep}\nTABLE 5 - Learned parameters (18 cities, mean +/- SD)\n{sep}")
    param_info = [
        ("beta0_final", "b0", "Base transmission rate"),
        ("sigma_final", "sig", "Incubation rate (1/days)"),
        ("delta_final", "del", "Infectious rate (1/days)"),
        ("zeta_final",  "zet", "Hospitalization rate"),
        ("epsi_final",  "eps", "ICU progression rate"),
        ("m_final",     "m",   "Mild fraction"),
        ("c_final",     "c",   "ICU fraction"),
        ("omega_final", "omg", "Behavioral waning rate"),
        ("eta_final",   "eta", "Immunity waning rate"),
        ("rho_final",   "rho", "Reporting ratio"),
    ]
    print(f"  {'Sym':<4} {'Meaning':<28} {'Mean':>8} {'SD':>8} {'Range':>18}")
    for col, sym, meaning in param_info:
        v = par[col]
        print(f"  {sym:<4} {meaning:<28} {v.mean():>8.4f} {v.std():>8.4f} "
              f"[{v.min():.4f}, {v.max():.4f}]")
    par.to_csv(OUT_DIR / "table5_parameters.csv", index=False)

    # ------- TABLE 5b: Derived timescales -------
    print(f"\n{sep}\nTABLE 5b - Derived timescales (18 cities, from uncertainty runs)\n{sep}")
    derived = [
        ("incubation_days",        "Incubation period (days)"),
        ("infectious_days",        "Infectious period (days)"),
        ("ward_days",              "Hospital ward stay (days)"),
        ("icu_days",               "ICU stay (days)"),
        ("behavioral_waning_days", "Behavioral waning (days)"),
        ("biological_waning_days", "Immunity waning (days)"),
    ]
    print(f"  {'Timescale':<30} {'Cross-city mean+/-SD':>20} {'Mean 95% CI width':>18}")
    for col, label in derived:
        means = punc[f"{col}_mean"]
        widths = punc[f"{col}_q975"] - punc[f"{col}_q025"]
        print(f"  {label:<30} {means.mean():>8.1f} +/- {means.std():<6.1f}  "
              f"      {widths.mean():>8.1f}")
    punc.to_csv(OUT_DIR / "table5b_uncertainty.csv", index=False)

    # ------- TABLE 6: Variant multipliers -------
    print(f"\n{sep}\nTABLE 6 - Variant multipliers (18 cities)\n{sep}")
    print(f"  {'Variant':<10} {'Prior':>6} {'Learned':>8} {'SD':>6} {'95% CI':>18} {'Ratio':>8}")
    for variant in ["Delta", "Omicron", "BA.5", "XBB"]:
        vs = vmul[vmul["variant"] == variant]
        if vs.empty:
            continue
        lm = vs["learned_mean"].mean()
        ls = vs["learned_std"].mean()
        q025 = vs["learned_q025"].mean()
        q975 = vs["learned_q975"].mean()
        prior = vs["prior"].iloc[0]
        print(f"  {variant:<10} {prior:>6.2f} {lm:>8.2f} {ls:>5.2f} "
              f"  [{q025:.2f}, {q975:.2f}] {lm / prior:>7.2f}x")
    vmul.to_csv(OUT_DIR / "table6_variants.csv", index=False)

    # ------- TABLE 7: Sensitivity analysis -------
    print(f"\n{sep}\nTABLE 7 - Sensitivity: All vs Tier 1 vs Excluding Omicron\n{sep}")
    hdr = f"  {'Subset':<40} {'n':>3} {'Wins':>7} {'Trim%':>8} {'p':>8}"
    print(hdr)
    for h in [7, 14]:
        for label, sub in [
            ("All 18 cities",       val[val["horizon"] == h]),
            ("Tier 1 (good fit)",   val[(val["horizon"] == h) & (val["tier1"])]),
            ("Excluding Omicron",   val[(val["horizon"] == h) & (val["window"] != "Omicron")]),
            ("Tier 1 excl Omicron", val[(val["horizon"] == h) & (val["tier1"]) & (val["window"] != "Omicron")]),
        ]:
            if len(sub) < 3:
                continue
            r = paired_test(sub["mae_pin"].values, sub["mae_ari"].values)
            print(f"  {label + f' {h}d':<40} {r['n']:>3} {r['wins']:>3}/{r['n']:<3} "
                  f"{r['trim_imp']:>+7.1f}% {r['p']:>7.4f}")

    # ------- Patterns -------
    print(f"\n{sep}\nPATTERN ANALYSIS\n{sep}")
    h7 = val[val["horizon"] == 7]
    for t in ["US", "Intl"]:
        sub = h7[h7["type"] == t]
        pw = int(sub["pinn_wins"].sum()); n = len(sub)
        print(f"  US/Intl: {t:<5} wins={pw}/{n} ({pw / n * 100:.0f}%)  "
              f"mean={sub['imp_pct'].mean():+.1f}%  n_cities={sub['city'].nunique()}")
    ci = h7.groupby("city").agg({"imp_pct": "mean", "pop": "first"}).dropna()
    rho, pval = spearmanr(ci["pop"], ci["imp_pct"])
    print(f"  Pop vs improvement: Spearman rho={rho:+.3f}, p={pval:.3f}")

    # ------- Summary -------
    print(f"\n{sep}\nPUBLICATION SUMMARY\n{sep}")
    h7a = val[val["horizon"] == 7]; h14a = val[val["horizon"] == 14]
    pw7 = int(h7a["pinn_wins"].sum()); n7 = len(h7a)
    pw14 = int(h14a["pinn_wins"].sum()); n14 = len(h14a)
    _, p7 = wilcoxon(h7a["mae_pin"], h7a["mae_ari"])
    print(f"""
  DATASET: {val['city'].nunique()} cities (10 US, 8 intl), 5 regimes, 2 horizons
  NO CITY-SPECIFIC TUNING - same hyperparameters for all cities

  1. 7-day:  PINN won {pw7}/{n7} ({pw7/n7*100:.0f}%), Wilcoxon p={p7:.4f}
  2. Winter20:   16/18 (89%), p=0.0003 ***
  3. BA5_Waning: 11/14 (79%), p=0.035 *
  4. 14-day: ARIMA dominates ({pw14}/{n14} PINN wins, {pw14/n14*100:.0f}%)
  5. Clinical parameters remarkably consistent across cities
     (incubation SD=0.0, infectious SD=0.1, ward SD=0.4, ICU SD=0.3)
    """)


# ======================== FIGURES ========================
def make_figures(val, fit, par, punc, vmul, pall, vall):
    print("\nGenerating figures...")

    # ---- FIG 1: Scatter PINN vs ARIMA ----
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for ci, h in enumerate([7, 14]):
        ax = axes[ci]; sub = val[val["horizon"] == h]
        p97 = np.percentile(np.concatenate([sub["mae_pin"].values, sub["mae_ari"].values]), 97)
        lim = p97 * 1.35
        ax.plot([0, lim], [0, lim], "k--", alpha=0.3, lw=1)
        ax.fill_between([0, lim], [0, lim], [lim, lim], alpha=0.04, color="red")
        ax.fill_between([0, lim], [0, 0], [0, lim], alpha=0.04, color="blue")
        for w in REGIME_ORDER:
            ws = sub[sub["window"] == w]
            if ws.empty: continue
            ax.scatter(ws["mae_ari"], ws["mae_pin"], c=REGIME_COLORS[w],
                       s=55, alpha=0.85, label=w, edgecolors="white", linewidth=0.5, zorder=5)
        pw = int(sub["pinn_wins"].sum()); n = len(sub)
        tm = trim_mean(sub["imp_pct"].values, 0.1) if n >= 10 else sub["imp_pct"].mean()
        try: _, pval = wilcoxon(sub["mae_pin"], sub["mae_ari"])
        except: pval = np.nan
        clr = "#1565C0" if pw > n / 2 else "#C62828"
        ax.text(0.03, 0.97, f"PINN wins: {pw}/{n} ({pw/n*100:.0f}%)\nTrimmed mean: {tm:+.1f}%\np = {pval:.4f}",
                transform=ax.transAxes, fontsize=10, va="top", fontweight="bold", color=clr,
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
        ax.set_xlabel("ARIMA MAE"); ax.set_ylabel("PINN MAE")
        ax.set_title(f"{h}-day forecast", fontsize=13, fontweight="bold")
        ax.legend(fontsize=8, loc="lower right"); ax.set_xlim(0, lim); ax.set_ylim(0, lim); ax.grid(alpha=0.15)
    fig.suptitle("PINN vs ARIMA - 18 Cities, 5 Epidemic Regimes", fontsize=14, fontweight="bold")
    fig.tight_layout(); fig.savefig(OUT_DIR / "fig1_scatter.png", bbox_inches="tight"); plt.close()
    print("  Fig 1: scatter")

    # ---- FIG 2: Regime bars ----
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
    for ci, h in enumerate([7, 14]):
        ax = axes[ci]; sub = val[val["horizon"] == h]
        regimes = [w for w in REGIME_ORDER if w in sub["window"].unique()]
        x = np.arange(len(regimes)); width = 0.35
        pm = [sub[sub["window"] == w]["mae_pin"].mean() for w in regimes]
        am = [sub[sub["window"] == w]["mae_ari"].mean() for w in regimes]
        pse = [sub[sub["window"] == w]["mae_pin"].sem() for w in regimes]
        ase = [sub[sub["window"] == w]["mae_ari"].sem() for w in regimes]
        ax.bar(x - width/2, pm, width, label="PINN", color="#1565C0", alpha=0.85, yerr=pse, capsize=3)
        ax.bar(x + width/2, am, width, label="ARIMA", color="#E65100", alpha=0.85, yerr=ase, capsize=3)
        for i, w in enumerate(regimes):
            ws = sub[sub["window"] == w]
            try: _, p = wilcoxon(ws["mae_pin"], ws["mae_ari"])
            except: p = 1.0
            if p < 0.05:
                ymax = max(pm[i], am[i]) * 1.12
                stars = "***" if p < 0.001 else "**" if p < 0.01 else "*"
                ax.text(i, ymax, stars, ha="center", fontsize=14, color="gold", fontweight="bold")
        ax.set_xticks(x); ax.set_xticklabels(regimes, fontsize=10, rotation=15)
        ax.set_ylabel("MAE (mean +/- SEM)"); ax.set_title(f"{h}-day forecast", fontsize=13, fontweight="bold")
        ax.legend(fontsize=10); ax.grid(alpha=0.15, axis="y")
    fig.suptitle("PINN vs ARIMA by Epidemic Regime (18 Cities)", fontsize=14, fontweight="bold")
    fig.tight_layout(); fig.savefig(OUT_DIR / "fig2_regime_bars.png", bbox_inches="tight"); plt.close()
    print("  Fig 2: regime bars")

    # ---- FIG 3: Heatmaps ----
    for h in [7, 14]:
        sub = val[val["horizon"] == h]
        cities = sorted(sub["city"].unique())
        regimes = [w for w in REGIME_ORDER if w in sub["window"].unique()]
        heat = np.full((len(cities), len(regimes)), np.nan)
        for i, city in enumerate(cities):
            for j, w in enumerate(regimes):
                row = sub[(sub["city"] == city) & (sub["window"] == w)]
                if len(row) > 0: heat[i, j] = row["imp_pct"].values[0]
        fig, ax = plt.subplots(figsize=(max(9, len(regimes)*2), max(6, len(cities)*0.45)))
        fv = heat[np.isfinite(heat)]
        vmax = min(80, np.percentile(np.abs(fv), 95)) if len(fv) > 0 else 50
        im = ax.imshow(heat, cmap="RdBu", aspect="auto", vmin=-vmax, vmax=vmax)
        ax.set_xticks(range(len(regimes))); ax.set_xticklabels(regimes, fontsize=11, rotation=15)
        ax.set_yticks(range(len(cities))); ax.set_yticklabels(cities, fontsize=10)
        for i in range(len(cities)):
            for j in range(len(regimes)):
                v = heat[i, j]
                if np.isfinite(v):
                    c = "white" if abs(v) > vmax*0.5 else "black"
                    ax.text(j, i, f"{v:+.0f}%", ha="center", va="center", fontsize=8, color=c, fontweight="bold")
        cbar = plt.colorbar(im, ax=ax, shrink=0.85)
        cbar.set_label("PINN improvement over ARIMA (%)\nBlue=PINN better  Red=ARIMA better", fontsize=9)
        ax.set_title(f"PINN vs ARIMA: % Improvement ({h}-day, 18 cities)", fontsize=13, fontweight="bold")
        fig.tight_layout(); fig.savefig(OUT_DIR / f"fig3_heatmap_{h}d.png", bbox_inches="tight"); plt.close()
    print("  Fig 3: heatmaps (7d + 14d)")

    # ---- FIG 4: Parameter forest plots ----
    derived = [("incubation_days","Incubation (days)"), ("infectious_days","Infectious (days)"),
               ("ward_days","Ward stay (days)"), ("icu_days","ICU stay (days)"),
               ("behavioral_waning_days","Behavioral waning (days)"), ("biological_waning_days","Immunity waning (days)")]
    fig, axes = plt.subplots(2, 3, figsize=(16, 9)); axes = axes.flatten()
    for i, (col, label) in enumerate(derived):
        ax = axes[i]
        cs = punc.sort_values(f"{col}_mean")["city"].values
        means = [float(punc[punc["city"]==c][f"{col}_mean"].iloc[0]) for c in cs]
        lo = [float(punc[punc["city"]==c][f"{col}_q025"].iloc[0]) for c in cs]
        hi = [float(punc[punc["city"]==c][f"{col}_q975"].iloc[0]) for c in cs]
        y = np.arange(len(cs))
        ax.barh(y, means, color="#1565C0", alpha=0.7, height=0.7)
        ax.errorbar(means, y, xerr=[np.array(means)-np.array(lo), np.array(hi)-np.array(means)],
                    fmt="none", ecolor="black", capsize=2, lw=1)
        ax.set_yticks(y); ax.set_yticklabels(cs, fontsize=8)
        ax.set_xlabel(label, fontsize=10); ax.grid(alpha=0.15, axis="x")
        ax.set_title(label, fontsize=11, fontweight="bold")
    fig.suptitle("Epidemiological Parameters by City (mean +/- 95% CI, 10-seed ensemble)", fontsize=13, fontweight="bold")
    fig.tight_layout(); fig.savefig(OUT_DIR / "fig4_parameters.png", bbox_inches="tight"); plt.close()
    print("  Fig 4: parameters")

    # ---- FIG 5: Variant multipliers ----
    fig, axes = plt.subplots(1, 4, figsize=(18, 5))
    for i, variant in enumerate(["Delta", "Omicron", "BA.5", "XBB"]):
        ax = axes[i]; vs = vmul[vmul["variant"]==variant].sort_values("learned_mean")
        if vs.empty: ax.axis("off"); continue
        y = np.arange(len(vs))
        ax.barh(y, vs["learned_mean"].values, color="#1565C0", alpha=0.7, height=0.7)
        ax.errorbar(vs["learned_mean"].values, y,
                    xerr=[vs["learned_mean"].values-vs["learned_q025"].values,
                          vs["learned_q975"].values-vs["learned_mean"].values],
                    fmt="none", ecolor="black", capsize=2, lw=1)
        ax.axvline(vs["prior"].iloc[0], color="red", ls="--", lw=2, label=f"Prior={vs['prior'].iloc[0]:.1f}")
        ax.set_yticks(y); ax.set_yticklabels(vs["city"].values, fontsize=8)
        ax.set_xlabel("Multiplier"); ax.set_title(variant, fontsize=12, fontweight="bold")
        ax.legend(fontsize=9); ax.grid(alpha=0.15, axis="x")
    fig.suptitle("Variant Transmissibility Multipliers by City (mean +/- 95% CI)", fontsize=13, fontweight="bold")
    fig.tight_layout(); fig.savefig(OUT_DIR / "fig5_variants.png", bbox_inches="tight"); plt.close()
    print("  Fig 5: variants")

    # ---- FIG 6: City profiles (7d) ----
    h7 = val[val["horizon"]==7]
    city_order = h7.groupby("city")["imp_pct"].mean().sort_values(ascending=False).index.tolist()
    ncols = 4; nrows = max(1, (len(city_order)+ncols-1)//ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*4.2, nrows*3.2)); axes = np.array(axes).flatten()
    for i, city in enumerate(city_order):
        ax = axes[i]; cs = h7[h7["city"]==city]
        rp = [w for w in REGIME_ORDER if w in cs["window"].values]; x = np.arange(len(rp))
        pv = [float(cs[cs["window"]==w]["mae_pin"].values[0]) for w in rp]
        av = [float(cs[cs["window"]==w]["mae_ari"].values[0]) for w in rp]
        ax.bar(x-0.2, pv, 0.35, color="#1565C0", alpha=0.85, label="PINN")
        ax.bar(x+0.2, av, 0.35, color="#E65100", alpha=0.85, label="ARIMA")
        mean_imp = cs["imp_pct"].mean()
        ax.set_title(f"{city} ({mean_imp:+.1f}%)", fontsize=10, fontweight="bold",
                     color="#1565C0" if mean_imp > 0 else "#C62828")
        ax.set_xticks(x); ax.set_xticklabels([w[:5] for w in rp], fontsize=7, rotation=30)
        if i == 0: ax.legend(fontsize=7)
        ax.grid(alpha=0.15, axis="y")
    for i in range(len(city_order), len(axes)): axes[i].axis("off")
    fig.suptitle("PINN vs ARIMA by City (7-day MAE)", fontsize=13, fontweight="bold")
    fig.tight_layout(); fig.savefig(OUT_DIR / "fig6_city_profiles.png", bbox_inches="tight"); plt.close()
    print("  Fig 6: city profiles")

    # ---- FIG 7: Parameter boxplots from all runs ----
    derived_raw = [("incubation_days","Incubation (days)"), ("infectious_days","Infectious (days)"),
                   ("ward_days","Ward stay (days)"), ("icu_days","ICU stay (days)"),
                   ("behavioral_waning_days","Behavioral waning (days)"), ("biological_waning_days","Immunity waning (days)")]
    cs_sorted = sorted(pall["city"].unique())
    fig, axes = plt.subplots(2, 3, figsize=(16, 9)); axes = axes.flatten()
    for i, (col, label) in enumerate(derived_raw):
        ax = axes[i]
        data = [pall[pall["city"]==c][col].values for c in cs_sorted]
        bp = ax.boxplot(data, vert=False, patch_artist=True,
                        boxprops=dict(facecolor="#1565C0", alpha=0.5),
                        medianprops=dict(color="red", lw=2))
        ax.set_yticks(range(1, len(cs_sorted)+1)); ax.set_yticklabels(cs_sorted, fontsize=8)
        ax.set_xlabel(label); ax.set_title(label, fontsize=11, fontweight="bold"); ax.grid(alpha=0.15, axis="x")
    fig.suptitle("Parameter Distributions (10-seed ensemble per city)", fontsize=13, fontweight="bold")
    fig.tight_layout(); fig.savefig(OUT_DIR / "fig7_param_boxplots.png", bbox_inches="tight"); plt.close()
    print("  Fig 7: parameter boxplots")

    # ---- FIG 8: Variant boxplots from all runs ----
    fig, axes = plt.subplots(1, 4, figsize=(18, 5.5))
    for i, variant in enumerate(["Delta", "Omicron", "BA.5", "XBB"]):
        ax = axes[i]; vs = vall[vall["variant"]==variant]
        if vs.empty: ax.axis("off"); continue
        cs_v = sorted(vs["city"].unique())
        data = [vs[vs["city"]==c]["learned"].values for c in cs_v]
        bp = ax.boxplot(data, vert=False, patch_artist=True,
                        boxprops=dict(facecolor="#1565C0", alpha=0.5),
                        medianprops=dict(color="red", lw=2))
        ax.axvline(vs["prior"].iloc[0], color="red", ls="--", lw=2, label=f"Prior={vs['prior'].iloc[0]:.1f}")
        ax.set_yticks(range(1, len(cs_v)+1)); ax.set_yticklabels(cs_v, fontsize=8)
        ax.set_xlabel("Multiplier"); ax.set_title(variant, fontsize=12, fontweight="bold")
        ax.legend(fontsize=9); ax.grid(alpha=0.15, axis="x")
    fig.suptitle("Variant Multiplier Distributions (10 seeds per city)", fontsize=13, fontweight="bold")
    fig.tight_layout(); fig.savefig(OUT_DIR / "fig8_variant_boxplots.png", bbox_inches="tight"); plt.close()
    print("  Fig 8: variant boxplots")

    print(f"\nAll figures saved to: {OUT_DIR}")


# ======================== MAIN ========================
def main():
    print("=" * 70)
    print("PUBLICATION ANALYSIS - Multi-City PINN SUEIHCDR")
    print("=" * 70)

    val, fit, par, punc, vmul, pall, vall = load_all()
    print_tables(val, fit, par, punc, vmul, pall, vall)
    make_figures(val, fit, par, punc, vmul, pall, vall)

    print(f"\n{'=' * 70}")
    print(f"DONE. All outputs in: {OUT_DIR}")
    print(f"{'=' * 70}")


if __name__ == "__main__":
    main()


PUBLICATION ANALYSIS - Multi-City PINN SUEIHCDR
Loaded 164 evaluations, 18 cities
  Tier 1 (fit MAE < 365): 7 cities = ['Berlin', 'Denver', 'NewYork', 'SanDiego', 'SanFrancisco', 'SaoPaulo', 'Seattle']
  Tier 2 (fit MAE >= 365): 11 cities = ['Chicago', 'Houston', 'London', 'LosAngeles', 'Miami', 'Moscow', 'Paris', 'Phoenix', 'Rome', 'Sydney', 'Tokyo']

TABLE 1 - Model fit quality (full timeline, 18 cities)
  City             Days    Fit MAE  Fit MAPE%  Tier
  Berlin            602       42.9      46.4%     1
  SaoPaulo          938       99.2     256.2%     1
  SanFrancisco     1265      126.4     556.4%     1
  Denver           1265      130.4     868.7%     1
  SanDiego         1265      270.7     284.9%     1
  Seattle          1265      328.8    1354.2%     1
  NewYork          1265      361.1    1114.2%     1
  Houston          1265      884.7    8910.2%     2
  Chicago          1265      900.2   20572.0%     2
  Paris             796      903.9      90.2%     2
  Phoenix         

In [7]:
from __future__ import annotations

from pathlib import Path
import pandas as pd
import numpy as np

# -------------------------
# SETTINGS (edit these)
# -------------------------
BASE_DIR = Path(r"C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\CODES_04032021")
FOLDER_PREFIX = "parameter_uncertainty_"

# If each city folder has a consistent filename, set it here (recommended).
# Otherwise leave as None and the script will pick the first *.csv/tsv/txt it finds.
RESULT_FILENAME = None  # e.g. "parameter_uncertainty_results.csv"

# Columns that are not parameters (will be kept as metadata)
META_COLS = {"run_id", "seed", "elapsed_sec"}

# -------------------------
# HELPERS
# -------------------------
def detect_sep(file_path: Path) -> str:
    """
    Detect delimiter: tab vs comma.
    Your example looks tab-separated, so this catches that.
    """
    sample = file_path.read_text(encoding="utf-8", errors="ignore").splitlines()
    sample = [ln for ln in sample if ln.strip()]
    if not sample:
        return ","
    header = sample[0]
    return "\t" if header.count("\t") >= header.count(",") else ","


def find_city_folders(base_dir: Path):
    for p in base_dir.iterdir():
        if p.is_dir() and p.name.startswith(FOLDER_PREFIX):
            city = p.name[len(FOLDER_PREFIX):].strip()
            if city:
                yield city, p


def find_result_file(city_folder: Path) -> Path:
    if RESULT_FILENAME is not None:
        f = city_folder / RESULT_FILENAME
        if not f.exists():
            raise FileNotFoundError(f"Expected {f} but it does not exist.")
        return f

    # otherwise, pick the first plausible file
    candidates = []
    for ext in (".csv", ".tsv", ".txt"):
        candidates.extend(sorted(city_folder.glob(f"*{ext}")))

    if not candidates:
        raise FileNotFoundError(f"No .csv/.tsv/.txt found in {city_folder}")

    # choose the biggest file (often the actual results)
    candidates = sorted(candidates, key=lambda x: x.stat().st_size, reverse=True)
    return candidates[0]


def read_city_results(city: str, folder: Path) -> pd.DataFrame:
    f = find_result_file(folder)
    sep = detect_sep(f)
    df = pd.read_csv(f, sep=sep)

    # normalize column names (strip spaces)
    df.columns = [c.strip() for c in df.columns]
    df["city"] = city
    df["source_file"] = str(f)
    return df


def summarize_uncertainty(df_all: pd.DataFrame) -> pd.DataFrame:
    # parameters are numeric columns excluding meta + city/source
    exclude = META_COLS | {"city", "source_file"}
    param_cols = [c for c in df_all.columns if c not in exclude and pd.api.types.is_numeric_dtype(df_all[c])]

    # long format
    df_long = df_all.melt(
        id_vars=["city"] + [c for c in META_COLS if c in df_all.columns],
        value_vars=param_cols,
        var_name="parameter",
        value_name="value",
    )

    def q(x, p):
        return np.nanquantile(x, p)

    summary = (
        df_long.groupby(["city", "parameter"])["value"]
        .agg(
            n="count",
            mean="mean",
            sd="std",
            min="min",
            p2_5=lambda x: q(x, 0.025),
            median="median",
            p97_5=lambda x: q(x, 0.975),
            max="max",
        )
        .reset_index()
    )
    summary["cv"] = summary["sd"] / summary["mean"]  # coefficient of variation
    summary["ci95_width"] = summary["p97_5"] - summary["p2_5"]
    return df_long, summary


def main():
    # 1) load all cities
    dfs = []
    for city, folder in find_city_folders(BASE_DIR):
        try:
            df_city = read_city_results(city, folder)
            dfs.append(df_city)
            print(f"Loaded {city}: {len(df_city)} rows")
        except Exception as e:
            print(f"[SKIP] {city} in {folder}: {e}")

    if not dfs:
        raise RuntimeError("No city results were loaded. Check BASE_DIR / folder names / filenames.")

    df_all = pd.concat(dfs, ignore_index=True)

    # 2) summarize
    df_long, df_summary = summarize_uncertainty(df_all)

    # 3) save outputs
    out_all = BASE_DIR / "all_city_samples_wide.csv"
    out_long = BASE_DIR / "all_samples_long.csv"
    out_sum = BASE_DIR / "uncertainty_summary_by_city.csv"

    df_all.to_csv(out_all, index=False)
    df_long.to_csv(out_long, index=False)
    df_summary.to_csv(out_sum, index=False)

    print("\nSaved:")
    print(f" - {out_all}")
    print(f" - {out_long}")
    print(f" - {out_sum}")

    # quick preview
    print("\nPreview (summary):")
    print(df_summary.head(20))


if __name__ == "__main__":
    main()


Loaded analysis: 10 rows
Loaded analysis_LA: 10 rows
Loaded analysis_NY: 10 rows
Loaded analysis_SF: 10 rows
Loaded Berlin: 10 rows
Loaded Chicago: 10 rows
Loaded Denver: 10 rows
Loaded Houston: 10 rows
Loaded London: 10 rows
Loaded LosAngeles: 10 rows
Loaded Miami: 10 rows
Loaded Moscow: 10 rows
Loaded NewYork: 10 rows
Loaded Paris: 10 rows
Loaded Phoenix: 10 rows
Loaded Rome: 10 rows
Loaded SanDiego: 10 rows
Loaded SanFrancisco: 10 rows
Loaded SaoPaulo: 10 rows
Loaded Seattle: 10 rows
Loaded Sydney: 10 rows
Loaded Tokyo: 10 rows

Saved:
 - C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\CODES_04032021\all_city_samples_wide.csv
 - C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\CODES_04032021\all_samples_long.csv
 - C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\CODES_04032021\uncertainty_summary_by_city.csv

Preview (summary):
       city               parameter   n       mean         sd        min  \
0    Berlin  behavioral_waning_days  10  73.855617  34.971176

In [10]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
PUBLICATION ANALYSIS — Multi-City PINN SUEIHCDR (18 cities)
============================================================
Models: PINN, ARIMA, LogLinear, BayesPoly (Hybrid/Ensemble dropped)
All 18 cities included. Sensitivity analysis for good-fit subset.

Generates: 9 figures + 9 CSV tables in publication_figures/

Usage:  python publication_analysis.py
"""

import os, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import wilcoxon, trim_mean, spearmanr

warnings.filterwarnings("ignore")

# ======================== CONFIGURATION ========================
DATA_DIR = Path(r"C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\CODES_04032021\Resultados_Cidades_02152026")
OUT_DIR  = DATA_DIR / "publication_figures"
OUT_DIR.mkdir(exist_ok=True, parents=True)

REGIME_ORDER  = ["FirstWave", "Winter20", "Delta", "Omicron", "BA5_Waning"]
REGIME_COLORS = {"FirstWave":"#42A5F5","Winter20":"#FFA726","Delta":"#66BB6A",
                 "Omicron":"#EF5350","BA5_Waning":"#AB47BC"}

META = {
    "SanDiego":{"pop":3.3e6,"type":"US"},      "SanFrancisco":{"pop":0.87e6,"type":"US"},
    "LosAngeles":{"pop":10e6,"type":"US"},      "Seattle":{"pop":2.3e6,"type":"US"},
    "Denver":{"pop":0.7e6,"type":"US"},         "Chicago":{"pop":5.2e6,"type":"US"},
    "Houston":{"pop":4.7e6,"type":"US"},        "Miami":{"pop":2.7e6,"type":"US"},
    "Phoenix":{"pop":4.4e6,"type":"US"},        "NewYork":{"pop":8.3e6,"type":"US"},
    "London":{"pop":9e6,"type":"Intl"},         "Rome":{"pop":2.87e6,"type":"Intl"},
    "Berlin":{"pop":3.7e6,"type":"Intl"},       "Paris":{"pop":2.16e6,"type":"Intl"},
    "Moscow":{"pop":12.6e6,"type":"Intl"},      "Tokyo":{"pop":14e6,"type":"Intl"},
    "SaoPaulo":{"pop":12.3e6,"type":"Intl"},    "Sydney":{"pop":5.3e6,"type":"Intl"},
}

TIER1_THRESHOLD = 365   # natural gap in fit MAE (362 -> 885)

plt.rcParams.update({"figure.dpi":250,"font.size":11,"font.family":"sans-serif",
                      "axes.spines.top":False,"axes.spines.right":False})


# ======================== HELPERS ========================
def find_csv(name):
    for d in [DATA_DIR, DATA_DIR / "MASTER_TABLES"]:
        p = d / name
        if p.exists(): return pd.read_csv(p)
    raise FileNotFoundError(f"{name} not in {DATA_DIR}")

def paired_test(pinn, arima):
    n = len(pinn); wins = int((pinn < arima).sum())
    imp = (arima - pinn) / arima * 100
    try:    _, pw = wilcoxon(pinn, arima)
    except: pw = np.nan
    tm = trim_mean(imp, 0.10) if n >= 5 else np.mean(imp)
    sig = "***" if pw<0.001 else "**" if pw<0.01 else "*" if pw<0.05 else "ns"
    return {"n":n,"wins":wins,"win_pct":round(wins/n*100,1),
            "mean_imp":round(np.mean(imp),1),"trim_imp":round(tm,1),
            "p":round(pw,4) if pd.notna(pw) else np.nan,"sig":sig}


# ======================== DATA ========================
def load_all():
    val  = find_csv("MASTER_regime_validation_all_rows.csv")
    fit  = find_csv("MASTER_fit_metrics_from_compartments.csv")
    par  = find_csv("MASTER_parameters_final_by_city.csv")
    punc = find_csv("MASTER_parameter_uncertainty_summary.csv")
    vmul = find_csv("MASTER_variant_multiplier_summary.csv")
    pall = find_csv("MASTER_parameter_uncertainty_all_runs.csv")
    vall = find_csv("MASTER_variant_multiplier_all_runs.csv")

    val = val[val["mae_pin"].notna() & np.isfinite(val["mae_pin"]) & (val["mae_ari"]>0.1)].copy()
    val["imp_pct"]   = (val["mae_ari"]-val["mae_pin"])/val["mae_ari"]*100
    val["pinn_wins"] = val["mae_pin"] < val["mae_ari"]
    val["fit_mae"]   = val["city"].map(fit.set_index("city")["fit_mae_7d"])
    val["tier1"]     = val["fit_mae"] < TIER1_THRESHOLD
    for k in ["pop","type"]:
        val[k] = val["city"].map(lambda c,k=k: META.get(c,{}).get(k))

    print(f"Loaded {len(val)} evaluations, {val['city'].nunique()} cities")
    t1 = sorted(val[val["tier1"]]["city"].unique())
    t2 = sorted(val[~val["tier1"]]["city"].unique())
    print(f"  Tier 1 ({len(t1)}): {t1}")
    print(f"  Tier 2 ({len(t2)}): {t2}")
    return val, fit, par, punc, vmul, pall, vall


# ======================== TABLES ========================
def print_tables(val, fit, par, punc, vmul, pall, vall):
    S = "=" * 90
    tier1 = val[val["tier1"]]

    # TABLE 1 — Fit quality
    print(f"\n{S}\nTABLE 1 — Model fit quality\n{S}")
    fs = fit.sort_values("fit_mae_7d")
    print(f"  {'City':<15} {'Days':>5} {'Fit MAE':>10} {'Fit MAPE%':>10} {'Tier':>5}")
    for _,r in fs.iterrows():
        t = "1" if r["fit_mae_7d"]<TIER1_THRESHOLD else "2"
        print(f"  {r['city']:<15} {r['T_days']:>5} {r['fit_mae_7d']:>10.1f} {r['fit_mape_7d_pct']:>9.1f}% {t:>5}")
    fs.to_csv(OUT_DIR/"table1_fit_quality.csv", index=False)

    # TABLE 2 — Overall
    print(f"\n{S}\nTABLE 2 — Overall PINN vs ARIMA\n{S}")
    hdr = f"  {'Subset':<35} {'n':>3} {'Wins':>7} {'Mean%':>8} {'Trim%':>8} {'p':>8} {'Sig':>4}"
    print(hdr); rows2=[]
    for h in [7,14]:
        for label,sub in [("All 18 cities",val[val["horizon"]==h]),
                          ("Tier 1 (sensitivity)",tier1[tier1["horizon"]==h])]:
            if len(sub)<3: continue
            r = paired_test(sub["mae_pin"].values,sub["mae_ari"].values)
            r["label"]=f"{label} {h}d"; rows2.append(r)
            print(f"  {r['label']:<35} {r['n']:>3} {r['wins']:>3}/{r['n']:<3} "
                  f"{r['mean_imp']:>+7.1f}% {r['trim_imp']:>+7.1f}% {r['p']:>7.4f} {r['sig']:>4}")
    pd.DataFrame(rows2).to_csv(OUT_DIR/"table2_overall.csv",index=False)

    # TABLE 3 — By regime
    print(f"\n{S}\nTABLE 3 — PINN vs ARIMA by regime\n{S}")
    print(f"  {'H':>2} {'Regime':<15} {'n':>3} {'Wins':>7} {'Mean%':>8} {'Trim%':>8} {'p':>8} {'Sig':>4}")
    rows3=[]
    for h in [7,14]:
        for w in REGIME_ORDER:
            sub=val[(val["horizon"]==h)&(val["window"]==w)]
            if len(sub)<3: continue
            r=paired_test(sub["mae_pin"].values,sub["mae_ari"].values)
            r.update({"horizon":h,"regime":w}); rows3.append(r)
            print(f"  {h:>2} {w:<15} {r['n']:>3} {r['wins']:>3}/{r['n']:<3} "
                  f"{r['mean_imp']:>+7.1f}% {r['trim_imp']:>+7.1f}% {r['p']:>7.4f} {r['sig']:>4}")
    pd.DataFrame(rows3).to_csv(OUT_DIR/"table3_regime.csv",index=False)

    # TABLE 4 — By city (7d)
    print(f"\n{S}\nTABLE 4 — PINN vs ARIMA by city (7-day)\n{S}")
    h7=val[val["horizon"]==7]; rows4=[]
    for city in sorted(h7["city"].unique()):
        cs=h7[h7["city"]==city]; pw=int(cs["pinn_wins"].sum()); n=len(cs)
        m=META.get(city,{})
        rows4.append({"city":city,"type":m.get("type",""),"pop_M":m.get("pop",0)/1e6,
            "n":n,"wins":pw,"win_pct":pw/n*100,"mean_imp":cs["imp_pct"].mean(),
            "fit_mae":cs["fit_mae"].iloc[0]})
    cdf=pd.DataFrame(rows4).sort_values("mean_imp",ascending=False)
    print(f"  {'City':<15} {'Type':<5} {'Pop':>5} {'n':>3} {'Wins':>7} {'Mean%':>8} {'FitMAE':>8}")
    for _,r in cdf.iterrows():
        tag="+" if r["mean_imp"]>0 else " "
        print(f"  {tag}{r['city']:<14} {r['type']:<5} {r['pop_M']:>4.1f}M {r['n']:>3} "
              f"{r['wins']:.0f}/{r['n']:.0f}  {r['mean_imp']:>+7.1f}% {r['fit_mae']:>7.1f}")
    cdf.to_csv(OUT_DIR/"table4_city_7d.csv",index=False)

    # =====================================================================
    # TABLE 5 — FULL PARAMETER IDENTIFIABILITY (the big table)
    # =====================================================================
    print(f"\n{S}\nTABLE 5 — Parameter identifiability across 18 cities\n{S}")

    # Raw parameters
    raw_params = [
        ("beta0", "beta_0",  "Base transmission rate"),
        ("sigma", "sigma",   "Incubation rate (1/days)"),
        ("delta", "delta",   "Infectious rate (1/days)"),
        ("zeta",  "zeta",    "Hospitalization rate"),
        ("epsi",  "epsilon", "ICU progression rate"),
        ("m",     "m",       "Mild case fraction"),
        ("c",     "c",       "ICU fraction (of hospitalized)"),
        ("omega", "omega",   "Behavioral waning rate"),
        ("eta",   "eta",     "Immunity waning rate"),
        ("rho",   "rho",     "Reporting ratio"),
    ]
    # Derived timescales
    derived_params = [
        ("incubation_days",        "1/sigma", "Incubation period (days)"),
        ("infectious_days",        "1/delta", "Infectious period (days)"),
        ("ward_days",              "1/zeta",  "Hospital ward stay (days)"),
        ("icu_days",               "1/epsi",  "ICU stay (days)"),
        ("behavioral_waning_days", "1/omega", "Behavioral waning (days)"),
        ("biological_waning_days", "1/eta",   "Immunity waning (days)"),
    ]

    ident_rows = []
    print(f"\n  {'Symbol':<10} {'Meaning':<30} {'Cross-city':>12} {'Cross-city':>12} "
          f"{'Within-city':>12} {'Identif.':>10}")
    print(f"  {'':<10} {'':<30} {'mean':>12} {'CV%':>12} {'mean relW%':>12} {'class':>10}")

    for col, sym, meaning in raw_params:
        means  = punc[f"{col}_mean"]
        widths = punc[f"{col}_q975"] - punc[f"{col}_q025"]
        cv  = means.std() / means.mean() * 100
        rw  = widths.mean() / means.mean() * 100
        # Classification
        if cv < 5 and rw < 15:   cls = "STRONG"
        elif cv < 10 and rw < 40: cls = "MODERATE"
        else:                     cls = "WEAK"
        ident_rows.append({"symbol":sym,"meaning":meaning,"type":"raw",
            "cross_city_mean":f"{means.mean():.5f}","cross_city_CV":f"{cv:.1f}%",
            "within_city_relW":f"{rw:.1f}%","classification":cls})
        print(f"  {sym:<10} {meaning:<30} {means.mean():>12.5f} {cv:>11.1f}% {rw:>11.1f}% {cls:>10}")

    print()
    for col, sym, meaning in derived_params:
        means  = punc[f"{col}_mean"]
        widths = punc[f"{col}_q975"] - punc[f"{col}_q025"]
        cv  = means.std() / means.mean() * 100
        rw  = widths.mean() / means.mean() * 100
        if cv < 5 and rw < 15:   cls = "STRONG"
        elif cv < 10 and rw < 40: cls = "MODERATE"
        else:                     cls = "WEAK"
        ident_rows.append({"symbol":sym,"meaning":meaning,"type":"derived",
            "cross_city_mean":f"{means.mean():.1f}","cross_city_CV":f"{cv:.1f}%",
            "within_city_relW":f"{rw:.1f}%","classification":cls})
        print(f"  {sym:<10} {meaning:<30} {means.mean():>12.1f} {cv:>11.1f}% {rw:>11.1f}% {cls:>10}")

    pd.DataFrame(ident_rows).to_csv(OUT_DIR/"table5_parameter_identifiability.csv",index=False)

    # Summary
    strong   = [r for r in ident_rows if r["classification"]=="STRONG"]
    moderate = [r for r in ident_rows if r["classification"]=="MODERATE"]
    weak     = [r for r in ident_rows if r["classification"]=="WEAK"]
    print(f"\n  STRONG  ({len(strong)}):  {', '.join(r['symbol'] for r in strong)}")
    print(f"  MODERATE({len(moderate)}): {', '.join(r['symbol'] for r in moderate)}")
    print(f"  WEAK    ({len(weak)}):  {', '.join(r['symbol'] for r in weak)}")

    # TABLE 5b — Per-city detail for paper supplementary
    print(f"\n{S}\nTABLE 5b — Per-city parameter estimates (mean [95% CI])\n{S}")
    print(f"  {'City':<14}", end="")
    for _,sym,_ in raw_params: print(f" {sym:>12}", end="")
    print()
    for _,row in punc.iterrows():
        city = row["city"]
        print(f"  {city:<14}", end="")
        for col,_,_ in raw_params:
            m = row[f"{col}_mean"]; lo = row[f"{col}_q025"]; hi = row[f"{col}_q975"]
            print(f" {m:>5.3f}[{hi-lo:.3f}]", end="")
        print()
    punc.to_csv(OUT_DIR/"table5b_per_city_params.csv",index=False)

    # TABLE 6 — Variant multipliers
    print(f"\n{S}\nTABLE 6 — Variant multipliers\n{S}")
    print(f"  {'Variant':<10} {'Prior':>6} {'Learned':>8} {'SD':>6} {'95% CI':>18} {'Ratio':>8}")
    for variant in ["Delta","Omicron","BA.5","XBB"]:
        vs=vmul[vmul["variant"]==variant]
        if vs.empty: continue
        lm=vs["learned_mean"].mean(); ls=vs["learned_std"].mean()
        q025=vs["learned_q025"].mean(); q975=vs["learned_q975"].mean()
        prior=vs["prior"].iloc[0]
        print(f"  {variant:<10} {prior:>6.2f} {lm:>8.2f} {ls:>5.2f} "
              f"  [{q025:.2f}, {q975:.2f}] {lm/prior:>7.2f}x")
    vmul.to_csv(OUT_DIR/"table6_variants.csv",index=False)

    # TABLE 7 — Sensitivity
    print(f"\n{S}\nTABLE 7 — Sensitivity analysis\n{S}")
    print(f"  {'Subset':<40} {'n':>3} {'Wins':>7} {'Trim%':>8} {'p':>8}")
    for h in [7,14]:
        for label,sub in [
            ("All 18 cities",      val[val["horizon"]==h]),
            ("Tier 1 (good fit)",  val[(val["horizon"]==h)&(val["tier1"])]),
            ("Excluding Omicron",  val[(val["horizon"]==h)&(val["window"]!="Omicron")]),
            ("Tier 1 excl Omicron",val[(val["horizon"]==h)&(val["tier1"])&(val["window"]!="Omicron")]),
        ]:
            if len(sub)<3: continue
            r=paired_test(sub["mae_pin"].values,sub["mae_ari"].values)
            print(f"  {label+f' {h}d':<40} {r['n']:>3} {r['wins']:>3}/{r['n']:<3} "
                  f"{r['trim_imp']:>+7.1f}% {r['p']:>7.4f}")

    # Patterns
    print(f"\n{S}\nPATTERN ANALYSIS\n{S}")
    h7=val[val["horizon"]==7]
    for t in ["US","Intl"]:
        sub=h7[h7["type"]==t]; pw=int(sub["pinn_wins"].sum()); n=len(sub)
        print(f"  {t:<5} wins={pw}/{n} ({pw/n*100:.0f}%)  mean={sub['imp_pct'].mean():+.1f}%")
    ci=h7.groupby("city").agg({"imp_pct":"mean","pop":"first"}).dropna()
    rho,pval=spearmanr(ci["pop"],ci["imp_pct"])
    print(f"  Pop vs improvement: Spearman rho={rho:+.3f}, p={pval:.3f}")

    # Summary
    print(f"\n{S}\nPUBLICATION SUMMARY\n{S}")
    h7a=val[val["horizon"]==7]; h14a=val[val["horizon"]==14]
    pw7=int(h7a["pinn_wins"].sum()); n7=len(h7a)
    pw14=int(h14a["pinn_wins"].sum()); n14=len(h14a)
    _,p7=wilcoxon(h7a["mae_pin"],h7a["mae_ari"])
    print(f"""
  DATASET: 18 cities (10 US, 8 intl), 5 regimes, 2 horizons
  NO CITY-SPECIFIC TUNING

  1. 7-day:  PINN won {pw7}/{n7} ({pw7/n7*100:.0f}%), Wilcoxon p={p7:.4f}
  2. Winter20:   16/18 (89%), p=0.0003 ***
  3. BA5_Waning: 11/14 (79%), p=0.035 *
  4. 14-day: ARIMA dominates ({pw14}/{n14} PINN wins, {pw14/n14*100:.0f}%)
  5. STRONG identifiability: beta0, sigma, delta, zeta, epsi, rho
     (CV<5%, within-city rel.width<15%)
  6. WEAK identifiability: omega, eta, behavioral/biological waning
     (CV>10%, within-city rel.width>35%)
    """)


# ======================== FIGURES ========================
def make_figures(val, fit, par, punc, vmul, pall, vall):
    print("\nGenerating figures...")

    # ---- FIG 1: Scatter PINN vs ARIMA ----
    fig,axes=plt.subplots(1,2,figsize=(14,6))
    for ci,h in enumerate([7,14]):
        ax=axes[ci]; sub=val[val["horizon"]==h]
        p97=np.percentile(np.concatenate([sub["mae_pin"].values,sub["mae_ari"].values]),97)
        lim=p97*1.35
        ax.plot([0,lim],[0,lim],"k--",alpha=0.3,lw=1)
        ax.fill_between([0,lim],[0,lim],[lim,lim],alpha=0.04,color="red")
        ax.fill_between([0,lim],[0,0],[0,lim],alpha=0.04,color="blue")
        for w in REGIME_ORDER:
            ws=sub[sub["window"]==w]
            if ws.empty: continue
            ax.scatter(ws["mae_ari"],ws["mae_pin"],c=REGIME_COLORS[w],s=55,alpha=0.85,
                       label=w,edgecolors="white",linewidth=0.5,zorder=5)
        pw=int(sub["pinn_wins"].sum()); n=len(sub)
        tm=trim_mean(sub["imp_pct"].values,0.1) if n>=10 else sub["imp_pct"].mean()
        try:    _,pval=wilcoxon(sub["mae_pin"],sub["mae_ari"])
        except: pval=np.nan
        clr="#1565C0" if pw>n/2 else "#C62828"
        ax.text(0.03,0.97,f"PINN wins: {pw}/{n} ({pw/n*100:.0f}%)\nTrimmed mean: {tm:+.1f}%\np = {pval:.4f}",
                transform=ax.transAxes,fontsize=10,va="top",fontweight="bold",color=clr,
                bbox=dict(boxstyle="round,pad=0.3",facecolor="white",alpha=0.8))
        ax.set_xlabel("ARIMA MAE"); ax.set_ylabel("PINN MAE")
        ax.set_title(f"{h}-day forecast",fontsize=13,fontweight="bold")
        ax.legend(fontsize=8,loc="lower right"); ax.set_xlim(0,lim); ax.set_ylim(0,lim); ax.grid(alpha=0.15)
    fig.suptitle("PINN vs ARIMA — 18 Cities, 5 Epidemic Regimes",fontsize=14,fontweight="bold")
    fig.tight_layout(); fig.savefig(OUT_DIR/"fig1_scatter.png",bbox_inches="tight"); plt.close()
    print("  Fig 1: scatter")

    # ---- FIG 2: Regime bars ----
    fig,axes=plt.subplots(1,2,figsize=(14,5.5))
    for ci,h in enumerate([7,14]):
        ax=axes[ci]; sub=val[val["horizon"]==h]
        regimes=[w for w in REGIME_ORDER if w in sub["window"].unique()]
        x=np.arange(len(regimes)); width=0.35
        pm=[sub[sub["window"]==w]["mae_pin"].mean() for w in regimes]
        am=[sub[sub["window"]==w]["mae_ari"].mean() for w in regimes]
        pse=[sub[sub["window"]==w]["mae_pin"].sem() for w in regimes]
        ase=[sub[sub["window"]==w]["mae_ari"].sem() for w in regimes]
        ax.bar(x-width/2,pm,width,label="PINN",color="#1565C0",alpha=0.85,yerr=pse,capsize=3)
        ax.bar(x+width/2,am,width,label="ARIMA",color="#E65100",alpha=0.85,yerr=ase,capsize=3)
        for i,w in enumerate(regimes):
            ws=sub[sub["window"]==w]
            try:    _,p=wilcoxon(ws["mae_pin"],ws["mae_ari"])
            except: p=1.0
            if p<0.05:
                ymax=max(pm[i],am[i])*1.12
                stars="***" if p<0.001 else "**" if p<0.01 else "*"
                ax.text(i,ymax,stars,ha="center",fontsize=14,color="gold",fontweight="bold")
        ax.set_xticks(x); ax.set_xticklabels(regimes,fontsize=10,rotation=15)
        ax.set_ylabel("MAE (mean +/- SEM)"); ax.set_title(f"{h}-day",fontsize=13,fontweight="bold")
        ax.legend(fontsize=10); ax.grid(alpha=0.15,axis="y")
    fig.suptitle("PINN vs ARIMA by Epidemic Regime (18 Cities)",fontsize=14,fontweight="bold")
    fig.tight_layout(); fig.savefig(OUT_DIR/"fig2_regime_bars.png",bbox_inches="tight"); plt.close()
    print("  Fig 2: regime bars")

    # ---- FIG 3: Heatmaps ----
    for h in [7,14]:
        sub=val[val["horizon"]==h]; cities=sorted(sub["city"].unique())
        regimes=[w for w in REGIME_ORDER if w in sub["window"].unique()]
        heat=np.full((len(cities),len(regimes)),np.nan)
        for i,city in enumerate(cities):
            for j,w in enumerate(regimes):
                row=sub[(sub["city"]==city)&(sub["window"]==w)]
                if len(row)>0: heat[i,j]=row["imp_pct"].values[0]
        fig,ax=plt.subplots(figsize=(max(9,len(regimes)*2),max(6,len(cities)*0.45)))
        fv=heat[np.isfinite(heat)]
        vmax=min(80,np.percentile(np.abs(fv),95)) if len(fv)>0 else 50
        im=ax.imshow(heat,cmap="RdBu",aspect="auto",vmin=-vmax,vmax=vmax)
        ax.set_xticks(range(len(regimes))); ax.set_xticklabels(regimes,fontsize=11,rotation=15)
        ax.set_yticks(range(len(cities))); ax.set_yticklabels(cities,fontsize=10)
        for i in range(len(cities)):
            for j in range(len(regimes)):
                v=heat[i,j]
                if np.isfinite(v):
                    c="white" if abs(v)>vmax*0.5 else "black"
                    ax.text(j,i,f"{v:+.0f}%",ha="center",va="center",fontsize=8,color=c,fontweight="bold")
        plt.colorbar(im,ax=ax,shrink=0.85).set_label("PINN improvement (%)\nBlue=PINN better  Red=ARIMA better",fontsize=9)
        ax.set_title(f"PINN vs ARIMA: % Improvement ({h}-day)",fontsize=13,fontweight="bold")
        fig.tight_layout(); fig.savefig(OUT_DIR/f"fig3_heatmap_{h}d.png",bbox_inches="tight"); plt.close()
    print("  Fig 3: heatmaps")

    # ---- FIG 4a: Derived timescales forest plot (6 panels) ----
    derived=[("incubation_days","Incubation (days)"),("infectious_days","Infectious (days)"),
             ("ward_days","Ward stay (days)"),("icu_days","ICU stay (days)"),
             ("behavioral_waning_days","Behavioral waning (days)"),("biological_waning_days","Immunity waning (days)")]
    fig,axes=plt.subplots(2,3,figsize=(16,9)); axes=axes.flatten()
    for i,(col,label) in enumerate(derived):
        ax=axes[i]
        cs=punc.sort_values(f"{col}_mean")["city"].values
        means=[float(punc[punc["city"]==c][f"{col}_mean"].iloc[0]) for c in cs]
        lo=[float(punc[punc["city"]==c][f"{col}_q025"].iloc[0]) for c in cs]
        hi=[float(punc[punc["city"]==c][f"{col}_q975"].iloc[0]) for c in cs]
        y=np.arange(len(cs))
        ax.barh(y,means,color="#1565C0",alpha=0.7,height=0.7)
        ax.errorbar(means,y,xerr=[np.array(means)-np.array(lo),np.array(hi)-np.array(means)],
                    fmt="none",ecolor="black",capsize=2,lw=1)
        ax.set_yticks(y); ax.set_yticklabels(cs,fontsize=8)
        ax.set_xlabel(label); ax.grid(alpha=0.15,axis="x")
        ax.set_title(label,fontsize=11,fontweight="bold")
    fig.suptitle("Derived Epidemiological Timescales by City (mean +/- 95% CI)",fontsize=13,fontweight="bold")
    fig.tight_layout(); fig.savefig(OUT_DIR/"fig4a_timescales.png",bbox_inches="tight"); plt.close()
    print("  Fig 4a: derived timescales")

    # ---- FIG 4b: Raw rate parameters forest plot (10 panels) ----
    raw_params=[("beta0","beta_0"),("sigma","sigma"),("delta","delta"),("zeta","zeta"),
                ("epsi","epsilon"),("m","m"),("c","c"),("omega","omega"),("eta","eta"),("rho","rho")]
    fig,axes=plt.subplots(2,5,figsize=(22,8)); axes=axes.flatten()
    for i,(col,sym) in enumerate(raw_params):
        ax=axes[i]
        cs=punc.sort_values(f"{col}_mean")["city"].values
        means=[float(punc[punc["city"]==c][f"{col}_mean"].iloc[0]) for c in cs]
        lo=[float(punc[punc["city"]==c][f"{col}_q025"].iloc[0]) for c in cs]
        hi=[float(punc[punc["city"]==c][f"{col}_q975"].iloc[0]) for c in cs]
        y=np.arange(len(cs))
        # Color by identifiability
        cv=np.std(means)/np.mean(means)*100
        rw=np.mean(np.array(hi)-np.array(lo))/np.mean(means)*100
        if cv<5 and rw<15:   color="#1565C0"   # strong
        elif cv<10 and rw<40: color="#FFA726"   # moderate
        else:                 color="#EF5350"   # weak
        ax.barh(y,means,color=color,alpha=0.7,height=0.7)
        ax.errorbar(means,y,xerr=[np.array(means)-np.array(lo),np.array(hi)-np.array(means)],
                    fmt="none",ecolor="black",capsize=2,lw=1)
        ax.set_yticks(y); ax.set_yticklabels(cs,fontsize=7)
        ax.grid(alpha=0.15,axis="x")
        cls_label = "STRONG" if cv<5 and rw<15 else "MOD." if cv<10 and rw<40 else "WEAK"
        ax.set_title(f"{sym} [{cls_label}]\nCV={cv:.1f}% relW={rw:.1f}%",fontsize=10,fontweight="bold")
    fig.suptitle("All Raw Parameters by City (Blue=strong, Orange=moderate, Red=weak identifiability)",
                 fontsize=13,fontweight="bold")
    fig.tight_layout(); fig.savefig(OUT_DIR/"fig4b_all_parameters.png",bbox_inches="tight"); plt.close()
    print("  Fig 4b: all raw parameters")

    # ---- FIG 5: Variant multipliers ----
    fig,axes=plt.subplots(1,4,figsize=(18,5))
    for i,variant in enumerate(["Delta","Omicron","BA.5","XBB"]):
        ax=axes[i]; vs=vmul[vmul["variant"]==variant].sort_values("learned_mean")
        if vs.empty: ax.axis("off"); continue
        y=np.arange(len(vs))
        ax.barh(y,vs["learned_mean"].values,color="#1565C0",alpha=0.7,height=0.7)
        ax.errorbar(vs["learned_mean"].values,y,
                    xerr=[vs["learned_mean"].values-vs["learned_q025"].values,
                          vs["learned_q975"].values-vs["learned_mean"].values],
                    fmt="none",ecolor="black",capsize=2,lw=1)
        ax.axvline(vs["prior"].iloc[0],color="red",ls="--",lw=2,label=f"Prior={vs['prior'].iloc[0]:.1f}")
        ax.set_yticks(y); ax.set_yticklabels(vs["city"].values,fontsize=8)
        ax.set_xlabel("Multiplier"); ax.set_title(variant,fontsize=12,fontweight="bold")
        ax.legend(fontsize=9); ax.grid(alpha=0.15,axis="x")
    fig.suptitle("Variant Transmissibility Multipliers (mean +/- 95% CI)",fontsize=13,fontweight="bold")
    fig.tight_layout(); fig.savefig(OUT_DIR/"fig5_variants.png",bbox_inches="tight"); plt.close()
    print("  Fig 5: variants")

    # ---- FIG 6: City profiles (7d) ----
    h7=val[val["horizon"]==7]
    city_order=h7.groupby("city")["imp_pct"].mean().sort_values(ascending=False).index.tolist()
    ncols=4; nrows=max(1,(len(city_order)+ncols-1)//ncols)
    fig,axes=plt.subplots(nrows,ncols,figsize=(ncols*4.2,nrows*3.2)); axes=np.array(axes).flatten()
    for i,city in enumerate(city_order):
        ax=axes[i]; cs=h7[h7["city"]==city]
        rp=[w for w in REGIME_ORDER if w in cs["window"].values]; x=np.arange(len(rp))
        pv=[float(cs[cs["window"]==w]["mae_pin"].values[0]) for w in rp]
        av=[float(cs[cs["window"]==w]["mae_ari"].values[0]) for w in rp]
        ax.bar(x-0.2,pv,0.35,color="#1565C0",alpha=0.85,label="PINN")
        ax.bar(x+0.2,av,0.35,color="#E65100",alpha=0.85,label="ARIMA")
        mean_imp=cs["imp_pct"].mean()
        ax.set_title(f"{city} ({mean_imp:+.1f}%)",fontsize=10,fontweight="bold",
                     color="#1565C0" if mean_imp>0 else "#C62828")
        ax.set_xticks(x); ax.set_xticklabels([w[:5] for w in rp],fontsize=7,rotation=30)
        if i==0: ax.legend(fontsize=7)
        ax.grid(alpha=0.15,axis="y")
    for i in range(len(city_order),len(axes)): axes[i].axis("off")
    fig.suptitle("PINN vs ARIMA by City (7-day MAE)",fontsize=13,fontweight="bold")
    fig.tight_layout(); fig.savefig(OUT_DIR/"fig6_city_profiles.png",bbox_inches="tight"); plt.close()
    print("  Fig 6: city profiles")

    # ---- FIG 7: Parameter boxplots from all 10-seed runs ----
    derived_raw=[("incubation_days","Incubation (days)"),("infectious_days","Infectious (days)"),
                 ("ward_days","Ward stay (days)"),("icu_days","ICU stay (days)"),
                 ("behavioral_waning_days","Behavioral waning (days)"),("biological_waning_days","Immunity waning (days)")]
    cs_sorted=sorted(pall["city"].unique())
    fig,axes=plt.subplots(2,3,figsize=(16,9)); axes=axes.flatten()
    for i,(col,label) in enumerate(derived_raw):
        ax=axes[i]
        data=[pall[pall["city"]==c][col].values for c in cs_sorted]
        ax.boxplot(data,vert=False,patch_artist=True,
                   boxprops=dict(facecolor="#1565C0",alpha=0.5),
                   medianprops=dict(color="red",lw=2))
        ax.set_yticks(range(1,len(cs_sorted)+1)); ax.set_yticklabels(cs_sorted,fontsize=8)
        ax.set_xlabel(label); ax.set_title(label,fontsize=11,fontweight="bold"); ax.grid(alpha=0.15,axis="x")
    fig.suptitle("Parameter Distributions (10-seed ensemble per city)",fontsize=13,fontweight="bold")
    fig.tight_layout(); fig.savefig(OUT_DIR/"fig7_param_boxplots.png",bbox_inches="tight"); plt.close()
    print("  Fig 7: parameter boxplots")

    # ---- FIG 8: Variant boxplots from all runs ----
    fig,axes=plt.subplots(1,4,figsize=(18,5.5))
    for i,variant in enumerate(["Delta","Omicron","BA.5","XBB"]):
        ax=axes[i]; vs=vall[vall["variant"]==variant]
        if vs.empty: ax.axis("off"); continue
        cs_v=sorted(vs["city"].unique())
        data=[vs[vs["city"]==c]["learned"].values for c in cs_v]
        ax.boxplot(data,vert=False,patch_artist=True,
                   boxprops=dict(facecolor="#1565C0",alpha=0.5),
                   medianprops=dict(color="red",lw=2))
        ax.axvline(vs["prior"].iloc[0],color="red",ls="--",lw=2,label=f"Prior={vs['prior'].iloc[0]:.1f}")
        ax.set_yticks(range(1,len(cs_v)+1)); ax.set_yticklabels(cs_v,fontsize=8)
        ax.set_xlabel("Multiplier"); ax.set_title(variant,fontsize=12,fontweight="bold")
        ax.legend(fontsize=9); ax.grid(alpha=0.15,axis="x")
    fig.suptitle("Variant Multiplier Distributions (10 seeds per city)",fontsize=13,fontweight="bold")
    fig.tight_layout(); fig.savefig(OUT_DIR/"fig8_variant_boxplots.png",bbox_inches="tight"); plt.close()
    print("  Fig 8: variant boxplots")

    # ---- FIG 9: Identifiability summary (visual table) ----
    all_params = [
        ("beta_0",  "Transmission",  "beta0"),  ("sigma",  "Incubation rate",  "sigma"),
        ("delta",   "Infectious rate","delta"),  ("zeta",   "Hospital rate",    "zeta"),
        ("epsilon", "ICU rate",       "epsi"),   ("rho",    "Reporting ratio",  "rho"),
        ("m",       "Mild fraction",  "m"),      ("c",      "ICU fraction",     "c"),
        ("omega",   "Behav. waning",  "omega"),  ("eta",    "Immun. waning",    "eta"),
    ]
    fig,ax=plt.subplots(figsize=(12,6))
    for i,(sym,label,col) in enumerate(all_params):
        means=punc[f"{col}_mean"]
        widths=punc[f"{col}_q975"]-punc[f"{col}_q025"]
        cv=means.std()/means.mean()*100
        rw=widths.mean()/means.mean()*100
        if cv<5 and rw<15:   color,cls="#1565C0","STRONG"
        elif cv<10 and rw<40: color,cls="#FFA726","MODERATE"
        else:                 color,cls="#EF5350","WEAK"
        ax.barh(i,cv,color=color,alpha=0.7,height=0.7,label=cls if cls not in [b.get_label() for b in ax.patches] else "")
        ax.errorbar(cv,i,xerr=rw/5,fmt="d",color="black",markersize=5,capsize=3)
        ax.text(max(cv,rw/5)+1,i,f"CV={cv:.1f}% relW={rw:.1f}%",va="center",fontsize=9)
    ax.set_yticks(range(len(all_params)))
    ax.set_yticklabels([f"{sym} ({label})" for sym,label,_ in all_params],fontsize=10)
    ax.set_xlabel("Cross-city CV (%)",fontsize=11)
    ax.set_title("Parameter Identifiability Summary\nBlue=STRONG  Orange=MODERATE  Red=WEAK",
                 fontsize=13,fontweight="bold")
    ax.grid(alpha=0.15,axis="x"); ax.invert_yaxis()
    fig.tight_layout(); fig.savefig(OUT_DIR/"fig9_identifiability.png",bbox_inches="tight"); plt.close()
    print("  Fig 9: identifiability summary")

    print(f"\nAll figures saved to: {OUT_DIR}")


# ======================== MAIN ========================
def main():
    print("="*70)
    print("PUBLICATION ANALYSIS — Multi-City PINN SUEIHCDR")
    print("="*70)
    val,fit,par,punc,vmul,pall,vall = load_all()
    print_tables(val,fit,par,punc,vmul,pall,vall)
    make_figures(val,fit,par,punc,vmul,pall,vall)
    print(f"\n{'='*70}\nDONE. All outputs in: {OUT_DIR}\n{'='*70}")

if __name__=="__main__":
    main()


PUBLICATION ANALYSIS — Multi-City PINN SUEIHCDR
Loaded 164 evaluations, 18 cities
  Tier 1 (7): ['Berlin', 'Denver', 'NewYork', 'SanDiego', 'SanFrancisco', 'SaoPaulo', 'Seattle']
  Tier 2 (11): ['Chicago', 'Houston', 'London', 'LosAngeles', 'Miami', 'Moscow', 'Paris', 'Phoenix', 'Rome', 'Sydney', 'Tokyo']

TABLE 1 — Model fit quality
  City             Days    Fit MAE  Fit MAPE%  Tier
  Berlin            602       42.9      46.4%     1
  SaoPaulo          938       99.2     256.2%     1
  SanFrancisco     1265      126.4     556.4%     1
  Denver           1265      130.4     868.7%     1
  SanDiego         1265      270.7     284.9%     1
  Seattle          1265      328.8    1354.2%     1
  NewYork          1265      361.1    1114.2%     1
  Houston          1265      884.7    8910.2%     2
  Chicago          1265      900.2   20572.0%     2
  Paris             796      903.9      90.2%     2
  Phoenix          1265      968.7    3367.7%     2
  Miami            1265     1103.3   164

In [13]:
import pandas as pd

df = pd.read_csv(r"C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\CODES_04032021\Resultados_Cidades_02152026\MASTER_regime_validation_all_rows.csv")

# best baseline MAE for each row (exclude PINN)
baseline_mae_cols = ["mae_ari", "mae_bayesian", "mae_loglin", "mae_hyb"]
df["best_baseline_mae"] = df[baseline_mae_cols].min(axis=1)

# negative = PINN better; positive = PINN worse than the best baseline
df["pinn_vs_best_mae"] = df["mae_pin"] - df["best_baseline_mae"]

# pick best and worst scenario per city
rows = []
for city, g in df.groupby("city"):
    g = g.sort_values("pinn_vs_best_mae")
    best = g.iloc[0]
    worst = g.iloc[-1]
    rows.append({
        "city": city,
        "BEST_cut": best["cut_date"],
        "BEST_window": best["window"],
        "BEST_horizon": best["horizon"],
        "BEST_delta_mae": best["pinn_vs_best_mae"],
        "WORST_cut": worst["cut_date"],
        "WORST_window": worst["window"],
        "WORST_horizon": worst["horizon"],
        "WORST_delta_mae": worst["pinn_vs_best_mae"],
    })

sel = pd.DataFrame(rows).sort_values("WORST_delta_mae", ascending=False)
print(sel)
# sel.to_csv("selected_best_worst_by_city.csv", index=False)


            city    BEST_cut BEST_window  BEST_horizon  BEST_delta_mae  \
5     LosAngeles  2022-06-15  BA5_Waning             7       -8.545558   
4         London  2021-01-15    Winter20             7     -243.643259   
7         Moscow  2021-08-15       Delta             7      156.802403   
11          Rome  2021-01-15    Winter20            14       -4.062528   
3        Houston  2022-06-15  BA5_Waning            14      -27.713400   
16        Sydney  2021-01-15    Winter20             7       -0.000006   
10       Phoenix  2021-01-15    Winter20             7      -18.447786   
17         Tokyo  2021-01-15    Winter20             7       -3.136377   
6          Miami  2022-06-15  BA5_Waning             7      -51.525049   
12      SanDiego  2021-08-15       Delta            14       -2.031011   
1        Chicago  2021-08-15       Delta            14       -8.989402   
8        NewYork  2022-01-15     Omicron             7      -10.969916   
9          Paris  2021-01-15    Winter

In [14]:
import re
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.backends.backend_pdf import PdfPages

# ---------- CONFIG ----------
BASE_DIR = Path(r"C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\CODES_04032021")

# Pattern for the city-level folder (your city appears after the last underscore)
CITY_FOLDER_GLOB = "pinn_sueihcdr_multiwindow_v2_*"

# Folder inside each city folder where the plots live
PLOTS_SUBDIR = "regime_plots"

# We will only use the 14-day overlays:
FIG_GLOB_14D = "overlay_*_14d.png"

# The canonical 5 cuts you’re using
CUT_DATES = ["2020-05-01", "2021-01-15", "2021-08-15", "2022-01-15", "2022-06-15"]

# Optional labels for the cuts (matches your figure titles)
CUT_LABELS = {
    "2020-05-01": "FirstWave",
    "2021-01-15": "Winter20",
    "2021-08-15": "Delta",
    "2022-01-15": "Omicron",
    "2022-06-15": "BA5_Waning",
}

# Output
OUT_DIR = BASE_DIR / "supplement_pages_optionA"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_PDF = OUT_DIR / "Supplement_AllCities_5windows_14d.pdf"
SAVE_CITY_PNGS = True  # set False if you only want the PDF
DPI = 200
# ---------------------------


def infer_city(folder: Path) -> str:
    # e.g. pinn_sueihcdr_multiwindow_v2_SanDiego -> SanDiego
    name = folder.name
    if "_" in name:
        return name.split("_")[-1]
    return name


def find_overlay_14d(city_folder: Path, cut_date: str) -> Path | None:
    # Preferred exact filename
    p = city_folder / PLOTS_SUBDIR / f"overlay_{cut_date}_14d.png"
    if p.exists():
        return p

    # Fallback: search for any overlay that contains the cut date and ends with _14d.png
    cand = list((city_folder / PLOTS_SUBDIR).glob(f"overlay*{cut_date}*_14d.png"))
    return cand[0] if cand else None


def make_city_page(city: str, paths_by_cut: dict[str, Path | None]):
    # 5 panels stacked vertically (clean, journal-friendly)
    fig, axes = plt.subplots(nrows=5, ncols=1, figsize=(11, 16), constrained_layout=True)

    fig.suptitle(f"{city} — 14-day Forecast Overlays (5 regimes)", fontsize=16, y=0.995)

    for ax, cut in zip(axes, CUT_DATES):
        img_path = paths_by_cut.get(cut)
        label = CUT_LABELS.get(cut, cut)

        ax.axis("off")

        if img_path is None:
            ax.text(
                0.5, 0.5,
                f"Missing figure: {city} — {label} (Cut {cut})",
                ha="center", va="center", fontsize=12
            )
            continue

        img = mpimg.imread(str(img_path))
        ax.imshow(img)

        # Small caption above each panel (keeps it readable even if the image titles vary)
        ax.set_title(f"{label} | Cut: {cut} | 14-day horizon", fontsize=11, pad=6)

    return fig


def main():
    city_folders = sorted([p for p in BASE_DIR.glob(CITY_FOLDER_GLOB) if p.is_dir()])

    if not city_folders:
        raise RuntimeError(f"No city folders found with: {BASE_DIR / CITY_FOLDER_GLOB}")

    # Optional: filter out folders that don’t have regime_plots at all
    city_folders = [p for p in city_folders if (p / PLOTS_SUBDIR).exists()]

    if not city_folders:
        raise RuntimeError("Found city folders, but none contained the expected 'regime_plots' subfolder.")

    print(f"Found {len(city_folders)} city folders.")

    with PdfPages(str(OUT_PDF)) as pdf:
        for folder in city_folders:
            city = infer_city(folder)

            # Collect the 5 target images
            paths_by_cut = {cut: find_overlay_14d(folder, cut) for cut in CUT_DATES}

            # Build the page
            fig = make_city_page(city, paths_by_cut)

            # Write to PDF
            pdf.savefig(fig, dpi=DPI)
            plt.close(fig)

            # Also write per-city PNG, if desired
            if SAVE_CITY_PNGS:
                # Rebuild quickly (or you can save before close)
                fig = make_city_page(city, paths_by_cut)
                out_png = OUT_DIR / f"Supplement_{city}_5windows_14d.png"
                fig.savefig(out_png, dpi=DPI, bbox_inches="tight")
                plt.close(fig)

            print(f"[OK] {city}")

    print("\nWrote:")
    print(f" - PDF:  {OUT_PDF}")
    if SAVE_CITY_PNGS:
        print(f" - PNGs: {OUT_DIR}/*.png")


if __name__ == "__main__":
    main()


Found 18 city folders.
[OK] Berlin
[OK] Chicago
[OK] Denver
[OK] Houston
[OK] London
[OK] LosAngeles
[OK] Miami
[OK] Moscow
[OK] NewYork
[OK] Paris
[OK] Phoenix
[OK] Rome
[OK] SanDiego
[OK] SanFrancisco
[OK] SaoPaulo
[OK] Seattle
[OK] Sydney
[OK] Tokyo

Wrote:
 - PDF:  C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\CODES_04032021\supplement_pages_optionA\Supplement_AllCities_5windows_14d.pdf
 - PNGs: C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\CODES_04032021\supplement_pages_optionA/*.png


In [16]:
import os
from PIL import Image
import matplotlib.pyplot as plt

# ----------------------------
# Config
# ----------------------------
REGIMES = [
    ("FirstWave",  "2020-05-01"),
    ("Winter20",   "2021-01-15"),
    ("Delta",      "2021-08-15"),
    ("Omicron",    "2022-01-15"),
    ("BA5_Waning", "2022-06-15"),
]

def build_city_page(
    city: str,
    base_root: str,
    horizon_days: int = 14,
    out_dir: str | None = None,
    dpi: int = 200,
):
    """
    Builds ONE page per city with 5 stacked regime plots, with fixed spacing so titles don't overlap.
    Assumes individual images exist as: overlay_<cut>_<horizon>d.png inside:
      <base_root>/pinn_sueihcdr_multiwindow_v2_<city>/regime_plots
    """

    # Folder pattern you described
    city_folder = f"pinn_sueihcdr_multiwindow_v2_{city}"
    regime_dir = os.path.join(base_root, city_folder, "regime_plots")

    # Input figure paths
    img_paths = []
    for _, cut in REGIMES:
        fn = f"overlay_{cut}_{horizon_days}d.png"
        img_paths.append(os.path.join(regime_dir, fn))

    if out_dir is None:
        out_dir = regime_dir
    os.makedirs(out_dir, exist_ok=True)

    # Create a clean stacked page using matplotlib + imshow
    # (imshow avoids reflow issues from the original plotting code)
    fig, axes = plt.subplots(
        nrows=5, ncols=1,
        figsize=(10, 18),
        constrained_layout=False
    )

    # --- KEY FIXES (top spacing) ---
    # Reserve a band for the suptitle and avoid collision with subplot #1 title.
    fig.subplots_adjust(top=0.93, bottom=0.04, hspace=0.55)

    fig.suptitle(
        f"{city} — {horizon_days}-day Forecast Overlays (5 regimes)",
        fontsize=20,
        y=0.985  # push title upward into reserved band
    )

    for ax, ((regime, cut), p) in zip(axes, zip(REGIMES, img_paths)):
        ax.axis("off")

        if os.path.exists(p):
            im = Image.open(p)
            ax.imshow(im)
            ax.set_title(
                f"{regime} | Cut: {cut} | {horizon_days}-day horizon",
                fontsize=12,
                pad=12  # more breathing room from the image
            )
        else:
            ax.text(
                0.5, 0.5,
                f"Missing figure: {city} {regime} (Cut {cut})",
                ha="center", va="center", fontsize=12
            )
            ax.set_title(
                f"{regime} | Cut: {cut} | {horizon_days}-day horizon",
                fontsize=12,
                pad=12
            )

    # Keep suptitle area protected
    fig.tight_layout(rect=[0.02, 0.02, 0.98, 0.92])

    out_path = os.path.join(out_dir, f"Supplement_{city}_5windows_{horizon_days}d.png")
    fig.savefig(out_path, dpi=dpi)
    plt.close(fig)
    return out_path


# ----------------------------
# Example usage
# ----------------------------
import os
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# ----------------------------
# Config
# ----------------------------
BASE_ROOT = Path(r"C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\CODES_04032021")
CITY_FOLDER_GLOB = "pinn_sueihcdr_multiwindow_v2_*"
PLOTS_SUBDIR = "regime_plots"

# Five regimes / cuts (your standard set)
REGIMES = [
    ("FirstWave",  "2020-05-01"),
    ("Winter20",   "2021-01-15"),
    ("Delta",      "2021-08-15"),
    ("Omicron",    "2022-01-15"),
    ("BA5_Waning", "2022-06-15"),
]

HORIZON_DAYS = 14           # change to 7 if you want
DPI = 200
SAVE_CITY_PNGS = True

OUT_DIR = BASE_ROOT / "supplement_pages_fixed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_PDF = OUT_DIR / f"Supplement_AllCities_5windows_{HORIZON_DAYS}d_fixed.pdf"


# ----------------------------
# Helpers
# ----------------------------
def infer_city(folder: Path) -> str:
    """
    From: pinn_sueihcdr_multiwindow_v2_SanDiego  -> SanDiego
    """
    return folder.name.split("_")[-1]

def find_overlay_png(folder: Path, cut: str, horizon_days: int) -> Path | None:
    """
    Finds overlay_<cut>_<horizon>d.png inside <folder>/regime_plots
    """
    p = folder / PLOTS_SUBDIR / f"overlay_{cut}_{horizon_days}d.png"
    return p if p.exists() else None

def build_city_page_figure(city: str, img_paths: list[Path | None], horizon_days: int) -> plt.Figure:
    """
    Builds a stacked page using imshow (avoids layout collisions).
    """
    fig, axes = plt.subplots(nrows=5, ncols=1, figsize=(10, 18), constrained_layout=False)

    # --- KEY FIXES to stop top overlap ---
    fig.subplots_adjust(top=0.93, bottom=0.04, hspace=0.55)
    fig.suptitle(
        f"{city} — {horizon_days}-day Forecast Overlays (5 regimes)",
        fontsize=20,
        y=0.985
    )

    for ax, ((regime, cut), p) in zip(axes, zip(REGIMES, img_paths)):
        ax.axis("off")

        ax.set_title(f"{regime} | Cut: {cut} | {horizon_days}-day horizon", fontsize=12, pad=12)

        if p is None:
            ax.text(0.5, 0.5, f"Missing: overlay_{cut}_{horizon_days}d.png", ha="center", va="center")
            continue

        im = Image.open(p)
        ax.imshow(im)

    # Protect the suptitle area
    fig.tight_layout(rect=[0.02, 0.02, 0.98, 0.92])
    return fig


# ----------------------------
# Main
# ----------------------------
def main():
    city_folders = sorted([p for p in BASE_ROOT.glob(CITY_FOLDER_GLOB) if p.is_dir()])

    if not city_folders:
        raise RuntimeError(f"No city folders found with: {BASE_ROOT / CITY_FOLDER_GLOB}")

    # keep only those that have regime_plots
    city_folders = [p for p in city_folders if (p / PLOTS_SUBDIR).exists()]

    if not city_folders:
        raise RuntimeError("Found city folders, but none contained the expected 'regime_plots' subfolder.")

    print(f"Found {len(city_folders)} city folders.")

    with PdfPages(str(OUT_PDF)) as pdf:
        for folder in city_folders:
            city = infer_city(folder)

            # Collect 5 target images (in the regime order)
            img_paths = []
            for _, cut in REGIMES:
                img_paths.append(find_overlay_png(folder, cut, HORIZON_DAYS))

            # Build the page figure
            fig = build_city_page_figure(city, img_paths, HORIZON_DAYS)

            # Write to PDF
            pdf.savefig(fig, dpi=DPI)
            plt.close(fig)

            # Also save per-city PNG page if desired
            if SAVE_CITY_PNGS:
                fig2 = build_city_page_figure(city, img_paths, HORIZON_DAYS)
                out_png = OUT_DIR / f"Supplement_{city}_5windows_{HORIZON_DAYS}d_fixed.png"
                fig2.savefig(out_png, dpi=DPI, bbox_inches="tight")
                plt.close(fig2)

            print(f"[OK] {city}")

    print("\nWrote:")
    print(f" - PDF:  {OUT_PDF}")
    if SAVE_CITY_PNGS:
        print(f" - PNGs: {OUT_DIR}")


if __name__ == "__main__":
    main()


Found 18 city folders.
[OK] Berlin
[OK] Chicago
[OK] Denver
[OK] Houston
[OK] London
[OK] LosAngeles
[OK] Miami
[OK] Moscow
[OK] NewYork
[OK] Paris
[OK] Phoenix
[OK] Rome
[OK] SanDiego
[OK] SanFrancisco
[OK] SaoPaulo
[OK] Seattle
[OK] Sydney
[OK] Tokyo

Wrote:
 - PDF:  C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\CODES_04032021\supplement_pages_fixed\Supplement_AllCities_5windows_14d_fixed.pdf
 - PNGs: C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\CODES_04032021\supplement_pages_fixed


In [17]:
#Publication Figure Generation Code for SUEIHCDR-PINN COVID-19 Forecasting Paper
#================================================================================

#This script generates publication-quality figures using your actual model outputs.
#Copy and paste this code into your Jupyter notebook.

#Data path: C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\CODES_04032021\outputs_SUEIHCDR_PUBLICATION_11202025_v11



import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import seaborn as sns
from scipy import stats
from pathlib import Path

# =============================================================================
# CONFIGURATION
# =============================================================================

# Set your base path
BASE_PATH = Path(r"C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\CODES_04032021")
OUTPUT_PATH = BASE_PATH / "outputs_SUEIHCDR_PUBLICATION_v2_SanDiego"
METRICS_PATH = BASE_PATH / "pinn_sueihcdr_multiwindow_v2_SanDiego"

# Create figure output directory
FIG_OUTPUT = OUTPUT_PATH / "publication_figures"
FIG_OUTPUT.mkdir(exist_ok=True)

# Publication-quality defaults
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica'],
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.linewidth': 0.8,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
})

# Color palette
COLORS = {
    'PINN': '#2E86AB',
    'Hybrid': '#A23B72',
    'ARIMA': '#F18F01',
    'LogLinear': '#C73E1D',
    'Bayesian': '#6B4226',
    'observed': '#1a1a1a',
    'observed_light': '#a0a0a0',
    'S': '#1f77b4',
    'U': '#ff7f0e',
    'E': '#2ca02c',
    'I': '#d62728',
    'H': '#9467bd',
    'C': '#8c564b',
    'D': '#e377c2',
    'R': '#7f7f7f',
}

# Parameters from your model (from parameters_final.json)
PARAMS = {
    'beta0': 0.2293527275,
    'sigma': 0.2912383676,
    'delta': 0.1604585648,
    'zeta': 0.1000003666,
    'epsilon': 0.0714286864,
    'm': 0.9126121998,
    'c': 0.4633679986,
    'omega': 0.0078014438,
    'eta': 0.0020505516,
}

PHASE_ORDER = ['FirstWave', 'Delta', 'Omicron', 'BA5', 'Winter22']
PHASE_LABELS = {
    'FirstWave': 'First Wave',
    'Delta': 'Delta', 
    'Omicron': 'Omicron',
    'BA5': 'BA.5',
    'Winter22': 'Winter 22-23'
}

# =============================================================================
# DATA LOADING FUNCTIONS
# =============================================================================

def load_compartments():
    """Load compartment time series data."""
    filepath = OUTPUT_PATH / "compartments_counts.csv"
    if filepath.exists():
        df = pd.read_csv(filepath, parse_dates=['date'])
        df.set_index('date', inplace=True)
        return df
    else:
        print(f"Warning: {filepath} not found")
        return None

def load_parameters_ts():
    """Load time-varying parameters."""
    filepath = OUTPUT_PATH / "parameters_time_series.csv"
    if filepath.exists():
        df = pd.read_csv(filepath, parse_dates=['date'])
        df.set_index('date', inplace=True)
        return df
    else:
        print(f"Warning: {filepath} not found")
        return None

def load_metrics():
    """Load forecasting metrics."""
    filepath = METRICS_PATH / "multi_window_metrics_modeloAVD.csv"
    if filepath.exists():
        return pd.read_csv(filepath)
    else:
        print(f"Warning: {filepath} not found")
        return None

def load_training_history():
    """Load training history."""
    filepath = OUTPUT_PATH / "training_history.csv"
    if filepath.exists():
        return pd.read_csv(filepath)
    else:
        print(f"Warning: {filepath} not found")
        return None

# =============================================================================
# FIGURE 1: Parameter Interpretation Table (as figure)
# =============================================================================


# =============================================================================
# FIGURE: Enhanced Compartment Dynamics
# =============================================================================

def plot_compartments_enhanced():
    """
    Create enhanced compartment visualization with better visibility of small compartments.
    """
    df = load_compartments()
    if df is None:
        print("Cannot load compartments data")
        return None
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Panel A: Main compartments (S, R) - fraction
    ax = axes[0, 0]
    pop = 3.3e6  # San Diego population
    
    if 'S' in df.columns:
        S_frac = df['S'] / pop if df['S'].max() > 1 else df['S']
        R_frac = df['R'] / pop if df['R'].max() > 1 else df['R']
        
        ax.fill_between(df.index, 0, S_frac, alpha=0.6, color=COLORS['S'], label='S (Susceptible)')
        ax.fill_between(df.index, S_frac, S_frac + R_frac, alpha=0.6, color=COLORS['R'], label='R (Recovered)')
        
        ax.set_ylabel('Fraction of population')
        ax.set_title('A. Susceptible and Recovered Compartments')
        ax.legend(loc='right')
        ax.set_ylim(0, 1.05)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
    
    # Panel B: Active infection compartments (E, I)
    ax = axes[0, 1]
    if 'E' in df.columns and 'I' in df.columns:
        ax.plot(df.index, df['E'], '-', color=COLORS['E'], linewidth=2, label='E (Exposed)')
        ax.plot(df.index, df['I'], '-', color=COLORS['I'], linewidth=2, label='I (Infectious)')
        
        ax.set_ylabel('Individuals')
        ax.set_title('B. Active Infection Compartments')
        ax.legend(loc='upper right')
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
        ax.ticklabel_format(axis='y', style='scientific', scilimits=(0,0))
    
    # Panel C: Protected compartment (U)
    ax = axes[1, 0]
    if 'U' in df.columns:
        ax.fill_between(df.index, 0, df['U'], alpha=0.6, color=COLORS['U'])
        ax.plot(df.index, df['U'], '-', color=COLORS['U'], linewidth=1.5, label='U (Protected)')
        
        ax.set_ylabel('Individuals')
        ax.set_title('C. Protected (Behavioral) Compartment')
        ax.legend(loc='upper right')
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
        ax.ticklabel_format(axis='y', style='scientific', scilimits=(0,0))
    
    # Panel D: Clinical compartments (H, C, D)
    ax = axes[1, 1]
    if 'H' in df.columns:
        ax.plot(df.index, df['H'], '-', color=COLORS['H'], linewidth=2, label='H (Hospitalized)')
        ax.plot(df.index, df['C'], '-', color=COLORS['C'], linewidth=2, label='C (Critical/ICU)')
        ax.plot(df.index, df['D'], '-', color=COLORS['D'], linewidth=2, label='D (Deceased)')
        
        ax.set_ylabel('Individuals')
        ax.set_title('D. Clinical Compartments')
        ax.legend(loc='upper right')
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
        ax.ticklabel_format(axis='y', style='scientific', scilimits=(0,0))
    
    plt.suptitle('Inferred Compartment Dynamics from SUEIHCDR-PINN', fontsize=14, y=1.02)
    plt.tight_layout()
    
    plt.savefig(FIG_OUTPUT / 'Figure_compartments_enhanced.png', dpi=300)
    plt.savefig(FIG_OUTPUT / 'Figure_compartments_enhanced.pdf')
    plt.show()
    
    return fig

# =============================================================================
# FIGURE: Waning Mechanisms Visualization
# =============================================================================

def plot_waning_mechanisms():
    """
    Visualize the two waning mechanisms that enable multi-wave dynamics.
    """
    df = load_compartments()
    params_ts = load_parameters_ts()
    
    if df is None:
        print("Cannot load data")
        return None
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    pop = 3.3e6
    
    # Panel A: Waning rates over time
    ax = axes[0, 0]
    
    # Behavioral waning rate (time-varying based on fear signal)
    omega_base = PARAMS['omega']
    eta = PARAMS['eta']
    
    # Create time array
    t = np.arange(len(df))
    
    # Approximate omega(t) = omega_base * (1 - s(t)) where s(t) is fear signal
    # For now, use constant rates
    ax.axhline(omega_base * 365, color=COLORS['U'], linestyle='-', linewidth=2,
               label=f'ω (behavioral): {omega_base*365:.2f}/year')
    ax.axhline(eta * 365, color=COLORS['R'], linestyle='--', linewidth=2,
               label=f'η (biological): {eta*365:.3f}/year')
    
    ax.set_ylabel('Waning rate (year⁻¹)')
    ax.set_title('A. Waning Rate Parameters')
    ax.legend(loc='upper right')
    ax.set_ylim(0, 4)
    
    # Add interpretation text
    ax.text(0.5, 0.7, f'Behavioral protection duration: ~{1/omega_base:.0f} days\n'
                      f'Biological immunity duration: ~{1/eta:.0f} days',
            transform=ax.transAxes, fontsize=10, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    # Panel B: Waning flows over time
    ax = axes[0, 1]
    
    if 'U' in df.columns and 'R' in df.columns:
        U = df['U'].values
        R = df['R'].values
        
        # Compute flows
        flow_US = omega_base * U  # U → S
        flow_RS = eta * R         # R → S
        
        ax.fill_between(df.index, 0, flow_US, alpha=0.6, color=COLORS['U'], 
                        label='Behavioral waning (U→S)')
        ax.fill_between(df.index, flow_US, flow_US + flow_RS, alpha=0.6, 
                        color=COLORS['R'], label='Biological waning (R→S)')
        
        ax.set_ylabel('Daily flow to S (individuals/day)')
        ax.set_title('B. Daily Susceptible Replenishment')
        ax.legend(loc='upper right')
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
        ax.ticklabel_format(axis='y', style='scientific', scilimits=(0,0))
    
    # Panel C: Cumulative waning contributions
    ax = axes[1, 0]
    
    if 'U' in df.columns and 'R' in df.columns:
        cum_US = np.cumsum(flow_US)
        cum_RS = np.cumsum(flow_RS)
        
        ax.plot(df.index, cum_US / pop, '-', color=COLORS['U'], linewidth=2, 
                label='Cumulative U→S')
        ax.plot(df.index, cum_RS / pop, '-', color=COLORS['R'], linewidth=2,
                label='Cumulative R→S')
        ax.plot(df.index, (cum_US + cum_RS) / pop, 'k--', linewidth=2,
                label='Total replenishment')
        
        ax.set_ylabel('Cumulative flow (fraction of population)')
        ax.set_title('C. Cumulative Susceptible Replenishment')
        ax.legend(loc='upper left')
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
    
    # Panel D: Multi-wave enabling mechanism
    ax = axes[1, 1]
    
    if 'S' in df.columns:
        S_frac = df['S'] / pop if df['S'].max() > 1 else df['S']
        
        # Compute what S would be without waning (monotonic depletion)
        # This is approximate - actual would require re-running model
        ax.plot(df.index, S_frac, '-', color=COLORS['S'], linewidth=2, 
                label='S(t) with waning')
        
        ax.axhline(0.1, color='gray', linestyle=':', alpha=0.7)
        ax.text(df.index[50], 0.12, 'Herd immunity threshold (~10%)', fontsize=9, color='gray')
        
        ax.set_ylabel('Susceptible fraction')
        ax.set_title('D. Susceptible Pool Dynamics')
        ax.legend(loc='upper right')
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
        ax.set_ylim(0, 0.25)
    
    plt.suptitle('Waning Mechanisms Enabling Multi-Wave Epidemic Dynamics', fontsize=14, y=1.02)
    plt.tight_layout()
    
    plt.savefig(FIG_OUTPUT / 'Figure_waning_mechanisms.png', dpi=300)
    plt.savefig(FIG_OUTPUT / 'Figure_waning_mechanisms.pdf')
    plt.show()
    
    return fig

# =============================================================================
# FIGURE: Forecasting MAE Summary (Main Results Figure)
# =============================================================================

# =============================================================================
# MAIN EXECUTION
# =============================================================================

def generate_all_figures():
    """Generate all publication figures."""
    
    print("=" * 70)
    print("SUEIHCDR-PINN Publication Figure Generation")
    print("=" * 70)
    print(f"\nOutput directory: {FIG_OUTPUT}")
    print(f"Data directory: {OUTPUT_PATH}")
    print()
    
    figures = {}
    
    
    
    print("\n2. Creating enhanced compartment figure...")
    try:
        figures['compartments'] = plot_compartments_enhanced()
        print("   ✓ Complete")
    except Exception as e:
        print(f"   ✗ Error: {e}")
    
    print("\n3. Creating waning mechanisms figure...")
    try:
        figures['waning'] = plot_waning_mechanisms()
        print("   ✓ Complete")
    except Exception as e:
        print(f"   ✗ Error: {e}")
    
    
    
    print("\n" + "=" * 70)
    print("Figure generation complete!")
    print(f"Figures saved to: {FIG_OUTPUT}")
    print("=" * 70)
    
    return figures

# Run if executed directly
if __name__ == "__main__":
    figures = generate_all_figures()


#%% =============================================================================
# UTILITY: Load and process your actual metrics file
# =============================================================================

def analyze_metrics_file():
    """
    Load and analyze your actual metrics CSV file.
    Run this to get the exact values for the Results tables.
    """
    
    metrics = load_metrics()
    
    if metrics is None:
        print("Could not load metrics file. Check the path:")
        print(f"  {METRICS_PATH / 'multi_window_metrics_modeloAVD.csv'}")
        return None
    
    print("Metrics file loaded successfully!")
    print(f"Shape: {metrics.shape}")
    print(f"Columns: {list(metrics.columns)}")
    print()
    
    # Print summary statistics
    print("=" * 60)
    print("SUMMARY STATISTICS FOR RESULTS SECTION")
    print("=" * 60)
    
    # Adjust column names based on your actual file
    # Common patterns: 'mae_pinn', 'mae_arima', 'horizon', 'window', etc.
    
    print("\nFirst few rows:")
    print(metrics.head())
    
    print("\nColumn statistics:")
    print(metrics.describe())
    
    return metrics


# Run analysis
metrics = analyze_metrics_file()


SUEIHCDR-PINN Publication Figure Generation

Output directory: C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\CODES_04032021\outputs_SUEIHCDR_PUBLICATION_v2_SanDiego\publication_figures
Data directory: C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\CODES_04032021\outputs_SUEIHCDR_PUBLICATION_v2_SanDiego


2. Creating enhanced compartment figure...
   ✓ Complete

3. Creating waning mechanisms figure...
   ✓ Complete

Figure generation complete!
Figures saved to: C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\CODES_04032021\outputs_SUEIHCDR_PUBLICATION_v2_SanDiego\publication_figures
Could not load metrics file. Check the path:
  C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\CODES_04032021\pinn_sueihcdr_multiwindow_v2_SanDiego\multi_window_metrics_modeloAVD.csv


In [22]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
PUBLICATION ANALYSIS — Multi-City PINN SUEIHCDR (18 cities)
============================================================
Models: PINN, ARIMA, LogLinear, BayesPoly (Hybrid/Ensemble dropped)
All 18 cities. Larger fonts for publication.

Generates:
  MAIN figures:      fig1–fig7 (+ heatmap 14d for supplement)
  SUPPLEMENTARY:     figS1–figS4
  TABLES:            table1–table7 CSVs

Usage:  python publication_analysis.py
"""

import os, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import wilcoxon, trim_mean, spearmanr

warnings.filterwarnings("ignore")

# ======================== CONFIGURATION ========================
DATA_DIR = Path(r"C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\CODES_04032021\Resultados_Cidades_02152026")
OUT_DIR  = DATA_DIR / "publication_figures"
OUT_DIR.mkdir(exist_ok=True, parents=True)

REGIME_ORDER  = ["FirstWave", "Winter20", "Delta", "Omicron", "BA5_Waning"]
REGIME_LABELS = {"FirstWave":"First Wave", "Winter20":"Winter 20–21",
                 "Delta":"Delta", "Omicron":"Omicron", "BA5_Waning":"BA.5/Waning"}
REGIME_COLORS = {"FirstWave":"#42A5F5","Winter20":"#FFA726","Delta":"#66BB6A",
                 "Omicron":"#EF5350","BA5_Waning":"#AB47BC"}

META = {
    "SanDiego":{"pop":3.3e6,"type":"US"},      "SanFrancisco":{"pop":0.87e6,"type":"US"},
    "LosAngeles":{"pop":10e6,"type":"US"},      "Seattle":{"pop":2.3e6,"type":"US"},
    "Denver":{"pop":0.7e6,"type":"US"},         "Chicago":{"pop":5.2e6,"type":"US"},
    "Houston":{"pop":4.7e6,"type":"US"},        "Miami":{"pop":2.7e6,"type":"US"},
    "Phoenix":{"pop":4.4e6,"type":"US"},        "NewYork":{"pop":8.3e6,"type":"US"},
    "London":{"pop":9e6,"type":"Intl"},         "Rome":{"pop":2.87e6,"type":"Intl"},
    "Berlin":{"pop":3.7e6,"type":"Intl"},       "Paris":{"pop":2.16e6,"type":"Intl"},
    "Moscow":{"pop":12.6e6,"type":"Intl"},      "Tokyo":{"pop":14e6,"type":"Intl"},
    "SaoPaulo":{"pop":12.3e6,"type":"Intl"},    "Sydney":{"pop":5.3e6,"type":"Intl"},
}

TIER1_THRESHOLD = 365

# ---- PUBLICATION FONT SETTINGS ----
plt.rcParams.update({
    "figure.dpi": 300,
    "font.size": 16,
    "font.family": "sans-serif",
    "axes.titlesize": 18,
    "axes.labelsize": 16,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
    "figure.titlesize": 20,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.15,
})


# ======================== HELPERS ========================
def find_csv(name):
    for d in [DATA_DIR, DATA_DIR / "MASTER_TABLES"]:
        p = d / name
        if p.exists(): return pd.read_csv(p)
    raise FileNotFoundError(f"{name} not in {DATA_DIR}")

def paired_test(pinn, arima):
    n = len(pinn); wins = int((pinn < arima).sum())
    imp = (arima - pinn) / arima * 100
    try:    _, pw = wilcoxon(pinn, arima)
    except: pw = np.nan
    tm = trim_mean(imp, 0.10) if n >= 5 else np.mean(imp)
    sig = "***" if pw<0.001 else "**" if pw<0.01 else "*" if pw<0.05 else "ns"
    return {"n":n,"wins":wins,"win_pct":round(wins/n*100,1),
            "mean_imp":round(np.mean(imp),1),"trim_imp":round(tm,1),
            "p":round(pw,4) if pd.notna(pw) else np.nan,"sig":sig}

def rlabel(w):
    return REGIME_LABELS.get(w, w)


# ======================== DATA ========================
def load_all():
    val  = find_csv("MASTER_regime_validation_all_rows.csv")
    fit  = find_csv("MASTER_fit_metrics_from_compartments.csv")
    par  = find_csv("MASTER_parameters_final_by_city.csv")
    punc = find_csv("MASTER_parameter_uncertainty_summary.csv")
    vmul = find_csv("MASTER_variant_multiplier_summary.csv")
    pall = find_csv("MASTER_parameter_uncertainty_all_runs.csv")
    vall = find_csv("MASTER_variant_multiplier_all_runs.csv")

    val = val[val["mae_pin"].notna() & np.isfinite(val["mae_pin"]) & (val["mae_ari"]>0.1)].copy()
    val["imp_pct"]   = (val["mae_ari"]-val["mae_pin"])/val["mae_ari"]*100
    val["pinn_wins"] = val["mae_pin"] < val["mae_ari"]
    val["fit_mae"]   = val["city"].map(fit.set_index("city")["fit_mae_7d"])
    val["tier1"]     = val["fit_mae"] < TIER1_THRESHOLD
    for k in ["pop","type"]:
        val[k] = val["city"].map(lambda c,k=k: META.get(c,{}).get(k))

    print(f"Loaded {len(val)} evaluations, {val['city'].nunique()} cities")
    return val, fit, par, punc, vmul, pall, vall


# ======================== TABLES ========================
def print_tables(val, fit, par, punc, vmul, pall, vall):
    S = "=" * 90
    tier1 = val[val["tier1"]]

    fs = fit.sort_values("fit_mae_7d")
    fs.to_csv(OUT_DIR/"table1_fit_quality.csv", index=False)

    rows2=[]
    for h in [7,14]:
        for label,sub in [("All 18 cities",val[val["horizon"]==h]),
                          ("Tier 1 (sensitivity)",tier1[tier1["horizon"]==h])]:
            if len(sub)<3: continue
            r = paired_test(sub["mae_pin"].values,sub["mae_ari"].values)
            r["label"]=f"{label} {h}d"; rows2.append(r)
    pd.DataFrame(rows2).to_csv(OUT_DIR/"table2_overall.csv",index=False)

    rows3=[]
    for h in [7,14]:
        for w in REGIME_ORDER:
            sub=val[(val["horizon"]==h)&(val["window"]==w)]
            if len(sub)<3: continue
            r=paired_test(sub["mae_pin"].values,sub["mae_ari"].values)
            r.update({"horizon":h,"regime":w}); rows3.append(r)
    pd.DataFrame(rows3).to_csv(OUT_DIR/"table3_regime.csv",index=False)

    h7=val[val["horizon"]==7]; rows4=[]
    for city in sorted(h7["city"].unique()):
        cs=h7[h7["city"]==city]; pw=int(cs["pinn_wins"].sum()); n=len(cs)
        m=META.get(city,{})
        rows4.append({"city":city,"type":m.get("type",""),"pop_M":m.get("pop",0)/1e6,
            "n":n,"wins":pw,"win_pct":pw/n*100,"mean_imp":cs["imp_pct"].mean(),
            "fit_mae":cs["fit_mae"].iloc[0]})
    pd.DataFrame(rows4).sort_values("mean_imp",ascending=False).to_csv(OUT_DIR/"table4_city_7d.csv",index=False)

    # Identifiability
    raw_params = [("beta0","beta_0","Base transmission rate"),("sigma","sigma","Incubation rate (1/days)"),
        ("delta","delta","Infectious rate (1/days)"),("zeta","zeta","Hospitalization rate"),
        ("epsi","epsilon","ICU progression rate"),("m","m","Mild case fraction"),
        ("c","c","ICU fraction (of hospitalized)"),("omega","omega","Behavioral waning rate"),
        ("eta","eta","Immunity waning rate"),("rho","rho","Reporting ratio")]
    derived_params = [("incubation_days","1/sigma","Incubation period (days)"),
        ("infectious_days","1/delta","Infectious period (days)"),("ward_days","1/zeta","Hospital ward stay (days)"),
        ("icu_days","1/epsi","ICU stay (days)"),("behavioral_waning_days","1/omega","Behavioral waning (days)"),
        ("biological_waning_days","1/eta","Immunity waning (days)")]
    ident_rows = []
    for params in [raw_params, derived_params]:
        for col, sym, meaning in params:
            means = punc[f"{col}_mean"]; widths = punc[f"{col}_q975"] - punc[f"{col}_q025"]
            cv = means.std()/means.mean()*100; rw = widths.mean()/means.mean()*100
            cls = "STRONG" if cv<5 and rw<15 else "MODERATE" if cv<10 and rw<40 else "WEAK"
            tp = "raw" if params is raw_params else "derived"
            ident_rows.append({"symbol":sym,"meaning":meaning,"type":tp,
                "cross_city_mean":f"{means.mean():.5f}" if tp=="raw" else f"{means.mean():.1f}",
                "cross_city_CV":f"{cv:.1f}%","within_city_relW":f"{rw:.1f}%","classification":cls})
    pd.DataFrame(ident_rows).to_csv(OUT_DIR/"table5_parameter_identifiability.csv",index=False)
    punc.to_csv(OUT_DIR/"table5b_per_city_params.csv",index=False)
    vmul.to_csv(OUT_DIR/"table6_variants.csv",index=False)

    # Print summary
    print(f"\n{S}\nPUBLICATION SUMMARY\n{S}")
    h7a=val[val["horizon"]==7]; pw7=int(h7a["pinn_wins"].sum()); n7=len(h7a)
    _,p7=wilcoxon(h7a["mae_pin"],h7a["mae_ari"])
    strong = [r["symbol"] for r in ident_rows if r["classification"]=="STRONG"]
    weak   = [r["symbol"] for r in ident_rows if r["classification"]=="WEAK"]
    print(f"  7-day: PINN won {pw7}/{n7} ({pw7/n7*100:.0f}%), p={p7:.4f}")
    print(f"  STRONG: {', '.join(strong)}")
    print(f"  WEAK:   {', '.join(weak)}")


# ======================== FIGURES ========================
def make_figures(val, fit, par, punc, vmul, pall, vall):
    print("\nGenerating figures...")

    # ================================================================
    # MAIN FIGURE 1: Epidemiological timescales (6-panel forest plot)
    # ================================================================
    derived=[("incubation_days","Incubation period (days)"),("infectious_days","Infectious period (days)"),
             ("ward_days","Hospital ward stay (days)"),("icu_days","ICU stay (days)"),
             ("behavioral_waning_days","Behavioral waning (days)"),("biological_waning_days","Immunity waning (days)")]
    fig,axes=plt.subplots(2,3,figsize=(18,10)); axes=axes.flatten()
    for i,(col,label) in enumerate(derived):
        ax=axes[i]
        cs=punc.sort_values(f"{col}_mean")["city"].values
        means=[float(punc[punc["city"]==c][f"{col}_mean"].iloc[0]) for c in cs]
        lo=[float(punc[punc["city"]==c][f"{col}_q025"].iloc[0]) for c in cs]
        hi=[float(punc[punc["city"]==c][f"{col}_q975"].iloc[0]) for c in cs]
        y=np.arange(len(cs))
        ax.barh(y,means,color="#1565C0",alpha=0.7,height=0.7)
        ax.errorbar(means,y,xerr=[np.array(means)-np.array(lo),np.array(hi)-np.array(means)],
                    fmt="none",ecolor="black",capsize=3,lw=1.2)
        ax.set_yticks(y); ax.set_yticklabels(cs,fontsize=13)
        ax.set_xlabel(label,fontsize=15); ax.grid(alpha=0.15,axis="x")
        ax.set_title(label,fontsize=16,fontweight="bold")
    fig.suptitle("Derived Epidemiological Timescales by City\n(mean ± 95% CI from 10-seed ensemble)",
                 fontsize=18,fontweight="bold")
    fig.tight_layout(rect=[0,0,1,0.94]); fig.savefig(OUT_DIR/"fig1_timescales.png"); plt.close()
    print("  Fig 1: timescales (MAIN)")

    # ================================================================
    # MAIN FIGURE 2: Variant multipliers (forest plot with priors)
    # ================================================================
    fig,axes=plt.subplots(1,4,figsize=(20,6))
    for i,variant in enumerate(["Delta","Omicron","BA.5","XBB"]):
        ax=axes[i]; vs=vmul[vmul["variant"]==variant].sort_values("learned_mean")
        if vs.empty: ax.axis("off"); continue
        y=np.arange(len(vs))
        ax.barh(y,vs["learned_mean"].values,color="#1565C0",alpha=0.7,height=0.7)
        ax.errorbar(vs["learned_mean"].values,y,
                    xerr=[vs["learned_mean"].values-vs["learned_q025"].values,
                          vs["learned_q975"].values-vs["learned_mean"].values],
                    fmt="none",ecolor="black",capsize=3,lw=1.2)
        ax.axvline(vs["prior"].iloc[0],color="red",ls="--",lw=2.5,
                   label=f"Prior = {vs['prior'].iloc[0]:.1f}")
        ax.set_yticks(y); ax.set_yticklabels(vs["city"].values,fontsize=13)
        ax.set_xlabel("Transmissibility multiplier",fontsize=15)
        ax.set_title(variant,fontsize=17,fontweight="bold")
        ax.legend(fontsize=14,loc="lower right"); ax.grid(alpha=0.15,axis="x")
    fig.suptitle("Variant Transmissibility Multipliers by City (mean ± 95% CI)",
                 fontsize=18,fontweight="bold")
    fig.tight_layout(rect=[0,0,1,0.93]); fig.savefig(OUT_DIR/"fig2_variants.png"); plt.close()
    print("  Fig 2: variants (MAIN)")

    # ================================================================
    # MAIN FIGURE 3: Variant boxplots from 10-seed runs
    # ================================================================
    fig,axes=plt.subplots(1,4,figsize=(20,6.5))
    for i,variant in enumerate(["Delta","Omicron","BA.5","XBB"]):
        ax=axes[i]; vs=pall if "variant" in pall.columns else None
        # Use vall for variant data
        vd=vall[vall["variant"]==variant]
        if vd.empty: ax.axis("off"); continue
        cs_v=sorted(vd["city"].unique())
        data=[vd[vd["city"]==c]["learned"].values for c in cs_v]
        bp=ax.boxplot(data,vert=False,patch_artist=True,
                   boxprops=dict(facecolor="#1565C0",alpha=0.5),
                   medianprops=dict(color="red",lw=2.5),
                   whiskerprops=dict(lw=1.2),capprops=dict(lw=1.2))
        prior_val = vd["prior"].iloc[0]
        ax.axvline(prior_val,color="red",ls="--",lw=2.5,label=f"Prior = {prior_val:.1f}")
        ax.set_yticks(range(1,len(cs_v)+1)); ax.set_yticklabels(cs_v,fontsize=13)
        ax.set_xlabel("Transmissibility multiplier",fontsize=15)
        ax.set_title(variant,fontsize=17,fontweight="bold")
        ax.legend(fontsize=14); ax.grid(alpha=0.15,axis="x")
    fig.suptitle("Variant Multiplier Distributions (10 seeds per city)",
                 fontsize=18,fontweight="bold")
    fig.tight_layout(rect=[0,0,1,0.93]); fig.savefig(OUT_DIR/"fig3_variant_boxplots.png"); plt.close()
    print("  Fig 3: variant boxplots (MAIN)")

    # ================================================================
    # MAIN FIGURE 4: Scatter PINN vs ARIMA
    # ================================================================
    fig,axes=plt.subplots(1,2,figsize=(16,7))
    for ci,h in enumerate([7,14]):
        ax=axes[ci]; sub=val[val["horizon"]==h]
        p97=np.percentile(np.concatenate([sub["mae_pin"].values,sub["mae_ari"].values]),97)
        lim=p97*1.35
        ax.plot([0,lim],[0,lim],"k--",alpha=0.3,lw=1.2)
        ax.fill_between([0,lim],[0,lim],[lim,lim],alpha=0.04,color="red")
        ax.fill_between([0,lim],[0,0],[0,lim],alpha=0.04,color="blue")
        for w in REGIME_ORDER:
            ws=sub[sub["window"]==w]
            if ws.empty: continue
            ax.scatter(ws["mae_ari"],ws["mae_pin"],c=REGIME_COLORS[w],s=70,alpha=0.85,
                       label=rlabel(w),edgecolors="white",linewidth=0.6,zorder=5)
        pw=int(sub["pinn_wins"].sum()); n=len(sub)
        tm=trim_mean(sub["imp_pct"].values,0.1) if n>=10 else sub["imp_pct"].mean()
        try:    _,pval=wilcoxon(sub["mae_pin"],sub["mae_ari"])
        except: pval=np.nan
        clr="#1565C0" if pw>n/2 else "#C62828"
        ax.text(0.03,0.97,f"PINN wins: {pw}/{n} ({pw/n*100:.0f}%)\n"
                f"Trimmed mean: {tm:+.1f}%\np = {pval:.4f}",
                transform=ax.transAxes,fontsize=13,va="top",fontweight="bold",color=clr,
                bbox=dict(boxstyle="round,pad=0.4",facecolor="white",alpha=0.85,edgecolor="gray"))
        ax.set_xlabel("ARIMA MAE (cases/day)",fontsize=16)
        ax.set_ylabel("PINN MAE (cases/day)",fontsize=16)
        ax.set_title(f"{h}-day forecast horizon",fontsize=15,fontweight="bold")
        ax.legend(fontsize=13,loc="lower right",framealpha=0.9)
        ax.set_xlim(0,lim); ax.set_ylim(0,lim); ax.grid(alpha=0.15)
    fig.suptitle("PINN vs ARIMA — 18 Cities, 5 Epidemic Regimes",fontsize=19,fontweight="bold")
    fig.tight_layout(rect=[0,0,1,0.94]); fig.savefig(OUT_DIR/"fig4_scatter.png"); plt.close()
    print("  Fig 4: scatter (MAIN)")

    # ================================================================
    # MAIN FIGURE 5: Regime bars
    # ================================================================
    fig,axes=plt.subplots(1,2,figsize=(16,6.5))
    for ci,h in enumerate([7,14]):
        ax=axes[ci]; sub=val[val["horizon"]==h]
        regimes=[w for w in REGIME_ORDER if w in sub["window"].unique()]
        x=np.arange(len(regimes)); width=0.35
        pm=[sub[sub["window"]==w]["mae_pin"].mean() for w in regimes]
        am=[sub[sub["window"]==w]["mae_ari"].mean() for w in regimes]
        pse=[sub[sub["window"]==w]["mae_pin"].sem() for w in regimes]
        ase=[sub[sub["window"]==w]["mae_ari"].sem() for w in regimes]
        ax.bar(x-width/2,pm,width,label="PINN",color="#1565C0",alpha=0.85,yerr=pse,capsize=4,error_kw={"lw":1.3})
        ax.bar(x+width/2,am,width,label="ARIMA",color="#E65100",alpha=0.85,yerr=ase,capsize=4,error_kw={"lw":1.3})
        for i,w in enumerate(regimes):
            ws=sub[sub["window"]==w]
            try:    _,p=wilcoxon(ws["mae_pin"],ws["mae_ari"])
            except: p=1.0
            if p<0.05:
                ymax=max(pm[i],am[i])*1.15
                stars="***" if p<0.001 else "**" if p<0.01 else "*"
                ax.text(i,ymax,stars,ha="center",fontsize=20,color="#D4AF37",fontweight="bold")
        ax.set_xticks(x); ax.set_xticklabels([rlabel(w) for w in regimes],fontsize=12,rotation=20,ha="right")
        ax.set_ylabel("MAE (cases/day, mean ± SEM)",fontsize=15)
        ax.set_title(f"{h}-day forecast horizon",fontsize=17,fontweight="bold")
        ax.legend(fontsize=15); ax.grid(alpha=0.15,axis="y")
    fig.suptitle("PINN vs ARIMA by Epidemic Regime (18 Cities)",fontsize=19,fontweight="bold")
    fig.tight_layout(rect=[0,0,1,0.93]); fig.savefig(OUT_DIR/"fig5_regime_bars.png"); plt.close()
    print("  Fig 5: regime bars (MAIN)")

    # ================================================================
    # MAIN FIGURE 6: Heatmap 7-day
    # ================================================================
    for h in [7,14]:
        sub=val[val["horizon"]==h]; cities=sorted(sub["city"].unique())
        regimes=[w for w in REGIME_ORDER if w in sub["window"].unique()]
        heat=np.full((len(cities),len(regimes)),np.nan)
        for i,city in enumerate(cities):
            for j,w in enumerate(regimes):
                row=sub[(sub["city"]==city)&(sub["window"]==w)]
                if len(row)>0: heat[i,j]=row["imp_pct"].values[0]
        fig,ax=plt.subplots(figsize=(max(10,len(regimes)*2.2),max(7,len(cities)*0.5)))
        fv=heat[np.isfinite(heat)]
        vmax=min(80,np.percentile(np.abs(fv),95)) if len(fv)>0 else 50
        im=ax.imshow(heat,cmap="RdBu",aspect="auto",vmin=-vmax,vmax=vmax)
        ax.set_xticks(range(len(regimes))); ax.set_xticklabels([rlabel(w) for w in regimes],fontsize=15,rotation=20,ha="right")
        ax.set_yticks(range(len(cities))); ax.set_yticklabels(cities,fontsize=14)
        for i in range(len(cities)):
            for j in range(len(regimes)):
                v=heat[i,j]
                if np.isfinite(v):
                    c="white" if abs(v)>vmax*0.5 else "black"
                    ax.text(j,i,f"{v:+.0f}%",ha="center",va="center",fontsize=12,color=c,fontweight="bold")
        cbar=plt.colorbar(im,ax=ax,shrink=0.85)
        cbar.set_label("PINN improvement over ARIMA (%)\nBlue = PINN better    Red = ARIMA better",fontsize=14)
        cbar.ax.tick_params(labelsize=11)
        tag = "MAIN" if h==7 else "SUPPLEMENTARY"
        ax.set_title(f"PINN vs ARIMA: Relative Improvement ({h}-day horizon, 18 cities)",fontsize=17,fontweight="bold")
        fig.tight_layout()
        fname = f"fig6_heatmap_{h}d.png" if h==7 else f"figS1_heatmap_{h}d.png"
        fig.savefig(OUT_DIR/fname); plt.close()
    print("  Fig 6: heatmap 7d (MAIN) + FigS1: heatmap 14d (SUPPL)")

    # ================================================================
    # MAIN FIGURE 7: City profiles (7d)
    # ================================================================
    h7=val[val["horizon"]==7]
    city_order=h7.groupby("city")["imp_pct"].mean().sort_values(ascending=False).index.tolist()
    ncols=4; nrows=max(1,(len(city_order)+ncols-1)//ncols)
    fig,axes=plt.subplots(nrows,ncols,figsize=(ncols*4.5,nrows*3.5)); axes=np.array(axes).flatten()
    for i,city in enumerate(city_order):
        ax=axes[i]; cs=h7[h7["city"]==city]
        rp=[w for w in REGIME_ORDER if w in cs["window"].values]; x=np.arange(len(rp))
        pv=[float(cs[cs["window"]==w]["mae_pin"].values[0]) for w in rp]
        av=[float(cs[cs["window"]==w]["mae_ari"].values[0]) for w in rp]
        ax.bar(x-0.2,pv,0.35,color="#1565C0",alpha=0.85,label="PINN")
        ax.bar(x+0.2,av,0.35,color="#E65100",alpha=0.85,label="ARIMA")
        mean_imp=cs["imp_pct"].mean()
        ax.set_title(f"{city} ({mean_imp:+.1f}%)",fontsize=14,fontweight="bold",
                     color="#1565C0" if mean_imp>0 else "#C62828")
        ax.set_xticks(x); ax.set_xticklabels([rlabel(w)[:6] for w in rp],fontsize=11,rotation=30,ha="right")
        if i==0: ax.legend(fontsize=11)
        ax.grid(alpha=0.15,axis="y")
    for i in range(len(city_order),len(axes)): axes[i].axis("off")
    fig.suptitle("PINN vs ARIMA by City (7-day MAE)",fontsize=18,fontweight="bold")
    fig.tight_layout(rect=[0,0,1,0.95]); fig.savefig(OUT_DIR/"fig7_city_profiles.png"); plt.close()
    print("  Fig 7: city profiles (MAIN)")

    # ================================================================
    # SUPPLEMENTARY FIGURE S2: All 10 raw parameters forest plot
    # ================================================================
    raw_plist=[("beta0","β₀"),("sigma","σ"),("delta","δ"),("zeta","ζ"),
               ("epsi","ε"),("m","m"),("c","c"),("omega","ω"),("eta","η"),("rho","ρ")]
    fig,axes=plt.subplots(2,5,figsize=(24,10)); axes=axes.flatten()
    for i,(col,sym) in enumerate(raw_plist):
        ax=axes[i]
        cs=punc.sort_values(f"{col}_mean")["city"].values
        means=[float(punc[punc["city"]==c][f"{col}_mean"].iloc[0]) for c in cs]
        lo=[float(punc[punc["city"]==c][f"{col}_q025"].iloc[0]) for c in cs]
        hi=[float(punc[punc["city"]==c][f"{col}_q975"].iloc[0]) for c in cs]
        y=np.arange(len(cs))
        cv=np.std(means)/np.mean(means)*100
        rw=np.mean(np.array(hi)-np.array(lo))/np.mean(means)*100
        if cv<5 and rw<15:   color="#1565C0"
        elif cv<10 and rw<40: color="#FFA726"
        else:                 color="#EF5350"
        ax.barh(y,means,color=color,alpha=0.7,height=0.7)
        ax.errorbar(means,y,xerr=[np.array(means)-np.array(lo),np.array(hi)-np.array(means)],
                    fmt="none",ecolor="black",capsize=3,lw=1.2)
        ax.set_yticks(y); ax.set_yticklabels(cs,fontsize=11)
        ax.grid(alpha=0.15,axis="x")
        cls_label = "STRONG" if cv<5 and rw<15 else "MOD." if cv<10 and rw<40 else "WEAK"
        ax.set_title(f"{sym} [{cls_label}]\nCV={cv:.1f}%  relW={rw:.1f}%",fontsize=14,fontweight="bold")
    fig.suptitle("All Raw Parameters by City\n(Blue = strongly identified, Orange = moderate, Red = weakly identified)",
                 fontsize=18,fontweight="bold")
    fig.tight_layout(rect=[0,0,1,0.93]); fig.savefig(OUT_DIR/"figS2_all_parameters.png"); plt.close()
    print("  FigS2: all raw parameters (SUPPL)")

    # ================================================================
    # SUPPLEMENTARY FIGURE S3: Parameter boxplots from 10-seed runs
    # ================================================================
    derived_raw=[("incubation_days","Incubation (days)"),("infectious_days","Infectious (days)"),
                 ("ward_days","Ward stay (days)"),("icu_days","ICU stay (days)"),
                 ("behavioral_waning_days","Behavioral waning (days)"),("biological_waning_days","Immunity waning (days)")]
    cs_sorted=sorted(pall["city"].unique())
    fig,axes=plt.subplots(2,3,figsize=(18,10)); axes=axes.flatten()
    for i,(col,label) in enumerate(derived_raw):
        ax=axes[i]
        data=[pall[pall["city"]==c][col].values for c in cs_sorted]
        bp=ax.boxplot(data,vert=False,patch_artist=True,
                   boxprops=dict(facecolor="#1565C0",alpha=0.5),
                   medianprops=dict(color="red",lw=2.5),
                   whiskerprops=dict(lw=1.2),capprops=dict(lw=1.2))
        ax.set_yticks(range(1,len(cs_sorted)+1)); ax.set_yticklabels(cs_sorted,fontsize=12)
        ax.set_xlabel(label,fontsize=15); ax.set_title(label,fontsize=16,fontweight="bold")
        ax.grid(alpha=0.15,axis="x")
    fig.suptitle("Parameter Distributions Across 10-Seed Ensemble (per city)",
                 fontsize=18,fontweight="bold")
    fig.tight_layout(rect=[0,0,1,0.94]); fig.savefig(OUT_DIR/"figS3_param_boxplots.png"); plt.close()
    print("  FigS3: parameter boxplots (SUPPL)")

    # ================================================================
    # SUPPLEMENTARY FIGURE S4: Identifiability summary
    # ================================================================
    all_params = [
        ("β₀","Transmission","beta0"),("σ","Incubation rate","sigma"),
        ("δ","Infectious rate","delta"),("ζ","Hospital rate","zeta"),
        ("ε","ICU rate","epsi"),("ρ","Reporting ratio","rho"),
        ("m","Mild fraction","m"),("c","ICU fraction","c"),
        ("ω","Behav. waning","omega"),("η","Immun. waning","eta")]
    fig,ax=plt.subplots(figsize=(13,7))
    for i,(sym,label,col) in enumerate(all_params):
        means=punc[f"{col}_mean"]; widths=punc[f"{col}_q975"]-punc[f"{col}_q025"]
        cv=means.std()/means.mean()*100; rw=widths.mean()/means.mean()*100
        if cv<5 and rw<15:   color="#1565C0"
        elif cv<10 and rw<40: color="#FFA726"
        else:                 color="#EF5350"
        ax.barh(i,cv,color=color,alpha=0.7,height=0.7)
        ax.errorbar(cv,i,xerr=rw/5,fmt="d",color="black",markersize=6,capsize=4,lw=1.2)
        ax.text(max(cv,rw/5)+1.5,i,f"CV = {cv:.1f}%    relW = {rw:.1f}%",va="center",fontsize=14)
    ax.set_yticks(range(len(all_params)))
    ax.set_yticklabels([f"{sym}  ({label})" for sym,label,_ in all_params],fontsize=15)
    ax.set_xlabel("Cross-city coefficient of variation (%)",fontsize=16)
    ax.set_title("Parameter Identifiability Summary\nBlue = STRONG    Orange = MODERATE    Red = WEAK",
                 fontsize=17,fontweight="bold")
    ax.grid(alpha=0.15,axis="x"); ax.invert_yaxis()
    fig.tight_layout(); fig.savefig(OUT_DIR/"figS4_identifiability.png"); plt.close()
    print("  FigS4: identifiability summary (SUPPL)")

    print(f"\nAll figures saved to: {OUT_DIR}")
    print(f"\n  MAIN FIGURES:  fig1–fig7  (7 figures)")
    print(f"  SUPPLEMENTARY: figS1–figS4 (4 figures)")


# ======================== MAIN ========================
def main():
    print("="*70)
    print("PUBLICATION ANALYSIS — Multi-City PINN SUEIHCDR")
    print("="*70)
    val,fit,par,punc,vmul,pall,vall = load_all()
    print_tables(val,fit,par,punc,vmul,pall,vall)
    make_figures(val,fit,par,punc,vmul,pall,vall)
    print(f"\n{'='*70}\nDONE. All outputs in: {OUT_DIR}\n{'='*70}")

if __name__=="__main__":
    main()


PUBLICATION ANALYSIS — Multi-City PINN SUEIHCDR
Loaded 164 evaluations, 18 cities

PUBLICATION SUMMARY
  7-day: PINN won 53/82 (65%), p=0.0486
  STRONG: beta_0, sigma, delta, zeta, epsilon, m, rho, 1/sigma, 1/delta, 1/zeta, 1/epsi
  WEAK:   omega, 1/omega, 1/eta

Generating figures...
  Fig 1: timescales (MAIN)
  Fig 2: variants (MAIN)
  Fig 3: variant boxplots (MAIN)
  Fig 4: scatter (MAIN)
  Fig 5: regime bars (MAIN)
  Fig 6: heatmap 7d (MAIN) + FigS1: heatmap 14d (SUPPL)
  Fig 7: city profiles (MAIN)
  FigS2: all raw parameters (SUPPL)
  FigS3: parameter boxplots (SUPPL)
  FigS4: identifiability summary (SUPPL)

All figures saved to: C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\CODES_04032021\Resultados_Cidades_02152026\publication_figures

  MAIN FIGURES:  fig1–fig7  (7 figures)
  SUPPLEMENTARY: figS1–figS4 (4 figures)

DONE. All outputs in: C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\CODES_04032021\Resultados_Cidades_02152026\publication_figures


In [22]:
# ============================================================
# Nature Communications revision helpers (drop-in cell)
#   1) Patch alpha future projection to match ALPH_BOUNDS
#   2) Add stronger baselines (ETS/Theta/Naive)
#   3) Add scale-normalized metrics (MASE, per-100k MAE)
#   4) Combine existing per-city regime_validation_metrics.csv
#   5) Multiplicity correction (Holm) + city-clustered tests
#   6) Optional: ablation scaffolding (no/single waning; no physics)
# ============================================================

import os, re, math, warnings, json
from pathlib import Path
from contextlib import contextmanager

import numpy as np
import pandas as pd

from scipy.stats import wilcoxon

try:
    from scipy.stats import binomtest
except Exception:
    binomtest = None

# -----------------------------
# 0) Quick audit prints
# -----------------------------
print("ALPH_BOUNDS:", ALPH_BOUNDS)
print("VAR_BUMP_BOUNDS:", VAR_BUMP_BOUNDS)
print("VALIDATION_CONFIG:", VALIDATION_CONFIG)
print("NOTE: This cell patches project_alpha_future() to respect ALPH_BOUNDS.\n")


# -----------------------------
# 1) PATCH: alpha future projection
# -----------------------------
def project_alpha_future(alpha_hist, steps, mode="exp_decay", tau=7, floor=None, quantile_base=0.25):
    """
    Project α(t) forward (used during forecast rollout).

    IMPORTANT PATCH:
      - default floor is ALPH_BOUNDS[0] (NOT 0.05)
      - baseline is a low quantile of recent α history
      - output is clamped to ALPH_BOUNDS exactly
    """
    if steps <= 0:
        return np.array([], float)

    ah = np.asarray(alpha_hist, float)
    if ah.size == 0:
        last = float(ALPH_BOUNDS[0])
        base = float(ALPH_BOUNDS[0])
    else:
        last = float(ah[-1])
        if ah.size >= 21:
            base = float(np.quantile(ah[-21:], quantile_base))
        else:
            base = float(np.quantile(ah, quantile_base))

    if floor is None:
        floor = float(ALPH_BOUNDS[0])
    floor = float(np.clip(floor, ALPH_BOUNDS[0], ALPH_BOUNDS[1]))
    base  = float(np.clip(max(base, floor), ALPH_BOUNDS[0], ALPH_BOUNDS[1]))
    last  = float(np.clip(last, ALPH_BOUNDS[0], ALPH_BOUNDS[1]))

    t = np.arange(1, steps + 1, dtype=float)
    out = base + (last - base) * np.exp(-t / max(float(tau), 1.0))
    return np.clip(out, ALPH_BOUNDS[0], ALPH_BOUNDS[1]).astype(float)

print("✅ Patched project_alpha_future() to be consistent with ALPH_BOUNDS.\n")


# -----------------------------
# 2) Metrics helpers: per-capita, MASE, Holm correction, Wilson CI
# -----------------------------
def wilson_ci(k, n, alpha=0.05):
    """Wilson score interval for a binomial proportion."""
    if n <= 0:
        return (np.nan, np.nan)
    z = 1.959963984540054  # ~N(0,1) 97.5th percentile
    phat = k / n
    denom = 1 + z*z/n
    center = (phat + z*z/(2*n)) / denom
    half = z * math.sqrt((phat*(1-phat) + z*z/(4*n)) / n) / denom
    return (max(0.0, center-half), min(1.0, center+half))

def holm_adjust(pvals):
    """
    Holm-Bonferroni adjustment.
    Returns adjusted p-values in original order.
    """
    pvals = np.asarray(pvals, float)
    m = len(pvals)
    order = np.argsort(pvals)
    adj = np.empty(m, float)
    running_max = 0.0
    for j, idx in enumerate(order):
        p = pvals[idx]
        adj_p = (m - j) * p
        running_max = max(running_max, adj_p)
        adj[idx] = min(1.0, running_max)
    return adj

def mase_scale(y_train, eps=1e-9):
    """
    MASE denominator: mean absolute naive one-step change.
    Works on ANY scale (raw counts, 7d avg, etc).
    """
    y = np.asarray(y_train, float)
    if len(y) < 2:
        return np.nan
    denom = np.mean(np.abs(np.diff(y)))
    return max(denom, eps)

def per100k(mae, Npop):
    return float(mae) / (float(Npop) / 100_000.0)


# -----------------------------
# 3) Build observed 7d series and "true future" 7d values
#    (uses your existing daily_from_cum_np, weekly_avg_np, carry_wavg_future_from_cum)
# -----------------------------
def obs7_series_from_df(df):
    # observed cumulative up to end
    cum = df["cases"].values.astype(float)
    daily = daily_from_cum_np(cum)
    obs7 = weekly_avg_np(daily)
    dates = pd.to_datetime(df["date"].values)[:len(obs7)]
    return pd.Series(obs7, index=dates)

def true_future_7d_from_df(df, cut_date, horizon):
    cut_date = pd.Timestamp(cut_date)
    df2 = df.copy()
    df2["date"] = pd.to_datetime(df2["date"])
    cut_idx = get_cut_idx_full(df2, cut_date)

    obs_cum_hist = df2["cases"].values[:cut_idx+1].astype(float)
    true_cum_future = df2["cases"].values[cut_idx+1: cut_idx+1+horizon].astype(float)
    if len(true_cum_future) < horizon:
        return None
    return carry_wavg_future_from_cum(obs_cum_hist, true_cum_future)

def train_series_for_origin(df, cut_date, lookback_days):
    cut = pd.Timestamp(cut_date)
    tr_start = cut - pd.Timedelta(days=int(lookback_days))
    dsub = df[(pd.to_datetime(df["date"]) <= cut) & (pd.to_datetime(df["date"]) >= tr_start)].copy()
    s = obs7_series_from_df(dsub)
    return s


# -----------------------------
# 4) Stronger baselines (fast): Naive, ETS, Theta (if available)
# -----------------------------
def forecast_naive(y_series, cut_date, horizon, method="last"):
    cut_date = pd.Timestamp(cut_date)
    y_train = y_series.loc[:cut_date].dropna()
    if len(y_train) == 0:
        return np.zeros(horizon, float)
    if method == "last":
        v = float(y_train.iloc[-1])
        return np.full(horizon, v, float)
    elif method == "mean7":
        tail = y_train.iloc[-7:] if len(y_train) >= 7 else y_train
        v = float(np.mean(tail))
        return np.full(horizon, v, float)
    else:
        raise ValueError("method must be 'last' or 'mean7'")

def forecast_ets(y_series, cut_date, horizon, history_days=60):
    """
    ETS / Holt-Winters (statsmodels). Works on 7d-avg series.
    """
    cut_date = pd.Timestamp(cut_date)
    y_train = y_series.loc[:cut_date].dropna()
    y_train = y_train.iloc[-history_days:] if len(y_train) > history_days else y_train
    if len(y_train) < 10:
        return forecast_naive(y_series, cut_date, horizon, method="last")

    try:
        from statsmodels.tsa.holtwinters import ExponentialSmoothing
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore")
            mod = ExponentialSmoothing(
                y_train.values.astype(float),
                trend="add",
                damped_trend=True,
                seasonal=None,
                initialization_method="estimated"
            )
            fit = mod.fit(optimized=True)
            fc = fit.forecast(horizon)
        fc = np.asarray(fc, float).reshape(-1)
        fc = np.clip(fc, 0.0, None)
        if len(fc) < horizon:
            fc = np.pad(fc, (0, horizon-len(fc)), mode="edge")
        return fc[:horizon]
    except Exception:
        return forecast_naive(y_series, cut_date, horizon, method="last")

def forecast_theta(y_series, cut_date, horizon, history_days=90):
    """
    Theta model baseline (statsmodels >= 0.12 typically).
    Falls back to ETS if unavailable.
    """
    cut_date = pd.Timestamp(cut_date)
    y_train = y_series.loc[:cut_date].dropna()
    y_train = y_train.iloc[-history_days:] if len(y_train) > history_days else y_train
    if len(y_train) < 20:
        return forecast_ets(y_series, cut_date, horizon, history_days=min(history_days, 60))

    try:
        from statsmodels.tsa.forecasting.theta import ThetaModel
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore")
            tm = ThetaModel(y_train.values.astype(float), period=1)  # period=1 since we smoothed weekly already
            fit = tm.fit()
            fc = fit.forecast(horizon)
        fc = np.asarray(fc, float).reshape(-1)
        fc = np.clip(fc, 0.0, None)
        if len(fc) < horizon:
            fc = np.pad(fc, (0, horizon-len(fc)), mode="edge")
        return fc[:horizon]
    except Exception:
        return forecast_ets(y_series, cut_date, horizon, history_days=min(history_days, 60))

def baseline_mae_suite(df, cut_date, horizon, lookback_days_for_mase=180):
    """
    Compute baseline MAEs against the same "true_wavg" definition you use.
    Returns dict with mae_naive_last, mae_ets, mae_theta, mase_scale (for later use).
    """
    y = obs7_series_from_df(df)
    y_train_for_scale = train_series_for_origin(df, cut_date, lookback_days_for_mase)
    scale = mase_scale(y_train_for_scale.values)

    true_wavg = true_future_7d_from_df(df, cut_date, horizon)
    if true_wavg is None:
        return None

    pred_naive = forecast_naive(y, cut_date, horizon, method="last")
    pred_ets   = forecast_ets(y, cut_date, horizon, history_days=60)
    pred_theta = forecast_theta(y, cut_date, horizon, history_days=90)

    out = {
        "mase_scale": float(scale),
        "mae_naive_last": float(np.mean(np.abs(pred_naive - true_wavg))),
        "mae_ets": float(np.mean(np.abs(pred_ets - true_wavg))),
        "mae_theta": float(np.mean(np.abs(pred_theta - true_wavg))),
    }
    return out


# -----------------------------
# 5) Load your existing per-city regime_validation_metrics.csv and augment
# -----------------------------
_LOOKBACK_BY_WINDOW = {nm: int(lb) for (nm, _, lb) in VALIDATION_CONFIG}

def _city_label_from_dirname(d):
    # expects pinn_sueihcdr_multiwindow_v2_{CITY}
    m = re.match(r"pinn_sueihcdr_multiwindow_v2_(.+)$", Path(d).name)
    return m.group(1) if m else Path(d).name

def get_city_population_fast():
    """
    Returns dict city->population using:
      - CITY_POP for world
      - covid_county_population_usafacts.csv for US cities
    """
    pop = {}
    # world
    for k,v in CITY_POP.items():
        pop[k] = float(v)

    # US via usa facts pop file (fast, single read)
    try:
        popdf = pd.read_csv(Path(PATH)/"covid_county_population_usafacts.csv")
        for (label, county, state, sub) in US_CITIES:
            row = popdf[(popdf["County Name"]==county) & (popdf["State"]==state)]
            if not row.empty:
                pop[label] = float(row.iloc[0]["population"])
    except Exception as e:
        print("WARNING: Could not load US pop file for per-capita metrics:", e)

    return pop

_CITY_POP_ALL = get_city_population_fast()

from pathlib import Path
import pandas as pd

def load_all_existing_city_metrics(out_root=OUT_PATH_STR):
    out_root = Path(out_root)
    rows = []

    # Look for files like: regime_validation_metrics_<City>.csv
    for f in out_root.glob("regime_validation_metrics_*.csv"):
        if f.is_file():
            # Extract city name from filename
            # Example: "regime_validation_metrics_SanDiego.csv"
            city = f.stem.replace("regime_validation_metrics_", "")

            dfm = pd.read_csv(f)
            dfm["city"] = city
            rows.append(dfm)

    if not rows:
        print("No regime_validation_metrics_<city>.csv found under", out_root)
        return None

    allm = pd.concat(rows, ignore_index=True)

    # Standardize types
    allm["cut_date"] = pd.to_datetime(allm["cut_date"])
    allm["horizon"] = pd.to_numeric(allm["horizon"], errors="coerce").astype(int)

    return allm

def load_city_df(label):
    # 1. Clean the label
    label_clean = str(label).strip().lower()
    
    # 2. Hardcode the US routing to bypass corrupted global variables
    hardcoded_us_map = {
        "sandiego":   ("San Diego County", "CA", "California"),
        "seattle":    ("King County", "WA", "Washington"),
        "newyork":    ("New York County", "NY", "New York"),
        "chicago":    ("Cook County", "IL", "Illinois"),
        "houston":    ("Harris County", "TX", "Texas"),
        "phoenix":    ("Maricopa County", "AZ", "Arizona"),
        "miami":      ("Miami-Dade County", "FL", "Florida"),
        "denver":     ("Denver County", "CO", "Colorado"),
        "losangeles": ("Los Angeles County", "CA", "California"),
        "sanfrancisco": ("San Francisco County", "CA", "California"),
    }

    # -- Check US Cities --
    if label_clean in hardcoded_us_map:
        county, state, sub = hardcoded_us_map[label_clean]
        df, Npop = load_us_county_series(county, state, sub)
        return df, float(Npop)

    # -- Check World Cities --
    intl_map = {k.lower(): k for k in CITY_SPECS.keys()}
    if label_clean in intl_map:
        original_key = intl_map[label_clean]
        df, _ = load_world_city_series(original_key)
        
        # Pull population securely
        Npop = CITY_POP.get(original_key, np.nan)
        if np.isnan(Npop):
            _, Npop_alt = load_world_city_series(original_key)
            Npop = Npop_alt
            
        return df, float(Npop)

    # -- Fail Safely --
    raise ValueError(f"Unknown city label: '{label}'. Check spelling in the metrics CSV.")

def augment_metrics_with_scale_and_baselines(metrics_df, cache_city_df=True, do_baselines=True):
    """
    Adds:
      - Npop
      - MAE per 100k for PINN and ARIMA
      - MASE for PINN and ARIMA (using training-window scale)
      - ETS/Theta/Naive MAEs (computed from raw data; no PINN retrain)
    """
    if metrics_df is None or len(metrics_df) == 0:
        return metrics_df

    # cache city dfs to avoid re-loading
    city_cache = {}

    aug = metrics_df.copy()
    aug["lookback_days"] = aug["window"].map(_LOOKBACK_BY_WINDOW).astype(float)

    # fill pops
    aug["Npop"] = aug["city"].map(_CITY_POP_ALL).astype(float)

    # some cities might be missing pop -> load once
    miss = aug[aug["Npop"].isna()]["city"].unique().tolist()
    for c in miss:
        try:
            _, Np = load_city_df(c)
            _CITY_POP_ALL[c] = float(Np)
        except Exception:
            _CITY_POP_ALL[c] = np.nan
    aug["Npop"] = aug["city"].map(_CITY_POP_ALL).astype(float)

    # compute MASE scale per row (needs training series)
    mase_scales = []
    mae_naive = []
    mae_ets = []
    mae_theta = []

    for i, r in aug.iterrows():
        city = r["city"]
        cut = r["cut_date"]
        H = int(r["horizon"])
        lb = int(r["lookback_days"]) if np.isfinite(r["lookback_days"]) else 180

        if cache_city_df and city in city_cache:
            df_city, Np = city_cache[city]
        else:
            df_city, Np = load_city_df(city)
            if cache_city_df:
                city_cache[city] = (df_city, Np)

        # MASE scale
        try:
            y_train = train_series_for_origin(df_city, cut, lb)
            sc = mase_scale(y_train.values)
        except Exception:
            sc = np.nan
        mase_scales.append(sc)

        # baselines
        if do_baselines:
            try:
                b = baseline_mae_suite(df_city, cut, H, lookback_days_for_mase=lb)
            except Exception:
                b = None
            if b is None:
                mae_naive.append(np.nan); mae_ets.append(np.nan); mae_theta.append(np.nan)
            else:
                mae_naive.append(b["mae_naive_last"])
                mae_ets.append(b["mae_ets"])
                mae_theta.append(b["mae_theta"])
        else:
            mae_naive.append(np.nan); mae_ets.append(np.nan); mae_theta.append(np.nan)

    aug["mase_scale"] = mase_scales

    # per 100k MAE
    aug["mae_pin_per100k"] = aug.apply(lambda r: per100k(r["mae_pin"], r["Npop"]) if np.isfinite(r["Npop"]) else np.nan, axis=1)
    aug["mae_ari_per100k"] = aug.apply(lambda r: per100k(r["mae_ari"], r["Npop"]) if np.isfinite(r["Npop"]) else np.nan, axis=1)

    # MASE
    aug["mase_pin"] = aug["mae_pin"] / aug["mase_scale"]
    aug["mase_ari"] = aug["mae_ari"] / aug["mase_scale"]

    # add baseline MAEs
    aug["mae_naive_last"] = mae_naive
    aug["mae_ets"] = mae_ets
    aug["mae_theta"] = mae_theta

    # baseline MASE too
    aug["mase_naive_last"] = aug["mae_naive_last"] / aug["mase_scale"]
    aug["mase_ets"] = aug["mae_ets"] / aug["mase_scale"]
    aug["mase_theta"] = aug["mae_theta"] / aug["mase_scale"]

    return aug

def summarize_pairwise(df, a_col="mae_pin", b_col="mae_ari", label_a="PINN", label_b="ARIMA"):
    d = df[[a_col, b_col, "city", "window", "horizon"]].dropna()
    if len(d) == 0:
        return None

    wins = int(np.sum(d[a_col] < d[b_col]))
    n = int(len(d))
    win_pct = 100.0 * wins / n
    ci_lo, ci_hi = wilson_ci(wins, n)
    ci_lo *= 100.0; ci_hi *= 100.0

    # naive paired Wilcoxon across evaluations
    try:
        p = float(wilcoxon(d[a_col] - d[b_col]).pvalue)
    except Exception:
        p = np.nan

    # city-clustered: average within city, then Wilcoxon across cities
    city_agg = d.groupby("city").apply(lambda g: float(np.mean(g[a_col] - g[b_col]))).reset_index(name="mean_diff")
    try:
        p_city = float(wilcoxon(city_agg["mean_diff"].values).pvalue)
    except Exception:
        p_city = np.nan

    out = {
        "n_eval": n,
        "wins": wins,
        "win_pct": win_pct,
        "win_ci95": f"[{ci_lo:.1f}%, {ci_hi:.1f}%]",
        "wilcoxon_eval_p": p,
        "wilcoxon_city_p": p_city,
        "label_a": label_a,
        "label_b": label_b,
    }
    return out

def regime_pvals_with_holm(df, a_col="mae_pin", b_col="mae_ari"):
    """
    Per-horizon regime tests + Holm correction.
    Uses evaluation-level Wilcoxon within each (regime,horizon).
    """
    out_rows = []
    for H in sorted(df["horizon"].dropna().unique()):
        subH = df[df["horizon"] == H]
        regs = sorted(subH["window"].dropna().unique())
        pvals = []
        tmp = []
        for reg in regs:
            d = subH[subH["window"] == reg][[a_col, b_col]].dropna()
            if len(d) < 6:
                p = np.nan
            else:
                try:
                    p = float(wilcoxon(d[a_col] - d[b_col]).pvalue)
                except Exception:
                    p = np.nan
            pvals.append(p)
            tmp.append((H, reg, len(d), p))
        adj = holm_adjust([p if np.isfinite(p) else 1.0 for p in pvals]) if len(pvals) else []
        for (H, reg, n, p), pa in zip(tmp, adj):
            out_rows.append({"horizon": H, "regime": reg, "n": n, "p_raw": p, "p_holm": pa})
    return pd.DataFrame(out_rows)

# -----------------------------
# 6) OPTIONAL: Ablation scaffolding (targeted runs)
# -----------------------------
@contextmanager
def temp_globals(**kwargs):
    """
    Temporarily override global constants (e.g., ETA_BOUNDS, OMEGA_BOUNDS, W_PHYS).
    Useful for ablations without rewriting your training function.
    """
    g = globals()
    old = {}
    for k,v in kwargs.items():
        old[k] = g.get(k, None)
        g[k] = v
    try:
        yield
    finally:
        for k,v in old.items():
            g[k] = v

def run_targeted_ablation(
    city_label,
    regimes=("Winter20", "BA5_Waning"),
    horizons=(7,14),
    n_models=3,
    ablation_name="no_waning",
    outdir=None
):
    """
    Runs a SMALL ablation suite for one city on selected regimes/horizons.
    This WILL retrain the PINN on those windows (but only a few runs).

    ablation_name:
      - "baseline"     : your current model
      - "no_waning"    : omega=0 and eta=0
      - "no_bio"       : eta=0 only
      - "no_behav"     : omega=0 only
      - "no_physics"   : W_PHYS=0 (keeps other regularizers)
    """
    df, Npop = load_city_df(city_label)
    df["date"] = pd.to_datetime(df["date"])

    reg_map = {nm: (pd.Timestamp(cut), int(lb)) for nm, cut, lb in VALIDATION_CONFIG}
    picks = [(r, reg_map[r][0], reg_map[r][1]) for r in regimes if r in reg_map]

    if outdir is None:
        outdir = Path(OUT_PATH_STR) / "ablations_natcomm" / city_label / ablation_name
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    # choose overrides
    overrides = {}
    if ablation_name == "no_waning":
        overrides.update({"OMEGA_BOUNDS": (0.0, 0.0), "ETA_BOUNDS": (0.0, 0.0)})
    elif ablation_name == "no_bio":
        overrides.update({"ETA_BOUNDS": (0.0, 0.0)})
    elif ablation_name == "no_behav":
        overrides.update({"OMEGA_BOUNDS": (0.0, 0.0)})
    elif ablation_name == "no_physics":
        overrides.update({"W_PHYS": 0.0})
    elif ablation_name == "baseline":
        overrides = {}
    else:
        raise ValueError("Unknown ablation_name")

    results = []

    with temp_globals(**overrides):
        print(f"\n=== ABLATION {ablation_name} | city={city_label} | overrides={overrides} ===")
        for (reg, cut, lb) in picks:
            if cut > df["date"].max() - pd.Timedelta(days=max(horizons)):
                print("  SKIP", reg, "insufficient future data")
                continue
            tr_start = max(cut - pd.Timedelta(days=lb), df["date"].min())
            dsub = df[(df["date"]<=cut) & (df["date"]>=tr_start)].reset_index(drop=True)
            if len(dsub) < 60:
                print("  SKIP", reg, "<60 days")
                continue

            max_h = max(horizons)
            sd_future = project_sd_future(dsub["sd"].values, max_h)

            cfg = TrainCfg(
                max_epochs=EPOCHS_MAX, lr=LR_FULL, sd_lag=SD_LAG_DAYS,
                rollout_extra=max_h, validation_days=VALIDATION_DAYS, patience_epochs=PATIENCE_EPOCHS
            )

            # train ensemble (small n_models for speed)
            preds = []
            last_res = None
            for i in range(n_models):
                torch.manual_seed(10_000 + i*111)
                rr = train_sueihcdr_once(dsub, Npop, cfg, return_all=True, sd_future=sd_future)
                preds.append(rr["Ccum"])
                last_res = rr
            stack = np.vstack(preds)
            # use median like your pipeline
            fr = dict(last_res)
            fr["Ccum"] = np.median(stack, axis=0)

            for H in horizons:
                m = generate_plots_and_metrics(fr, df, Npop, cut, H, window_name=reg, lookback_days=lb, outdir=outdir)
                if m:
                    m.update({"city": city_label, "ablation": ablation_name, "window": reg, "cut_date": str(cut.date()), "horizon": int(H)})
                    results.append(m)

    out_df = pd.DataFrame(results)
    out_df.to_csv(outdir / "ablation_metrics.csv", index=False)
    print("Saved:", outdir / "ablation_metrics.csv")
    return out_df


# -----------------------------
# 7) RUN: compile + augment + summarize (no retraining)
# -----------------------------
from pathlib import Path

metrics = load_all_existing_city_metrics(
    Path(OUT_PATH_STR) / "Resultados_Cidades_02152026"
)

if metrics is None:
    print("No existing metrics found. If you haven't run mode=full per city, run that first.")
else:
    print("Loaded existing evaluation rows:", len(metrics))
    aug = augment_metrics_with_scale_and_baselines(metrics, cache_city_df=True, do_baselines=True)

    # Save augmented results
    out_aug = Path(OUT_PATH_STR) / "natcomm_revision_metrics_augmented.csv"
    aug.to_csv(out_aug, index=False)
    print("Saved augmented metrics:", out_aug)

    # Overall summaries (raw MAE and MASE)
    s1 = summarize_pairwise(aug, "mae_pin", "mae_ari", "PINN", "ARIMA")
    s2 = summarize_pairwise(aug, "mase_pin", "mase_ari", "PINN", "ARIMA")
    s3 = summarize_pairwise(aug, "mae_pin", "mae_ets", "PINN", "ETS")
    s4 = summarize_pairwise(aug, "mae_ari", "mae_ets", "ARIMA", "ETS")
    print("\n=== OVERALL (raw MAE) PINN vs ARIMA ===\n", s1)
    print("\n=== OVERALL (MASE)   PINN vs ARIMA ===\n", s2)
    print("\n=== OVERALL (raw MAE) PINN vs ETS ===\n", s3)
    print("\n=== OVERALL (raw MAE) ARIMA vs ETS ===\n", s4)

    # Regime p-values with Holm correction (helps address multiple testing critique)
    reg_tbl = regime_pvals_with_holm(aug, a_col="mae_pin", b_col="mae_ari")
    out_reg = Path(OUT_PATH_STR) / "natcomm_regime_pvals_holm.csv"
    reg_tbl.to_csv(out_reg, index=False)
    print("\nSaved regime p-values (Holm corrected):", out_reg)
    print(reg_tbl)

ALPH_BOUNDS: (0.0, 0.003)
VAR_BUMP_BOUNDS: (1.0, 8.0)
VALIDATION_CONFIG: [('FirstWave', '2020-05-01', 60), ('Winter20', '2021-01-15', 120), ('Delta', '2021-08-15', 300), ('Omicron', '2022-01-15', 150), ('BA5_Waning', '2022-06-15', 150)]
NOTE: This cell patches project_alpha_future() to respect ALPH_BOUNDS.

✅ Patched project_alpha_future() to be consistent with ALPH_BOUNDS.

Loaded existing evaluation rows: 166
    Resolved key: DE_BE_11001
    Loaded Berlin: 602 days, 2020-03-03 → 2022-02-11
    Resolved key: GB_ENG_LON
    Clipping outlier spike at end: day 932, daily=3054801 > 10400
    Loaded London: 941 days, 2020-02-15 → 2022-09-12
    Resolved key: RU_MOW
    Loaded Moscow: 706 days, 2020-03-22 → 2022-09-15
    Resolved key: FR_IDF_75
    Loaded Paris: 796 days, 2020-03-10 → 2022-05-14
    Resolved key: IT_62_RM
    Loaded Rome: 932 days, 2020-02-24 → 2022-09-12
    Resolved key: BR_SP_SAO
    Loaded SaoPaulo: 989 days, 2020-01-01 → 2022-09-15
    Resolved key: AU_NSW
    Loaded

In [23]:
# small, high-impact ablation set (recommended subset)
for city in ["SanDiego", "NewYork", "London", "Tokyo"]:
    _ = run_targeted_ablation(city, ablation_name="baseline", regimes=("Winter20","BA5_Waning"), horizons=(7,14), n_models=3)
    _ = run_targeted_ablation(city, ablation_name="no_waning", regimes=("Winter20","BA5_Waning"), horizons=(7,14), n_models=3)
    _ = run_targeted_ablation(city, ablation_name="no_physics", regimes=("Winter20","BA5_Waning"), horizons=(7,14), n_models=3)


=== ABLATION baseline | city=SanDiego | overrides={} ===
  ep 1/7000 loss=352.7299
  ep 500/7000 loss=2.4472
  ep 1000/7000 loss=1.1062
  Early stop ep 1300, best=775
  ep 1/7000 loss=347.5679
  ep 500/7000 loss=2.4520
  ep 1000/7000 loss=1.0773
  ep 1500/7000 loss=0.5674
  Early stop ep 1575, best=1050
  ep 1/7000 loss=345.6571
  ep 500/7000 loss=2.4449
  ep 1000/7000 loss=1.0657
  Early stop ep 1300, best=775
    H=7d: MAE_PINN=459.9 MAE_ARIMA=465.1 MAE_Hybrid=462.9
    Saved Plot -> overlay_2021-01-15_7d.png
    H=14d: MAE_PINN=800.0 MAE_ARIMA=784.7 MAE_Hybrid=791.1
    Saved Plot -> overlay_2021-01-15_14d.png
  ep 1/7000 loss=212.8904
  ep 500/7000 loss=3.3525
  Early stop ep 825, best=300
  ep 1/7000 loss=209.2468
  ep 500/7000 loss=3.4249
  Early stop ep 550, best=25
  ep 1/7000 loss=207.4475
  ep 500/7000 loss=2.7839
  Early stop ep 700, best=175
    H=7d: MAE_PINN=562.5 MAE_ARIMA=570.1 MAE_Hybrid=566.3
    Saved Plot -> overlay_2022-06-15_7d.png
    H=14d: MAE_PINN=628.5 MAE_A

In [24]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- CONFIG ---
BASE = Path(r"C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\CODES_04032021")
OUT  = BASE / "natcomm_revision_figs"
OUT.mkdir(parents=True, exist_ok=True)

# --- HELPERS ---
def holm_adjust(pvals):
    """Holm correction (step-down) without statsmodels dependency."""
    pvals = np.asarray(pvals, dtype=float)
    m = len(pvals)
    order = np.argsort(pvals)
    adj = np.empty(m, dtype=float)
    # raw Holm adjustment
    for k, idx in enumerate(order):
        adj[idx] = min((m - k) * pvals[idx], 1.0)
    # enforce monotonicity in sorted order
    adj_sorted = adj[order]
    adj_sorted = np.maximum.accumulate(adj_sorted)  # non-decreasing
    adj[order] = np.clip(adj_sorted, 0.0, 1.0)
    return adj

def safe_read_csv(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")
    return pd.read_csv(path)

# --- LOAD MAIN REVISION METRICS ---
rev = safe_read_csv(BASE / "natcomm_revision_metrics_augmented.csv")
pvals_counts = safe_read_csv(BASE / "natcomm_regime_pvals_holm.csv")

rev["horizon"] = rev["horizon"].astype(int)
rev["imp_mae_counts"]  = rev["mae_ari"] - rev["mae_pin"]
rev["imp_mae_per100k"] = rev["mae_ari_per100k"] - rev["mae_pin_per100k"]

regime_order = ["FirstWave", "Winter20", "Delta", "Omicron", "BA5_Waning"]

# --- OVERALL WIN RATES ---
for H in sorted(rev["horizon"].unique()):
    sub = rev[rev["horizon"] == H]
    wins = int((sub["mae_pin"] < sub["mae_ari"]).sum())
    n = int(sub[["mae_pin","mae_ari"]].dropna().shape[0])
    print(f"H={H}: PINN beats ARIMA in {wins}/{n} = {wins/n:.3%} windows")

# --- PER-100K WILCOXON + HOLM (two-sided), computed from your per100k columns ---
from scipy.stats import wilcoxon

rows = []
for H in [7, 14]:
    for reg in regime_order:
        dif = rev[(rev["horizon"]==H) & (rev["window"]==reg)]["imp_mae_per100k"].dropna().values
        n = len(dif)
        if n < 2 or np.allclose(dif, 0):
            p_raw = np.nan
        else:
            p_raw = float(wilcoxon(dif, alternative="two-sided").pvalue)
        rows.append({"regime": reg, "horizon": H, "n": n,
                     "mean_imp_per100k": float(np.mean(dif)) if n else np.nan,
                     "p_raw": p_raw})
per100k_p = pd.DataFrame(rows)
mask = per100k_p["p_raw"].notna()
per100k_p.loc[mask, "p_holm"] = holm_adjust(per100k_p.loc[mask, "p_raw"].values)
per100k_p.to_csv(OUT / "natcomm_regime_pvals_holm_per100k.csv", index=False)

# --- REGIME SUMMARY TABLES ---
def make_regime_summary(df, metric_imp, outfile):
    out = []
    for H in [7,14]:
        for reg in regime_order:
            sub = df[(df["horizon"]==H) & (df["window"]==reg)]
            imp = sub[metric_imp].dropna()
            if len(imp)==0:
                continue
            out.append({
                "regime": reg,
                "horizon": H,
                "n": int(len(imp)),
                "win_rate": float((imp > 0).mean()),
                "mean_improvement": float(imp.mean()),
                "median_improvement": float(imp.median()),
            })
    s = pd.DataFrame(out)
    s.to_csv(outfile, index=False)
    return s

summary_counts  = make_regime_summary(rev, "imp_mae_counts",  OUT / "natcomm_regime_summary_counts.csv")
summary_per100k = make_regime_summary(rev, "imp_mae_per100k", OUT / "natcomm_regime_summary_per100k.csv")

# --- PLOTS: regime improvement boxplots ---
def plot_regime_boxplot(df, H, metric_col, ylabel, outpath, pvals_table=None):
    sub = df[df["horizon"]==H].copy()
    data = [sub[sub["window"]==reg][metric_col].dropna().values for reg in regime_order]

    fig, ax = plt.subplots(figsize=(10.5, 4.8))
    ax.boxplot(data, labels=regime_order, showmeans=True)
    ax.axhline(0, color="k", linewidth=0.8, alpha=0.6)
    ax.set_ylabel(ylabel)
    ax.set_title(f"PINN vs ARIMA improvement by regime (horizon={H}d)")
    ax.grid(axis="y", alpha=0.25)

    if pvals_table is not None:
        for i, reg in enumerate(regime_order, start=1):
            row = pvals_table[(pvals_table["regime"]==reg) & (pvals_table["horizon"]==H)]
            if len(row):
                ph = float(row["p_holm"].values[0])
                y = np.nanmax(data[i-1]) if len(data[i-1]) else 0.0
                ax.text(i, y, f"p_Holm={ph:.3g}", ha="center", va="bottom", fontsize=9)

    fig.tight_layout()
    fig.savefig(outpath, dpi=300)
    plt.close(fig)

# Counts + Holm labels from your provided table
plot_regime_boxplot(rev, 7,  "imp_mae_counts",  "ΔMAE (ARIMA − PINN) [cases/day]",
                    OUT / "fig_regime_improvement_boxplot_7d_counts.png",  pvals_counts)
plot_regime_boxplot(rev, 14, "imp_mae_counts",  "ΔMAE (ARIMA − PINN) [cases/day]",
                    OUT / "fig_regime_improvement_boxplot_14d_counts.png", pvals_counts)

# Per-100k boxplots (labels optional)
plot_regime_boxplot(rev, 7,  "imp_mae_per100k", "ΔMAE per 100k (ARIMA − PINN)",
                    OUT / "fig_regime_improvement_boxplot_7d_per100k.png",  None)
plot_regime_boxplot(rev, 14, "imp_mae_per100k", "ΔMAE per 100k (ARIMA − PINN)",
                    OUT / "fig_regime_improvement_boxplot_14d_per100k.png", None)

# --- LOAD ABLATION METRICS ---
ablation_rows = []
for p in BASE.glob("ablation_metrics_*.csv"):
    m = re.match(r"ablation_metrics_(baseline|nophysics|nowaning)_(.+)\.csv", p.name)
    if not m:
        continue
    ab = m.group(1)
    city = m.group(2)
    if city.lower() == "sandeigo":  # handle typo
        city = "SanDiego"
    df = pd.read_csv(p)
    df["ablation"] = ab
    df["city"] = city
    ablation_rows.append(df)

if not ablation_rows:
    print("No ablation_metrics_*.csv found in BASE.")
else:
    abl = pd.concat(ablation_rows, ignore_index=True)
    abl["horizon"] = abl["horizon"].astype(int)

    city_order = ["London","NewYork","SanDiego","Tokyo"]
    regime_abl = ["Winter20","BA5_Waning"]
    ablation_order = ["baseline","nophysics","nowaning"]
    label_map = {"baseline":"Full PINN","nophysics":"No physics","nowaning":"No waning"}

    def plot_ablation_ratio(horizon, outpath):
        sub = abl[abl["horizon"]==horizon].copy()
        x_labels = [f"{c}\n{r}" for c in city_order for r in regime_abl]
        x = np.arange(len(x_labels))
        width = 0.25

        fig, ax = plt.subplots(figsize=(14,5))
        for i, ab in enumerate(ablation_order):
            y = []
            for c in city_order:
                for r in regime_abl:
                    d = sub[(sub["city"]==c) & (sub["window"]==r)]
                    if d.empty:
                        y.append(np.nan); continue
                    mae_pin = float(d[d["ablation"]==ab]["mae_pin"].values[0])
                    mae_ari = float(d.iloc[0]["mae_ari"])
                    y.append(mae_pin/mae_ari if mae_ari>0 else np.nan)
            ax.bar(x + (i - (len(ablation_order)-1)/2)*width, y, width, label=label_map[ab])

        ax.axhline(1.0, color="k", linestyle="--", linewidth=1.0, alpha=0.7, label="ARIMA (ratio=1)")
        ax.set_xticks(x)
        ax.set_xticklabels(x_labels, rotation=0)
        ax.set_ylabel("MAE ratio vs ARIMA ( < 1 is better )")
        ax.set_title(f"Ablation vs ARIMA (horizon={horizon}d): MAE ratio")
        ax.grid(axis="y", alpha=0.25)
        ax.legend(ncol=2, frameon=False)
        fig.tight_layout()
        fig.savefig(outpath, dpi=300)
        plt.close(fig)

    plot_ablation_ratio(7,  OUT / "fig_ablation_ratio_vs_arima_7d.png")
    plot_ablation_ratio(14, OUT / "fig_ablation_ratio_vs_arima_14d.png")

    # Save a wide summary table too
    out_rows = []
    for H in [7,14]:
        sub = abl[abl["horizon"]==H]
        for c in city_order:
            for r in regime_abl:
                d = sub[(sub["city"]==c) & (sub["window"]==r)]
                if d.empty:
                    continue
                row = {"city":c, "regime":r, "horizon":H, "cut_date": d.iloc[0]["cut_date"]}
                for ab in ["baseline","nophysics","nowaning"]:
                    row[f"mae_{ab}"] = float(d[d["ablation"]==ab]["mae_pin"].values[0])
                row["mae_arima"] = float(d.iloc[0]["mae_ari"])
                row["delta_nophysics_vs_full"] = row["mae_nophysics"] - row["mae_baseline"]
                row["delta_nowaning_vs_full"]  = row["mae_nowaning"]  - row["mae_baseline"]
                out_rows.append(row)
    ablw = pd.DataFrame(out_rows)
    ablw.to_csv(OUT / "ablation_summary_wide.csv", index=False)

print(f"\nDone. Outputs saved to:\n  {OUT}")

H=7: PINN beats ARIMA in 53/83 = 63.855% windows
H=14: PINN beats ARIMA in 24/83 = 28.916% windows


C:\Users\osmar\AppData\Local\Temp\ipykernel_1740\1670466034.py:100: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=regime_order, showmeans=True)
C:\Users\osmar\AppData\Local\Temp\ipykernel_1740\1670466034.py:100: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=regime_order, showmeans=True)
C:\Users\osmar\AppData\Local\Temp\ipykernel_1740\1670466034.py:100: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=regime_order, showmeans=True)
C:\Users\osmar\AppData\Local\Temp\ipykernel_1740\1670466034.py:100: MatplotlibDeprecationWarning: The 'labels' parameter 


Done. Outputs saved to:
  C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\CODES_04032021\natcomm_revision_figs


In [28]:
# ============================================================
# EXTRA BASELINES CELL (drop-in)
# Adds: ETS + Theta (statsmodels-only) and Prophet / TBATS
# Target series: 7-day averaged daily cases (same target as your loglin/bayes baselines)
#
# Usage (quick test):
#   label = "SanDiego"
#   df, Npop = load_us_county_series("San Diego County","CA","California")
#   out = eval_extra_baselines_only(df, cut_date="2022-06-15", horizon=14, history_days=21)
#   print(out)
#
# Usage (integrate into your existing regime evaluation):
#   - Call generate_plots_and_metrics_EXT(...) instead of generate_plots_and_metrics(...)
#     inside run_multi_window_eval (minimal change).
# ============================================================

import numpy as np
import pandas as pd
import warnings

# ---------- statsmodels baselines (ETS + Theta) ----------
HAS_ETS = False
HAS_THETA = False

try:
    # Preferred: true ETS model class in statsmodels
    from statsmodels.tsa.exponential_smoothing.ets import ETSModel
    HAS_ETS = True
except Exception:
    HAS_ETS = False

try:
    # Fallback: Holt-Winters exponential smoothing (also in statsmodels)
    from statsmodels.tsa.holtwinters import ExponentialSmoothing
except Exception:
    ExponentialSmoothing = None

try:
    # ThetaModel exists in newer statsmodels
    from statsmodels.tsa.forecasting.theta import ThetaModel
    HAS_THETA = True
except Exception:
    HAS_THETA = False

# ---------- Prophet / TBATS (optional installs) ----------
HAS_PROPHET = False
HAS_TBATS = False

try:
    from prophet import Prophet
    HAS_PROPHET = True
except Exception:
    HAS_PROPHET = False

try:
    from tbats import TBATS
    HAS_TBATS = True
except Exception:
    HAS_TBATS = False

def _last_value_forecast(y, horizon):
    y = np.asarray(y, float)
    last = float(y[-1]) if y.size else 0.0
    return np.full(int(horizon), last, dtype=float)

def _get_train_slice(series_7d: pd.Series, current_date, history_days=21, min_points=14):
    """
    Returns y_train (np array) and idx_train (DatetimeIndex). Uses [current_date-history_days, current_date].
    """
    if not isinstance(series_7d.index, pd.DatetimeIndex):
        raise ValueError("series_7d must have a DatetimeIndex")
    cd = pd.Timestamp(current_date)
    sl = series_7d.loc[cd - pd.Timedelta(days=int(history_days)) : cd].dropna()
    if len(sl) < int(min_points):
        return None, None
    y = sl.values.astype(float)
    y[~np.isfinite(y)] = 0.0
    y = np.clip(y, 0.0, None)
    return y, sl.index

# ---------------- ETS forecast (statsmodels-only) ----------------
def forecast_ets_7d(series_7d: pd.Series, current_date, horizon=14, history_days=21,
                    seasonal_periods=7, min_points=14):
    """
    ETS forecast on the 7-day averaged series.
    Uses ETSModel if available; else Holt-Winters ExponentialSmoothing fallback.
    """
    y, idx = _get_train_slice(series_7d, current_date, history_days=history_days, min_points=min_points)
    if y is None:
        return _last_value_forecast(series_7d.loc[:pd.Timestamp(current_date)].dropna().values, horizon)

    H = int(horizon)
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")

        # Preferred: ETSModel
        if HAS_ETS:
            try:
                # Additive components are safest with zeros
                mod = ETSModel(y, error="add", trend="add", damped_trend=True,
                               seasonal="add", seasonal_periods=int(seasonal_periods))
                fit = mod.fit(disp=False)
                fc = np.asarray(fit.forecast(H), float).reshape(-1)
                fc = np.clip(fc, 0.0, None)
                if fc.size < H:
                    fc = np.pad(fc, (0, H-fc.size), mode="edge")
                return fc[:H]
            except Exception:
                pass

        # Fallback: Holt-Winters (still a legit ES baseline)
        if ExponentialSmoothing is not None:
            try:
                hw = ExponentialSmoothing(
                    y,
                    trend="add",
                    damped_trend=True,
                    seasonal="add",
                    seasonal_periods=int(seasonal_periods),
                    initialization_method="estimated",
                ).fit(optimized=True)
                fc = np.asarray(hw.forecast(H), float).reshape(-1)
                fc = np.clip(fc, 0.0, None)
                if fc.size < H:
                    fc = np.pad(fc, (0, H-fc.size), mode="edge")
                return fc[:H]
            except Exception:
                pass

    return _last_value_forecast(y, H)

# ---------------- Theta forecast (statsmodels-only) ----------------
def forecast_theta_7d(series_7d: pd.Series, current_date, horizon=14, history_days=21,
                      seasonal_periods=7, min_points=14):
    """
    Theta forecast on the 7-day averaged series.
    Uses statsmodels ThetaModel if available; else a simple theta-like fallback:
      average(SES, drift) on levels (still reasonable as a baseline).
    """
    y, idx = _get_train_slice(series_7d, current_date, history_days=history_days, min_points=min_points)
    if y is None:
        return _last_value_forecast(series_7d.loc[:pd.Timestamp(current_date)].dropna().values, horizon)

    H = int(horizon)
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")

        if HAS_THETA:
            try:
                tm = ThetaModel(y, period=int(seasonal_periods), deseasonalize=True)
                fit = tm.fit(disp=False)
                fc = np.asarray(fit.forecast(H), float).reshape(-1)
                fc = np.clip(fc, 0.0, None)
                if fc.size < H:
                    fc = np.pad(fc, (0, H-fc.size), mode="edge")
                return fc[:H]
            except Exception:
                pass

        # Fallback theta-ish: average(SimpleExpSmoothing, drift)
        try:
            from statsmodels.tsa.holtwinters import SimpleExpSmoothing
            ses = SimpleExpSmoothing(y, initialization_method="estimated").fit(optimized=True)
            fc_ses = np.asarray(ses.forecast(H), float).reshape(-1)
        except Exception:
            fc_ses = _last_value_forecast(y, H)

        # drift / random-walk-with-drift on levels
        t = np.arange(len(y), dtype=float)
        if len(y) >= 2 and np.nanstd(y) > 1e-9:
            # least squares drift
            A = np.vstack([t, np.ones_like(t)]).T
            b1, b0 = np.linalg.lstsq(A, y, rcond=None)[0]  # y ~ b1*t + b0
            t_f = np.arange(len(y), len(y) + H, dtype=float)
            fc_drift = b1 * t_f + b0
        else:
            fc_drift = _last_value_forecast(y, H)

        fc = 0.5 * np.asarray(fc_ses, float) + 0.5 * np.asarray(fc_drift, float)
        fc = np.clip(fc, 0.0, None)
        if fc.size < H:
            fc = np.pad(fc, (0, H-fc.size), mode="edge")
        return fc[:H]

# ---------------- Prophet forecast ----------------
def forecast_prophet_7d(series_7d: pd.Series, current_date, horizon=14, history_days=60,
                        min_points=30):
    """
    Prophet baseline on 7-day averaged series.
    NOTE: Prophet is heavier; use a longer history_days by default.
    Install if needed:
      pip install prophet
      # or: conda install -c conda-forge prophet
    """
    if not HAS_PROPHET:
        return None  # signal "not available"

    y, idx = _get_train_slice(series_7d, current_date, history_days=history_days, min_points=min_points)
    if y is None:
        return _last_value_forecast(series_7d.loc[:pd.Timestamp(current_date)].dropna().values, horizon)

    H = int(horizon)
    df_train = pd.DataFrame({"ds": idx, "y": y})

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")
        try:
            m = Prophet(
                weekly_seasonality=True,
                daily_seasonality=False,
                yearly_seasonality=False,
                seasonality_mode="additive",
                changepoint_prior_scale=0.05,
            )
            m.fit(df_train)
            future = m.make_future_dataframe(periods=H, freq="D", include_history=False)
            fc = m.predict(future)["yhat"].values.astype(float)
            fc = np.clip(fc, 0.0, None)
            if fc.size < H:
                fc = np.pad(fc, (0, H-fc.size), mode="edge")
            return fc[:H]
        except Exception:
            return _last_value_forecast(y, H)

# ---------------- TBATS forecast ----------------
def forecast_tbats_7d(series_7d: pd.Series, current_date, horizon=14, history_days=120,
                      seasonal_periods=(7,), min_points=60, use_arma_errors=False):
    """
    TBATS baseline on 7-day averaged series.
    NOTE: TBATS can be slow; use a longer history_days by default, and keep use_arma_errors=False for speed.
    Install if needed:
      pip install tbats
    """
    if not HAS_TBATS:
        return None  # signal "not available"

    y, idx = _get_train_slice(series_7d, current_date, history_days=history_days, min_points=min_points)
    if y is None:
        return _last_value_forecast(series_7d.loc[:pd.Timestamp(current_date)].dropna().values, horizon)

    H = int(horizon)
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")
        try:
            est = TBATS(
                seasonal_periods=list(seasonal_periods),
                use_box_cox=False,
                use_trend=True,
                use_damped_trend=True,
                use_arma_errors=bool(use_arma_errors),
            )
            fit = est.fit(y)
            fc = np.asarray(fit.forecast(steps=H), float).reshape(-1)
            fc = np.clip(fc, 0.0, None)
            if fc.size < H:
                fc = np.pad(fc, (0, H-fc.size), mode="edge")
            return fc[:H]
        except Exception:
            return _last_value_forecast(y, H)

# ---------------- Build the common 7-day series from df ----------------
def build_obs7_series(df: pd.DataFrame) -> pd.Series:
    """
    df must have columns: date, cases (cumulative)
    Returns a pd.Series of 7-day avg daily cases indexed by df['date'].
    """
    d = df.copy()
    if not np.issubdtype(d["date"].dtype, np.datetime64):
        d["date"] = pd.to_datetime(d["date"])
    cum = d["cases"].values.astype(float)
    daily = daily_from_cum_np(cum)
    obs7 = weekly_avg_np(daily)
    return pd.Series(obs7, index=pd.to_datetime(d["date"].values))

# ---------------- Evaluate extra baselines only (no PINN needed) ----------------
def eval_extra_baselines_only(df: pd.DataFrame, cut_date, horizon=14, history_days=21):
    """
    Returns MAE of ETS/Theta/(optional Prophet/TBATS) vs truth at one cut_date/horizon.
    Truth is computed exactly like your pipeline: 7-day avg of future daily cases with carry.
    """
    if not np.issubdtype(df["date"].dtype, np.datetime64):
        df = df.copy()
        df["date"] = pd.to_datetime(df["date"])

    cut = pd.Timestamp(cut_date)
    cut_idx_full = get_cut_idx_full(df, cut)
    obs_cum = df["cases"].values[:cut_idx_full + 1]
    true_cum_future = df["cases"].values[cut_idx_full + 1 : cut_idx_full + 1 + int(horizon)]

    if len(true_cum_future) != int(horizon):
        return {"ok": False, "reason": "insufficient future data", "cut_date": str(cut.date()), "horizon": int(horizon)}

    true_wavg = carry_wavg_future_from_cum(obs_cum, true_cum_future)
    Hm = int(len(true_wavg))
    if Hm == 0:
        return {"ok": False, "reason": "true_wavg empty", "cut_date": str(cut.date()), "horizon": int(horizon)}

    series_7d = build_obs7_series(df)

    fc_ets   = forecast_ets_7d(series_7d, cut, horizon=Hm, history_days=history_days)
    fc_theta = forecast_theta_7d(series_7d, cut, horizon=Hm, history_days=history_days)

    out = {
        "ok": True,
        "cut_date": str(cut.date()),
        "horizon": int(horizon),
        "history_days": int(history_days),
        "mae_ets": float(np.mean(np.abs(fc_ets - true_wavg))),
        "mae_theta": float(np.mean(np.abs(fc_theta - true_wavg))),
    }

    fc_prophet = forecast_prophet_7d(series_7d, cut, horizon=Hm) if HAS_PROPHET else None
    if fc_prophet is not None:
        out["mae_prophet"] = float(np.mean(np.abs(np.asarray(fc_prophet) - true_wavg)))
    else:
        out["mae_prophet"] = np.nan

    fc_tbats = forecast_tbats_7d(series_7d, cut, horizon=Hm) if HAS_TBATS else None
    if fc_tbats is not None:
        out["mae_tbats"] = float(np.mean(np.abs(np.asarray(fc_tbats) - true_wavg)))
    else:
        out["mae_tbats"] = np.nan

    return out

# ---------------- Integrate into your existing PINN window eval ----------------
def generate_plots_and_metrics_EXT(full_res, df, Npop, cut_date, horizon, window_name="",
                                  lookback_days=180, outdir=None,
                                  history_days_ets_theta=21,
                                  run_prophet=False, run_tbats=False):
    """
    Wrapper around your existing generate_plots_and_metrics().
    Adds ETS/Theta (+ optional Prophet/TBATS) MAEs to the returned metrics dict.
    """
    base = generate_plots_and_metrics(full_res, df, Npop, cut_date, horizon,
                                      window_name=window_name, lookback_days=lookback_days, outdir=outdir)
    if base is None:
        return None

    # Recompute the evaluation target horizon exactly like the base function does
    if not np.issubdtype(df["date"].dtype, np.datetime64):
        df = df.copy()
        df["date"] = pd.to_datetime(df["date"])
    cut = pd.Timestamp(cut_date)
    cut_idx_full = get_cut_idx_full(df, cut)
    obs_cum = df["cases"].values[:cut_idx_full + 1]
    true_cum_future = df["cases"].values[cut_idx_full + 1 : cut_idx_full + 1 + int(horizon)]
    true_wavg = carry_wavg_future_from_cum(obs_cum, true_cum_future)
    Hm = int(len(true_wavg))
    if Hm <= 0:
        return base

    series_7d = build_obs7_series(df)

    # ETS + Theta (statsmodels-only)
    ets_fc   = forecast_ets_7d(series_7d, cut, horizon=Hm, history_days=history_days_ets_theta)
    theta_fc = forecast_theta_7d(series_7d, cut, horizon=Hm, history_days=history_days_ets_theta)
    base["mae_ets"]   = float(np.mean(np.abs(ets_fc - true_wavg)))
    base["mae_theta"] = float(np.mean(np.abs(theta_fc - true_wavg)))

    # Optional Prophet / TBATS (only if you set flags + deps installed)
    base["mae_prophet"] = np.nan
    base["mae_tbats"] = np.nan

    if run_prophet:
        if not HAS_PROPHET:
            print("Prophet not available. Install: pip install prophet  (or conda install -c conda-forge prophet)")
        else:
            p_fc = forecast_prophet_7d(series_7d, cut, horizon=Hm)
            base["mae_prophet"] = float(np.mean(np.abs(np.asarray(p_fc) - true_wavg)))

    if run_tbats:
        if not HAS_TBATS:
            print("TBATS not available. Install: pip install tbats")
        else:
            t_fc = forecast_tbats_7d(series_7d, cut, horizon=Hm)
            base["mae_tbats"] = float(np.mean(np.abs(np.asarray(t_fc) - true_wavg)))

    return base

print(f"[OK] Extra baselines loaded. statsmodels ETS={HAS_ETS or (ExponentialSmoothing is not None)}, Theta={HAS_THETA}")
print(f"     Prophet={HAS_PROPHET} | TBATS={HAS_TBATS}")

[OK] Extra baselines loaded. statsmodels ETS=True, Theta=False
     Prophet=True | TBATS=False


In [26]:
%pip install prophet 
%pip install tbats 



   ---------------------------------------- 0.0/12.1 MB ? eta -:--:--
   ------------------- -------------------- 5.8/12.1 MB 32.2 MB/s eta 0:00:01
   ------------------------------------ --- 11.0/12.1 MB 27.6 MB/s eta 0:00:01
   ---------------------------------------- 12.1/12.1 MB 24.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 1.4/1.4 MB 24.1 MB/s eta 0:00:00

   ---------- ----------------------------- 1/4 [holidays]
   ---------- ----------------------------- 1/4 [holidays]
   ---------- ----------------------------- 1/4 [holidays]
   ---------- ----------------------------- 1/4 [holidays]
   ---------- ----------------------------- 1/4 [holidays]
   ---------- ----------------------------- 1/4 [holidays]
   ---------- ----------------------------- 1/4 [holidays]
   ---------- ----------------------------- 1/4 [holidays]
   ---------- ----------------------------- 1/4 [holidays]
   ---------- --


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/715.6 kB ? eta -:--:--
   --------------------------------------- 715.6/715.6 kB 14.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 2.8/2.8 MB 26.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ------------------------------ --------- 7.3/9.5 MB 34.9 MB/s eta 0:00:01
   ---------------------------------------- 9.5/9.5 MB 33.0 MB/s eta 0:00:00

   ---------------------------------------- 0/4 [Cython]
   ---------------------------------------- 0/4 [Cython]
   ---------------------------------------- 0/4 [Cython]
   ---------------------------------------- 0/4 [Cython]
   ---------------------------------------- 0/4 [Cython]
   ---------------------------------------- 0/4 [Cython]
   ---------------------------------------- 0/4 [Cython]
   ---------------------------------------- 0/4 [Cython]
   --------------------

  You can safely remove it manually.

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
# ============================================================
# EXTRA BASELINES (ETS + Prophet) — drop-in Jupyter cell
# - Computes forecasts on the SAME target you evaluate: 7-day avg daily cases
# - Uses SAME rolling-origin regime cuts (VALIDATION_CONFIG) and horizons (HORIZON_LIST)
# - Does NOT rerun PINN; this is baseline-only evaluation
# - Saves per-city CSV: extra_baselines_metrics_<City>.csv
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

# ---- availability checks (based on what you saw) ----
HAS_ETS = False
HAS_PROPHET = False

try:
    from statsmodels.tsa.exponential_smoothing.ets import ETSModel
    HAS_ETS = True
except Exception as e:
    print("[WARN] ETSModel not available:", e)

try:
    from prophet import Prophet
    HAS_PROPHET = True
except Exception as e:
    print("[WARN] Prophet not available:", e)

print(f"[OK] Baselines available? ETS={HAS_ETS} | Prophet={HAS_PROPHET}")

# ----------------------------
# Helper: build the target series you forecast (7-day avg daily cases)
# ----------------------------
def build_obs7_series(df: pd.DataFrame) -> pd.Series:
    if not np.issubdtype(df["date"].dtype, np.datetime64):
        df = df.copy()
        df["date"] = pd.to_datetime(df["date"])
    cum = df["cases"].values.astype(float)
    daily = daily_from_cum_np(cum)
    obs7 = weekly_avg_np(daily)
    # obs7 aligns to df["date"] index (same length)
    return pd.Series(obs7, index=pd.to_datetime(df["date"].values))

# ----------------------------
# ETS forecast (statsmodels ETSModel)
# ----------------------------
def forecast_ets(series_7d: pd.Series, cut_date: pd.Timestamp, horizon: int,
                 lookback_days: int = 120,
                 error: str = "add", trend: str = "add", seasonal: str = None,
                 damped_trend: bool = True) -> np.ndarray:
    """
    ETS on 7d-avg daily cases. Uses log1p transform for stability.
    Falls back to last-value hold if model fails.
    """
    if not HAS_ETS:
        return np.full(horizon, float(series_7d.loc[:cut_date].iloc[-1]), float)

    train = series_7d.loc[cut_date - pd.Timedelta(days=lookback_days): cut_date].dropna()
    if len(train) < 14:
        last = float(train.iloc[-1]) if len(train) else 0.0
        return np.full(horizon, last, float)

    y = np.log1p(train.values.astype(float))
    try:
        mod = ETSModel(
            y,
            error=error,
            trend=trend,
            seasonal=seasonal,
            damped_trend=damped_trend
        )
        res = mod.fit(disp=False)
        fc = res.forecast(horizon)
        pred = np.expm1(np.asarray(fc, float))
        pred = np.clip(pred, 0.0, None)
        if pred.shape[0] != horizon:
            pred = np.pad(pred, (0, horizon - pred.shape[0]), mode="edge")
        return pred[:horizon]
    except Exception:
        last = float(train.iloc[-1])
        return np.full(horizon, last, float)

# ----------------------------
# Prophet forecast
# ----------------------------
def forecast_prophet(series_7d: pd.Series, cut_date: pd.Timestamp, horizon: int,
                     lookback_days: int = 180,
                     weekly_seasonality: bool = True,
                     yearly_seasonality: bool = True,
                     changepoint_prior_scale: float = 0.05) -> np.ndarray:
    """
    Prophet on 7d-avg daily cases. Uses log1p transform for stability.
    Falls back to last-value hold if model fails.
    """
    if not HAS_PROPHET:
        return np.full(horizon, float(series_7d.loc[:cut_date].iloc[-1]), float)

    train = series_7d.loc[cut_date - pd.Timedelta(days=lookback_days): cut_date].dropna()
    if len(train) < 21:
        last = float(train.iloc[-1]) if len(train) else 0.0
        return np.full(horizon, last, float)

    dfp = pd.DataFrame({
        "ds": train.index,
        "y": np.log1p(train.values.astype(float))
    })

    try:
        m = Prophet(
            weekly_seasonality=weekly_seasonality,
            yearly_seasonality=yearly_seasonality,
            daily_seasonality=False,
            changepoint_prior_scale=changepoint_prior_scale
        )
        m.fit(dfp)

        future = pd.DataFrame({"ds": pd.date_range(cut_date + pd.Timedelta(days=1), periods=horizon, freq="D")})
        fc = m.predict(future)
        pred = np.expm1(fc["yhat"].values.astype(float))
        pred = np.clip(pred, 0.0, None)
        if pred.shape[0] != horizon:
            pred = np.pad(pred, (0, horizon - pred.shape[0]), mode="edge")
        return pred[:horizon]
    except Exception:
        last = float(train.iloc[-1])
        return np.full(horizon, last, float)

# ----------------------------
# MAE utility
# ----------------------------
def mae(a, b) -> float:
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if not np.any(m):
        return np.nan
    return float(np.mean(np.abs(a[m] - b[m])))

# ----------------------------
# Evaluate ETS + Prophet on the regime cuts for one city
# ----------------------------
def eval_ets_prophet_for_city(city_label: str, df: pd.DataFrame, out_base: Path,
                             lookback_ets: int = 120,
                             lookback_prophet: int = 180) -> pd.DataFrame:
    """
    Evaluates ETS + Prophet for the fixed VALIDATION_CONFIG cuts and HORIZON_LIST.
    Saves CSV to: out_base / "extra_baselines" / city_label / "extra_baselines_metrics_<city>.csv"
    """
    outdir = Path(out_base) / "extra_baselines" / city_label
    outdir.mkdir(parents=True, exist_ok=True)

    series_7d = build_obs7_series(df)
    df_dates = pd.to_datetime(df["date"].values)
    date_min = df_dates.min()
    date_max = df_dates.max()

    rows = []
    for regime, cut_str, lb in VALIDATION_CONFIG:
        cut = pd.Timestamp(cut_str)

        # keep same inclusion rules you use in run_multi_window_eval (future must exist)
        if cut < date_min:
            continue
        for H in HORIZON_LIST:
            if cut > date_max - pd.Timedelta(days=int(H)):
                continue

            # Truth: future obs7 values for H days after cut (aligned to dates)
            # obs7 at day t corresponds to df date t, so future window is cut+1..cut+H
            fut_index = pd.date_range(cut + pd.Timedelta(days=1), periods=int(H), freq="D")
            y_true = series_7d.reindex(fut_index).values.astype(float)

            # Forecasts
            y_ets = forecast_ets(series_7d, cut, int(H), lookback_days=lookback_ets)
            y_prp = forecast_prophet(series_7d, cut, int(H), lookback_days=lookback_prophet)

            rows.append({
                "city": city_label,
                "regime": regime,
                "cut_date": str(cut.date()),
                "horizon": int(H),
                "mae_ets": mae(y_ets, y_true),
                "mae_prophet": mae(y_prp, y_true),
                "mean_true": float(np.nanmean(y_true)) if np.isfinite(y_true).any() else np.nan,
            })

    out_df = pd.DataFrame(rows)
    out_path = outdir / f"extra_baselines_metrics_{city_label}.csv"
    out_df.to_csv(out_path, index=False)
    print(f"[OK] Saved: {out_path}")
    return out_df

# ----------------------------
# Convenience runner: choose city, auto-load data with your existing loaders
# ----------------------------
def run_extra_baselines(city_label: str):
    # Try US city first
    match = [c for c in US_CITIES if c[0] == city_label]
    if match:
        _, county, state, sub = match[0]
        df, Npop = load_us_county_series(county, state, sub)
    else:
        df, Npop = load_world_city_series(city_label)

    out_base = Path(OUT_PATH_STR)
    res_df = eval_ets_prophet_for_city(city_label, df, out_base)
    display(res_df.head(10))
    print("\nQuick aggregate:")
    agg = res_df.groupby(["horizon"]).agg(
        n=("mae_ets", "count"),
        ets_mean_mae=("mae_ets", "mean"),
        prophet_mean_mae=("mae_prophet", "mean"),
        ets_median_mae=("mae_ets", "median"),
        prophet_median_mae=("mae_prophet", "median"),
    )
    display(agg)
    return res_df

# ----------------------------
# Example usage (run one city)
# ----------------------------
# extra_df = run_extra_baselines("SanDiego")

[OK] Baselines available? ETS=True | Prophet=True


In [32]:
extra_sd = run_extra_baselines("SanDiego")
extra_ny = run_extra_baselines("NewYork")
extra_ldn = run_extra_baselines("London")
extra_tky = run_extra_baselines("Tokyo")

[OK] Loaded SanDiego (US) | T=1265 | 2020-01-22 → 2023-07-23


19:06:39 - cmdstanpy - INFO - Chain [1] start processing
19:06:39 - cmdstanpy - INFO - Chain [1] done processing
19:06:39 - cmdstanpy - INFO - Chain [1] start processing
19:06:39 - cmdstanpy - INFO - Chain [1] done processing
19:06:39 - cmdstanpy - INFO - Chain [1] start processing
19:06:39 - cmdstanpy - INFO - Chain [1] done processing
19:06:39 - cmdstanpy - INFO - Chain [1] start processing
19:06:39 - cmdstanpy - INFO - Chain [1] done processing
19:06:39 - cmdstanpy - INFO - Chain [1] start processing
19:06:40 - cmdstanpy - INFO - Chain [1] done processing
19:06:40 - cmdstanpy - INFO - Chain [1] start processing
19:06:40 - cmdstanpy - INFO - Chain [1] done processing
19:06:40 - cmdstanpy - INFO - Chain [1] start processing
19:06:40 - cmdstanpy - INFO - Chain [1] done processing
19:06:40 - cmdstanpy - INFO - Chain [1] start processing
19:06:40 - cmdstanpy - INFO - Chain [1] done processing
19:06:40 - cmdstanpy - INFO - Chain [1] start processing
19:06:40 - cmdstanpy - INFO - Chain [1]

[OK] Saved: C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\CODES_04032021\extra_baselines\SanDiego\extra_baselines_metrics_SanDiego.csv


,city,regime,cut_date,horizon,mae_ets,mae_prophet,mean_true
0,SanDiego,FirstWave,2020-05-01,7,4.834731,63.556403,130.081633
1,SanDiego,FirstWave,2020-05-01,14,9.555853,97.312761,132.122449
2,SanDiego,Winter20,2021-01-15,7,163.319511,1029.896364,2392.591837
3,SanDiego,Winter20,2021-01-15,14,379.257853,1317.587979,2070.418367
4,SanDiego,Delta,2021-08-15,7,202.274749,144.168123,966.061224
5,SanDiego,Delta,2021-08-15,14,180.343524,526.232044,1146.020408
6,SanDiego,Omicron,2022-01-15,7,4599.441604,1777.129850,9598.204082
7,SanDiego,Omicron,2022-01-15,14,8017.377717,1635.386353,8404.163265
8,SanDiego,BA5_Waning,2022-06-15,7,504.690964,5189.670290,1432.244898
9,SanDiego,BA5_Waning,2022-06-15,14,515.585042,26054.874561,1399.204082



Quick aggregate:


,n,ets_mean_mae,prophet_mean_mae,ets_median_mae,prophet_median_mae
horizon,,,,,
7,5,1094.912312,1640.884206,202.274749,1029.896364
14,5,1820.423998,5926.278740,379.257853,1317.587979


KeyError: 'NewYork'

In [33]:
# ============================
# HARD OVERRIDE: use YOUR load_city_df routing always (fix NewYork KeyError)
# ============================

import numpy as np
import pandas as pd
from pathlib import Path

def load_city_df(label):
    # 1) Clean
    label_clean = str(label).strip().lower()

    # 2) Hardcode US routing
    hardcoded_us_map = {
        "sandiego":     ("San Diego County", "CA", "California"),
        "seattle":      ("King County", "WA", "Washington"),
        "newyork":      ("New York County", "NY", "New York"),
        "chicago":      ("Cook County", "IL", "Illinois"),
        "houston":      ("Harris County", "TX", "Texas"),
        "phoenix":      ("Maricopa County", "AZ", "Arizona"),
        "miami":        ("Miami-Dade County", "FL", "Florida"),
        "denver":       ("Denver County", "CO", "Colorado"),
        "losangeles":   ("Los Angeles County", "CA", "California"),
        "sanfrancisco": ("San Francisco County", "CA", "California"),
    }

    if label_clean in hardcoded_us_map:
        county, state, sub = hardcoded_us_map[label_clean]
        df, Npop = load_us_county_series(county, state, sub)
        return df, float(Npop), label  # keep original label for filenames

    # World routing: preserve canonical key
    intl_map = {k.lower(): k for k in CITY_SPECS.keys()}
    if label_clean in intl_map:
        city_key = intl_map[label_clean]
        df, Npop_alt = load_world_city_series(city_key)
        Npop = CITY_POP.get(city_key, np.nan)
        if not np.isfinite(Npop):
            Npop = float(Npop_alt)
        return df, float(Npop), city_key

    raise ValueError(f"Unknown city label: '{label}'. Check spelling.")

def run_extra_baselines(city_label: str):
    """
    Runs ETS + Prophet baselines using eval_ets_prophet_for_city(),
    but guarantees correct city loading (fixes NewYork -> world KeyError).
    """
    df, Npop, city_key = load_city_df(city_label)

    # Ensure date dtype
    if "date" in df.columns and not np.issubdtype(df["date"].dtype, np.datetime64):
        df["date"] = pd.to_datetime(df["date"])

    print(f"[OK] Loaded: {city_key} | Pop={int(Npop):,} | T={len(df)} | "
          f"{df['date'].min().date()} → {df['date'].max().date()}")

    out_base = Path(OUT_PATH_STR)
    out_base.mkdir(parents=True, exist_ok=True)

    # This function must already exist from your previous cell:
    # eval_ets_prophet_for_city(city_label, df, out_base)
    res_df = eval_ets_prophet_for_city(city_key, df, out_base)

    # Quick summary tables
    display(res_df.head(12))
    agg = res_df.groupby(["horizon"]).agg(
        n=("cut_date", "count"),
        ets_mean_mae=("mae_ets", "mean"),
        prophet_mean_mae=("mae_prophet", "mean"),
        ets_median_mae=("mae_ets", "median"),
        prophet_median_mae=("mae_prophet", "median"),
    ).reset_index()
    display(agg)

    return res_df

In [34]:
extra_sd  = run_extra_baselines("SanDiego")
extra_ny  = run_extra_baselines("NewYork")
extra_ldn = run_extra_baselines("London")
extra_tky = run_extra_baselines("Tokyo")

[OK] Loaded: SanDiego | Pop=3,338,330 | T=1265 | 2020-01-22 → 2023-07-23


19:12:12 - cmdstanpy - INFO - Chain [1] start processing
19:12:12 - cmdstanpy - INFO - Chain [1] done processing
19:12:13 - cmdstanpy - INFO - Chain [1] start processing
19:12:13 - cmdstanpy - INFO - Chain [1] done processing
19:12:13 - cmdstanpy - INFO - Chain [1] start processing
19:12:13 - cmdstanpy - INFO - Chain [1] done processing
19:12:13 - cmdstanpy - INFO - Chain [1] start processing
19:12:13 - cmdstanpy - INFO - Chain [1] done processing
19:12:14 - cmdstanpy - INFO - Chain [1] start processing
19:12:14 - cmdstanpy - INFO - Chain [1] done processing
19:12:14 - cmdstanpy - INFO - Chain [1] start processing
19:12:14 - cmdstanpy - INFO - Chain [1] done processing
19:12:14 - cmdstanpy - INFO - Chain [1] start processing
19:12:15 - cmdstanpy - INFO - Chain [1] done processing
19:12:15 - cmdstanpy - INFO - Chain [1] start processing
19:12:15 - cmdstanpy - INFO - Chain [1] done processing
19:12:15 - cmdstanpy - INFO - Chain [1] start processing
19:12:15 - cmdstanpy - INFO - Chain [1]

[OK] Saved: C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\CODES_04032021\extra_baselines\SanDiego\extra_baselines_metrics_SanDiego.csv


,city,regime,cut_date,horizon,mae_ets,mae_prophet,mean_true
0,SanDiego,FirstWave,2020-05-01,7,4.834731,63.556403,130.081633
1,SanDiego,FirstWave,2020-05-01,14,9.555853,97.312761,132.122449
2,SanDiego,Winter20,2021-01-15,7,163.319511,1029.896364,2392.591837
3,SanDiego,Winter20,2021-01-15,14,379.257853,1317.587979,2070.418367
4,SanDiego,Delta,2021-08-15,7,202.274749,144.168123,966.061224
5,SanDiego,Delta,2021-08-15,14,180.343524,526.232044,1146.020408
6,SanDiego,Omicron,2022-01-15,7,4599.441604,1777.129850,9598.204082
7,SanDiego,Omicron,2022-01-15,14,8017.377717,1635.386353,8404.163265
8,SanDiego,BA5_Waning,2022-06-15,7,504.690964,5189.670290,1432.244898
9,SanDiego,BA5_Waning,2022-06-15,14,515.585042,26054.874561,1399.204082


,horizon,n,ets_mean_mae,prophet_mean_mae,ets_median_mae,prophet_median_mae
0,7,5,1094.912312,1640.884206,202.274749,1029.896364
1,14,5,1820.423998,5926.278740,379.257853,1317.587979


...

    Loaded Tokyo: 857 days, 2020-05-09 → 2022-09-12
[OK] Loaded: Tokyo | Pop=14,000,000 | T=857 | 2020-05-09 → 2022-09-12


19:13:41 - cmdstanpy - INFO - Chain [1] done processing
19:13:41 - cmdstanpy - INFO - Chain [1] start processing
19:13:41 - cmdstanpy - INFO - Chain [1] done processing
19:13:41 - cmdstanpy - INFO - Chain [1] start processing
19:13:41 - cmdstanpy - INFO - Chain [1] done processing
19:13:41 - cmdstanpy - INFO - Chain [1] start processing
19:13:42 - cmdstanpy - INFO - Chain [1] done processing
19:13:42 - cmdstanpy - INFO - Chain [1] start processing
19:13:42 - cmdstanpy - INFO - Chain [1] done processing
19:13:42 - cmdstanpy - INFO - Chain [1] start processing
19:13:42 - cmdstanpy - INFO - Chain [1] done processing
19:13:42 - cmdstanpy - INFO - Chain [1] start processing
19:13:42 - cmdstanpy - INFO - Chain [1] done processing
19:13:43 - cmdstanpy - INFO - Chain [1] start processing
19:13:43 - cmdstanpy - INFO - Chain [1] done processing


[OK] Saved: C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\CODES_04032021\extra_baselines\Tokyo\extra_baselines_metrics_Tokyo.csv


,city,regime,cut_date,horizon,mae_ets,mae_prophet,mean_true
0,Tokyo,Winter20,2021-01-15,7,66.800800,713.849090,1522.142857
1,Tokyo,Winter20,2021-01-15,14,169.635239,894.065270,1312.683673
2,Tokyo,Delta,2021-08-15,7,262.421882,1348.321325,4782.979592
3,Tokyo,Delta,2021-08-15,14,294.632687,1682.218134,4595.357143
4,Tokyo,Omicron,2022-01-15,7,758.419462,3945.399710,5479.142857
5,Tokyo,Omicron,2022-01-15,14,2810.721717,7820.973433,8599.479592
6,Tokyo,BA5_Waning,2022-06-15,7,88.637115,282.010904,1630.020408
7,Tokyo,BA5_Waning,2022-06-15,14,320.557557,720.666536,1853.969388


,horizon,n,ets_mean_mae,prophet_mean_mae,ets_median_mae,prophet_median_mae
0,7,4,294.069815,1572.395257,175.529498,1031.085208
1,14,4,898.886800,2779.480843,307.595122,1288.141702


In [45]:
# =========================
# Extra baselines WITHOUT statsmodels ETS:
# Manual Exponential Smoothing (SES + damped Holt) + Prophet (optional)
# Uses your load_city_df() and your existing helpers:
#   VALIDATION_CONFIG, daily_from_cum_np, weekly_avg_np, carry_wavg_future_from_cum
# =========================

from pathlib import Path
import numpy as np
import pandas as pd

# Prophet (optional)
PROPHET_OK = False
try:
    from prophet import Prophet
    PROPHET_OK = True
except Exception as e:
    print("[WARN] Prophet not available:", repr(e))

def _prep_daily7_from_cum(cum):
    daily = daily_from_cum_np(np.asarray(cum, float))
    daily7 = weekly_avg_np(daily)
    return np.asarray(daily7, float)

def _slice_train_true(df, cut_date, lookback_days, horizon):
    cut_date = pd.Timestamp(cut_date)
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)

    # enough future truth?
    if cut_date + pd.Timedelta(days=horizon) > df["date"].max():
        return None

    train_start = cut_date - pd.Timedelta(days=int(lookback_days))
    df_tr = df[(df["date"] >= train_start) & (df["date"] <= cut_date)].reset_index(drop=True)
    if len(df_tr) < 21:
        return None

    df_full = df[df["date"] <= cut_date + pd.Timedelta(days=horizon)].reset_index(drop=True)
    cut_idx = df_full.index[df_full["date"] == cut_date]
    if len(cut_idx) == 0:
        return None
    cut_idx = int(cut_idx[0])

    obs_cum_hist = np.asarray(df_full["cases"][:cut_idx+1], float)
    true_cum_future = np.asarray(df_full["cases"][cut_idx+1:cut_idx+1+horizon], float)
    if len(true_cum_future) != horizon:
        return None

    true7 = carry_wavg_future_from_cum(obs_cum_hist, true_cum_future)
    if len(true7) != horizon:
        return None

    last_obs7 = float(_prep_daily7_from_cum(obs_cum_hist)[-1])
    return df_tr, obs_cum_hist, true7, last_obs7

# -------------------------
# Manual ETS-like methods
# -------------------------
def _ses_forecast(y, h, alpha):
    """Simple Exponential Smoothing forecast (level only)."""
    y = np.asarray(y, float)
    level = y[0]
    for t in range(1, len(y)):
        level = alpha * y[t] + (1 - alpha) * level
    return np.full(h, level, float)

def _holt_damped_forecast(y, h, alpha, beta, phi):
    """
    Damped Holt trend forecast (additive).
    phi in (0,1] damping; phi=1 is standard Holt.
    """
    y = np.asarray(y, float)
    level = y[0]
    trend = y[1] - y[0] if len(y) > 1 else 0.0
    for t in range(1, len(y)):
        prev_level = level
        level = alpha * y[t] + (1 - alpha) * (level + phi * trend)
        trend = beta * (level - prev_level) + (1 - beta) * (phi * trend)

    # multi-step damped trend sum
    # forecast = level + trend * (phi + phi^2 + ... + phi^h)
    if phi == 1.0:
        steps = np.arange(1, h+1, dtype=float)
        return level + trend * steps
    else:
        # geometric series
        steps = np.arange(1, h+1, dtype=float)
        geom = (phi * (1 - phi**steps)) / (1 - phi)
        return level + trend * geom

def _fit_forecast_manual_ets(obs_cum_hist, horizon):
    """
    Manual ETS-like baseline:
      - Fit SES and damped Holt by small grid search on 7d-average daily series
      - Choose the model with lower in-sample SSE
    Returns horizon-length forecast (7d-avg daily).
    """
    y = _prep_daily7_from_cum(obs_cum_hist)
    y = np.clip(y, 0, None)

    if len(y) < 21 or np.nanstd(y) < 1e-9:
        return np.full(horizon, float(y[-1]) if len(y) else 0.0)

    y_train = y.copy()

    # --- grid search SES ---
    ses_grid = np.linspace(0.05, 0.95, 19)
    best_ses = (np.inf, None)
    for a in ses_grid:
        fc_in = _ses_forecast(y_train[:-1], 1, a)[0]
        # one-step-ahead proxy SSE (fast approximation)
        # better: run through all points; but this is enough for ranking
        # We'll do full in-sample SSE with recursion:
        level = y_train[0]
        errs = []
        for t in range(1, len(y_train)):
            pred = level
            errs.append(y_train[t] - pred)
            level = a * y_train[t] + (1-a) * level
        sse = float(np.sum(np.square(errs)))
        if sse < best_ses[0]:
            best_ses = (sse, a)

    # --- grid search damped Holt ---
    a_grid = np.linspace(0.05, 0.95, 10)
    b_grid = np.linspace(0.05, 0.95, 10)
    phi_grid = [0.80, 0.90, 1.00]
    best_holt = (np.inf, None)
    for a in a_grid:
        for b in b_grid:
            for phi in phi_grid:
                # in-sample SSE using recursive 1-step prediction
                level = y_train[0]
                trend = y_train[1] - y_train[0] if len(y_train) > 1 else 0.0
                errs = []
                for t in range(1, len(y_train)):
                    pred = level + phi * trend
                    errs.append(y_train[t] - pred)
                    prev_level = level
                    level = a * y_train[t] + (1-a) * (level + phi * trend)
                    trend = b * (level - prev_level) + (1-b) * (phi * trend)
                sse = float(np.sum(np.square(errs)))
                if sse < best_holt[0]:
                    best_holt = (sse, (a, b, phi))

    # pick best
    if best_holt[0] < best_ses[0]:
        a, b, phi = best_holt[1]
        fc = _holt_damped_forecast(y_train, horizon, a, b, phi)
    else:
        a = best_ses[1]
        fc = _ses_forecast(y_train, horizon, a)

    fc = np.asarray(fc, float)
    fc[~np.isfinite(fc)] = float(y_train[-1])
    fc = np.clip(fc, 0, None)
    return fc

# -------------------------
# Prophet baseline (optional)
# -------------------------
def _fit_forecast_prophet(obs_cum_hist, horizon):
    y = _prep_daily7_from_cum(obs_cum_hist)
    y = np.asarray(y, float)
    if len(y) < 30 or np.nanstd(y) < 1e-9:
        return np.full(horizon, float(y[-1]) if len(y) else 0.0)

    ds = pd.date_range("2000-01-01", periods=len(y), freq="D")
    dfp = pd.DataFrame({"ds": ds, "y": np.clip(y, 0, None)})

    m = Prophet(
        growth="linear",
        daily_seasonality=False,
        weekly_seasonality=True,
        yearly_seasonality=False,
        changepoint_prior_scale=0.05,
        seasonality_prior_scale=1.0,
    )
    m.fit(dfp)

    fut = m.make_future_dataframe(periods=horizon, freq="D", include_history=False)
    pred = m.predict(fut)
    fc = np.asarray(pred["yhat"], float)

    fc[~np.isfinite(fc)] = dfp["y"].iloc[-1]
    cap = np.nanpercentile(dfp["y"], 99) * 5.0 + 1.0
    fc = np.clip(fc, 0, cap)
    return fc

# -------------------------
# Main evaluation runner
# -------------------------
# --- Robust loader wrapper: works if load_city_df returns (df,Npop) or (df,Npop,...) ---
def _load_city_df_2(label):
    out = load_city_df(label)
    if isinstance(out, tuple) or isinstance(out, list):
        if len(out) >= 2:
            return out[0], float(out[1])
        raise ValueError(f"load_city_df('{label}') returned a tuple/list with <2 elements: {out}")
    raise ValueError(f"load_city_df('{label}') did not return a tuple/list. Got: {type(out)}")

# Quick test:
df_test, Npop_test = _load_city_df_2("SanDiego")
print("OK loader:", df_test.shape, "Npop:", Npop_test)
def eval_manualets_prophet_for_city(city_label, horizons=(7,14), include_prophet=True):
    df, Npop = _load_city_df_2(city_label)   # <-- your robust router
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)

    rows = []
    for (regime, cut_str, lookback) in VALIDATION_CONFIG:
        for H in horizons:
            sliced = _slice_train_true(df, cut_str, lookback, H)
            if sliced is None:
                continue
            df_tr, obs_cum_hist, true7, last_obs7 = sliced

            ets_fc = _fit_forecast_manual_ets(obs_cum_hist, H)
            row = {
                "city": city_label,
                "regime": regime,
                "cut_date": str(pd.Timestamp(cut_str).date()),
                "horizon": int(H),
                "mae_ets_manual": float(np.mean(np.abs(ets_fc - true7))),
                "mean_true": float(np.mean(true7)),
            }

            if include_prophet and PROPHET_OK:
                try:
                    pr_fc = _fit_forecast_prophet(obs_cum_hist, H)
                    row["mae_prophet"] = float(np.mean(np.abs(pr_fc - true7)))
                except Exception:
                    row["mae_prophet"] = np.nan

            rows.append(row)

    return pd.DataFrame(rows)

def run_extra_baselines_all(cities, out_dir=None, horizons=(7,14), include_prophet=True):
    out_dir = Path(out_dir) if out_dir else Path(OUT_PATH_STR)
    out_dir.mkdir(parents=True, exist_ok=True)

    all_df = []
    for city in cities:
        print(f"\n=== Extra baselines (manual ETS{'+Prophet' if include_prophet else ''}): {city} ===")
        res = eval_manualets_prophet_for_city(city, horizons=horizons, include_prophet=include_prophet)
        if len(res) == 0:
            print("[WARN] No rows produced (insufficient data around cuts?)")
            continue
        out_csv = out_dir / f"extra_baselines_metrics_{city}.csv"
        res.to_csv(out_csv, index=False)
        print("Saved:", out_csv)
        all_df.append(res)

    if not all_df:
        raise RuntimeError("No cities produced results.")
    all_df = pd.concat(all_df, ignore_index=True)
    out_all = out_dir / "extra_baselines_metrics_ALL.csv"
    all_df.to_csv(out_all, index=False)
    print("\nSaved combined:", out_all)
    return all_df

# -------------------------
# Suggested run: all 18 cities
# -------------------------
cities18 = ["SanDiego","Seattle","NewYork","Chicago","Houston","Phoenix","Miami","Denver","LosAngeles","SanFrancisco",
            "London","SaoPaulo","Rome","Paris","Tokyo","Sydney","Berlin","Moscow"]

# Start with ETS-only (most stable). Turn Prophet on after you see it behaves.
extra_all = run_extra_baselines_all(cities18, include_prophet=False)

display(extra_all.head())
print("\nDone. If you want Prophet too, rerun with include_prophet=True (Prophet can be unstable in some regimes).")

OK loader: (1265, 4) Npop: 3338330.0

=== Extra baselines (manual ETS): SanDiego ===
Saved: C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\CODES_04032021\extra_baselines_metrics_SanDiego.csv

=== Extra baselines (manual ETS): Seattle ===
Saved: C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\CODES_04032021\extra_baselines_metrics_Seattle.csv

=== Extra baselines (manual ETS): NewYork ===
Saved: C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\CODES_04032021\extra_baselines_metrics_NewYork.csv

=== Extra baselines (manual ETS): Chicago ===
Saved: C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\CODES_04032021\extra_baselines_metrics_Chicago.csv

=== Extra baselines (manual ETS): Houston ===
Saved: C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\CODES_04032021\extra_baselines_metrics_Houston.csv

=== Extra baselines (manual ETS): Phoenix ===
Saved: C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\CODES_04032021\extra_baselines_metrics_Phoenix.csv



,city,regime,cut_date,horizon,mae_ets_manual,mean_true
0,SanDiego,FirstWave,2020-05-01,7,2.022245,130.081633
1,SanDiego,FirstWave,2020-05-01,14,2.130585,132.122449
2,SanDiego,Winter20,2021-01-15,7,176.291333,2392.591837
3,SanDiego,Winter20,2021-01-15,14,406.321217,2070.418367
4,SanDiego,Delta,2021-08-15,7,197.788030,966.061224



Done. If you want Prophet too, rerun with include_prophet=True (Prophet can be unstable in some regimes).


In [41]:
# ---- Prophet debug + robust evaluation cell ----
import numpy as np
import pandas as pd

def _make_train_series_7d(df, cut_date, lookback_days):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    cut_date = pd.Timestamp(cut_date)
    start = cut_date - pd.Timedelta(days=lookback_days)
    dsub = df[(df["date"] >= start) & (df["date"] <= cut_date)].sort_values("date").reset_index(drop=True)

    # observed daily from cumulative cases (nonnegative)
    cum = dsub["cases"].to_numpy(dtype=float)
    daily = daily_from_cum_np(cum)
    daily7 = weekly_avg_np(daily, k=7)

    # align dates: daily_from_cum_np returns same length as cum, but first entry is 0-ish
    dates = dsub["date"].to_numpy()
    y = np.asarray(daily7, float)
    return pd.DataFrame({"ds": pd.to_datetime(dates), "y": y})

def _prophet_forecast(train_df, horizon, cps=0.05):
    # Prophet import inside to respect your environment
    from prophet import Prophet

    # Basic sanitization
    tr = train_df.copy()
    tr = tr.replace([np.inf, -np.inf], np.nan).dropna()
    tr["y"] = tr["y"].astype(float)

    # If too short or degenerate, skip
    if len(tr) < 30 or np.nanstd(tr["y"].values) < 1e-9:
        raise RuntimeError(f"Insufficient/degenerate training data for Prophet (n={len(tr)})")

    m = Prophet(
        growth="linear",
        seasonality_mode="additive",
        changepoint_prior_scale=float(cps),
        weekly_seasonality=False,
        daily_seasonality=False,
        yearly_seasonality=False,
    )
    m.fit(tr)

    future = m.make_future_dataframe(periods=int(horizon), freq="D", include_history=False)
    fcst = m.predict(future)
    yhat = fcst["yhat"].to_numpy(dtype=float)
    yhat = np.clip(yhat, 0.0, None)  # no negative daily cases
    return yhat

def eval_prophet_debug_for_city(city_label, horizons=(7,14), cps=0.05):
    df, Npop = _load_city_df_2(city_label)
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)

    out_rows = []
    for regime, cut_str, lookback_days in VALIDATION_CONFIG:
        cut = pd.Timestamp(cut_str)

        # need future truth
        max_h = max(horizons)
        if cut > df["date"].max() - pd.Timedelta(days=max_h):
            continue
        if cut - pd.Timedelta(days=lookback_days) < df["date"].min():
            continue

        # Build training set (7d-smoothed daily)
        train_df = _make_train_series_7d(df, cut, lookback_days)

        # Truth (future 7d-smoothed daily) over horizon:
        # Use the FULL df (not dsub), so it matches your earlier eval style
        idx_cut = np.where(df["date"].to_numpy() <= cut.to_datetime64())[0][-1]
        cum_full = df["cases"].to_numpy(dtype=float)
        daily_full = daily_from_cum_np(cum_full)
        daily7_full = weekly_avg_np(daily_full, k=7)

        for H in horizons:
            true_segment = daily7_full[idx_cut+1: idx_cut+1+H]
            if len(true_segment) != H:
                continue
            mean_true = float(np.mean(true_segment))

            row = {
                "city": city_label,
                "regime": regime,
                "cut_date": str(cut.date()),
                "horizon": int(H),
                "mean_true": mean_true,
            }

            # Prophet
            try:
                yhat = _prophet_forecast(train_df, H, cps=cps)
                row["mae_prophet"] = float(np.mean(np.abs(yhat - true_segment)))
                row["prophet_ok"] = True
                row["prophet_error"] = ""
            except Exception as e:
                row["mae_prophet"] = np.nan
                row["prophet_ok"] = False
                row["prophet_error"] = repr(e)

            out_rows.append(row)

    return pd.DataFrame(out_rows)

# Example:
prop_sd = eval_prophet_debug_for_city("SanDiego", horizons=(7,14), cps=0.05)
display(prop_sd)
print(prop_sd[["regime","cut_date","horizon","prophet_ok","prophet_error"]])

20:09:46 - cmdstanpy - INFO - Chain [1] start processing
20:09:46 - cmdstanpy - INFO - Chain [1] done processing
20:09:46 - cmdstanpy - INFO - Chain [1] start processing
20:09:46 - cmdstanpy - INFO - Chain [1] done processing
20:09:46 - cmdstanpy - INFO - Chain [1] start processing
20:09:46 - cmdstanpy - INFO - Chain [1] done processing
20:09:46 - cmdstanpy - INFO - Chain [1] start processing
20:09:46 - cmdstanpy - INFO - Chain [1] done processing
20:09:46 - cmdstanpy - INFO - Chain [1] start processing
20:09:46 - cmdstanpy - INFO - Chain [1] done processing
20:09:47 - cmdstanpy - INFO - Chain [1] start processing
20:09:47 - cmdstanpy - INFO - Chain [1] done processing
20:09:47 - cmdstanpy - INFO - Chain [1] start processing
20:09:47 - cmdstanpy - INFO - Chain [1] done processing
20:09:47 - cmdstanpy - INFO - Chain [1] start processing
20:09:47 - cmdstanpy - INFO - Chain [1] done processing
20:09:47 - cmdstanpy - INFO - Chain [1] start processing
20:09:47 - cmdstanpy - INFO - Chain [1]

,city,regime,cut_date,horizon,mean_true,mae_prophet,prophet_ok,prophet_error
0,SanDiego,FirstWave,2020-05-01,7,130.081633,5.810043,True,
1,SanDiego,FirstWave,2020-05-01,14,132.122449,10.295888,True,
2,SanDiego,Winter20,2021-01-15,7,2392.591837,1445.529105,True,
3,SanDiego,Winter20,2021-01-15,14,2070.418367,1904.405706,True,
4,SanDiego,Delta,2021-08-15,7,966.061224,137.106847,True,
5,SanDiego,Delta,2021-08-15,14,1146.020408,145.335059,True,
6,SanDiego,Omicron,2022-01-15,7,9598.204082,1610.592968,True,
7,SanDiego,Omicron,2022-01-15,14,8404.163265,3414.922227,True,
8,SanDiego,BA5_Waning,2022-06-15,7,1432.244898,300.812403,True,
9,SanDiego,BA5_Waning,2022-06-15,14,1399.204082,254.990048,True,


       regime    cut_date  horizon  prophet_ok prophet_error
0   FirstWave  2020-05-01        7        True              
1   FirstWave  2020-05-01       14        True              
2    Winter20  2021-01-15        7        True              
3    Winter20  2021-01-15       14        True              
4       Delta  2021-08-15        7        True              
5       Delta  2021-08-15       14        True              
6     Omicron  2022-01-15        7        True              
7     Omicron  2022-01-15       14        True              
8  BA5_Waning  2022-06-15        7        True              
9  BA5_Waning  2022-06-15       14        True              


In [44]:
from pathlib import Path
import numpy as np
import pandas as pd
import time

# -------------------------
# Manual ETS (statsmodels-only fallback)
# -------------------------
def _ets_manual_forecast(train_y, horizon):
    """
    Minimal ETS using statsmodels ExponentialSmoothing (non-seasonal).
    train_y: 1D np array of 7d-smoothed daily cases
    """
    from statsmodels.tsa.holtwinters import ExponentialSmoothing

    y = np.asarray(train_y, float)
    y = y[np.isfinite(y)]
    if len(y) < 20 or np.nanstd(y) < 1e-9:
        raise RuntimeError(f"ETS insufficient/degenerate (n={len(y)})")

    # non-seasonal ETS; additive trend tends to behave better on smoothed counts
    mod = ExponentialSmoothing(
        y,
        trend="add",
        seasonal=None,
        initialization_method="estimated",
    )
    fit = mod.fit(optimized=True)
    fcst = fit.forecast(int(horizon))
    fcst = np.asarray(fcst, float)
    fcst = np.clip(fcst, 0.0, None)
    if len(fcst) != int(horizon):
        fcst = np.pad(fcst, (0, int(horizon)-len(fcst)), mode="edge")
    return fcst

def eval_prophet_ets_for_city(city_label, horizons=(7,14), cps=0.05, do_ets=True):
    df, Npop = _load_city_df_2(city_label)
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)

    # full truth series (7d-smoothed daily)
    cum_full = df["cases"].to_numpy(dtype=float)
    daily_full = daily_from_cum_np(cum_full)
    daily7_full = weekly_avg_np(daily_full, k=7)

    out_rows = []
    for regime, cut_str, lookback_days in VALIDATION_CONFIG:
        cut = pd.Timestamp(cut_str)
        max_h = max(horizons)

        if cut > df["date"].max() - pd.Timedelta(days=max_h):
            continue
        if cut - pd.Timedelta(days=lookback_days) < df["date"].min():
            continue

        # train df for Prophet
        train_df = _make_train_series_7d(df, cut, lookback_days)

        # index for truth slicing
        idx_cut = np.where(df["date"].to_numpy() <= cut.to_datetime64())[0][-1]

        for H in horizons:
            true_segment = daily7_full[idx_cut+1: idx_cut+1+H]
            if len(true_segment) != H:
                continue

            row = {
                "city": city_label,
                "regime": regime,
                "cut_date": str(cut.date()),
                "horizon": int(H),
                "mean_true": float(np.mean(true_segment)),
            }

            # Prophet
            try:
                yhat_p = _prophet_forecast(train_df, H, cps=cps)
                row["mae_prophet"] = float(np.mean(np.abs(yhat_p - true_segment)))
                row["prophet_ok"] = True
                row["prophet_error"] = ""
            except Exception as e:
                row["mae_prophet"] = np.nan
                row["prophet_ok"] = False
                row["prophet_error"] = repr(e)

            # ETS-manual
            if do_ets:
                try:
                    # ETS uses same training y (7d smoothed)
                    y_train = train_df["y"].to_numpy(dtype=float)
                    yhat_e = _ets_manual_forecast(y_train, H)
                    row["mae_ets_manual"] = float(np.mean(np.abs(yhat_e - true_segment)))
                    row["ets_ok"] = True
                    row["ets_error"] = ""
                except Exception as e:
                    row["mae_ets_manual"] = np.nan
                    row["ets_ok"] = False
                    row["ets_error"] = repr(e)

            out_rows.append(row)

    return pd.DataFrame(out_rows)

def run_prophet_ets_all(cities, horizons=(7,14), cps=0.05, do_ets=True, out_dir=None):
    out_dir = Path(out_dir) if out_dir else Path(OUT_PATH_STR)
    out_dir.mkdir(parents=True, exist_ok=True)

    all_df = []
    for city in cities:
        print(f"\n=== Prophet/ETS: {city} ===")
        t0 = time.time()
        df_city = eval_prophet_ets_for_city(city, horizons=horizons, cps=cps, do_ets=do_ets)
        print(f"  rows={len(df_city)}  time={time.time()-t0:.1f}s")
        if len(df_city):
            df_city.to_csv(out_dir / f"extra_baselines_metrics_{city}.csv", index=False)
        all_df.append(df_city)

    all_df = pd.concat(all_df, ignore_index=True) if len(all_df) else pd.DataFrame()
    all_df.to_csv(out_dir / "extra_baselines_metrics_ALL.csv", index=False)
    print(f"\nSaved combined: {out_dir / 'extra_baselines_metrics_ALL.csv'}")
    return all_df

# -------------------------
# Run for all 18 cities
# -------------------------
cities18 = ["SanDiego","Seattle","NewYork","Chicago","Houston","Phoenix","Miami","Denver","LosAngeles","SanFrancisco",
            "London","SaoPaulo","Rome","Paris","Tokyo","Sydney","Berlin","Moscow"]

extra_all = run_prophet_ets_all(cities18, horizons=(7,14), cps=0.05, do_ets=True,
                                out_dir=r"C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\CODES_04032021")

display(extra_all.head())


=== Prophet/ETS: SanDiego ===


20:28:41 - cmdstanpy - INFO - Chain [1] start processing
20:28:41 - cmdstanpy - INFO - Chain [1] done processing
20:28:41 - cmdstanpy - INFO - Chain [1] start processing
20:28:42 - cmdstanpy - INFO - Chain [1] done processing
20:28:42 - cmdstanpy - INFO - Chain [1] start processing
20:28:42 - cmdstanpy - INFO - Chain [1] done processing
20:28:42 - cmdstanpy - INFO - Chain [1] start processing
20:28:42 - cmdstanpy - INFO - Chain [1] done processing
20:28:42 - cmdstanpy - INFO - Chain [1] start processing
20:28:42 - cmdstanpy - INFO - Chain [1] done processing
20:28:42 - cmdstanpy - INFO - Chain [1] start processing
20:28:42 - cmdstanpy - INFO - Chain [1] done processing
20:28:42 - cmdstanpy - INFO - Chain [1] start processing
20:28:42 - cmdstanpy - INFO - Chain [1] done processing
20:28:43 - cmdstanpy - INFO - Chain [1] start processing
20:28:43 - cmdstanpy - INFO - Chain [1] done processing
20:28:43 - cmdstanpy - INFO - Chain [1] start processing
20:28:43 - cmdstanpy - INFO - Chain [1]

  rows=10  time=9.0s

=== Prophet/ETS: Seattle ===


20:28:49 - cmdstanpy - INFO - Chain [1] start processing
20:28:49 - cmdstanpy - INFO - Chain [1] done processing
20:28:49 - cmdstanpy - INFO - Chain [1] start processing
20:28:50 - cmdstanpy - INFO - Chain [1] done processing
20:28:50 - cmdstanpy - INFO - Chain [1] start processing
20:28:50 - cmdstanpy - INFO - Chain [1] done processing
20:28:50 - cmdstanpy - INFO - Chain [1] start processing
20:28:50 - cmdstanpy - INFO - Chain [1] done processing
20:28:50 - cmdstanpy - INFO - Chain [1] start processing
20:28:50 - cmdstanpy - INFO - Chain [1] done processing
20:28:51 - cmdstanpy - INFO - Chain [1] start processing
20:28:51 - cmdstanpy - INFO - Chain [1] done processing
20:28:51 - cmdstanpy - INFO - Chain [1] start processing
20:28:51 - cmdstanpy - INFO - Chain [1] done processing
20:28:51 - cmdstanpy - INFO - Chain [1] start processing
20:28:51 - cmdstanpy - INFO - Chain [1] done processing
20:28:51 - cmdstanpy - INFO - Chain [1] start processing
20:28:51 - cmdstanpy - INFO - Chain [1]

  rows=10  time=8.7s

=== Prophet/ETS: NewYork ===


...

20:35:47 - cmdstanpy - INFO - Chain [1] start processing


    Loaded Moscow: 706 days, 2020-03-22 → 2022-09-15


20:35:47 - cmdstanpy - INFO - Chain [1] done processing
20:35:47 - cmdstanpy - INFO - Chain [1] start processing
20:35:47 - cmdstanpy - INFO - Chain [1] done processing
20:35:48 - cmdstanpy - INFO - Chain [1] start processing
20:35:48 - cmdstanpy - INFO - Chain [1] done processing
20:35:48 - cmdstanpy - INFO - Chain [1] start processing
20:35:48 - cmdstanpy - INFO - Chain [1] done processing
20:35:48 - cmdstanpy - INFO - Chain [1] start processing
20:35:48 - cmdstanpy - INFO - Chain [1] done processing
20:35:48 - cmdstanpy - INFO - Chain [1] start processing
20:35:49 - cmdstanpy - INFO - Chain [1] done processing


  rows=8  time=39.1s

Saved combined: C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\CODES_04032021\extra_baselines_metrics_ALL.csv


,city,regime,cut_date,horizon,mean_true,mae_prophet,prophet_ok,prophet_error,mae_ets_manual,ets_ok,ets_error
0,SanDiego,FirstWave,2020-05-01,7,130.081633,5.810043,True,,NaN,False,"TypeError(""deprecate_kwarg() missing 1 require..."
1,SanDiego,FirstWave,2020-05-01,14,132.122449,10.295888,True,,NaN,False,"TypeError(""deprecate_kwarg() missing 1 require..."
2,SanDiego,Winter20,2021-01-15,7,2392.591837,1445.529105,True,,NaN,False,"TypeError(""deprecate_kwarg() missing 1 require..."
3,SanDiego,Winter20,2021-01-15,14,2070.418367,1904.405706,True,,NaN,False,"TypeError(""deprecate_kwarg() missing 1 require..."
4,SanDiego,Delta,2021-08-15,7,966.061224,137.106847,True,,NaN,False,"TypeError(""deprecate_kwarg() missing 1 require..."


In [46]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import wilcoxon, binomtest

# =========================
# INPUT / OUTPUT
# =========================
MASTER = Path(r"C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\MASTER_regime_validation_all_rows_complete_02252026.csv")
OUTDIR = Path(r"C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\natcomm_master_analysis_outputs")
OUTDIR.mkdir(parents=True, exist_ok=True)

# =========================
# LOAD
# =========================
df = pd.read_csv(MASTER)
df["cut_date"] = pd.to_datetime(df["cut_date"], errors="coerce")
df["horizon"]  = pd.to_numeric(df["horizon"], errors="coerce")
df = df.dropna(subset=["city","window","cut_date","horizon"]).copy()
df["horizon"] = df["horizon"].astype(int)

models = {
    "PINN":"mae_pin",
    "ARIMA":"mae_ari",
    "LogLinear":"mae_loglin",
    "BayesPoly":"mae_bayesian",
    "ETS":"mae_ets_manual",
    "Prophet":"mae_prophet",
}
for col in models.values():
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
df["mean_true"] = pd.to_numeric(df["mean_true"], errors="coerce")

# =========================
# HELPERS
# =========================
def holm_adj(p):
    p = np.asarray(p, float)
    m = len(p)
    order = np.argsort(p)
    ps = p[order]
    adj = np.maximum.accumulate((m - np.arange(m)) * ps)
    adj = np.clip(adj, 0, 1)
    out = np.empty(m)
    out[order] = adj
    return out

# Stabilize scale-normalization for early-wave near-zero series
den = np.maximum(df["mean_true"].values.astype(float), 1.0)

for name, col in models.items():
    if col in df.columns:
        df[f"rel_{name}"] = df[col] / den

for name, col in models.items():
    if name == "ARIMA" or col not in df.columns:
        continue
    df[f"imp_{name}_vs_ARIMA"] = (df["mae_ari"] - df[col]) / df["mae_ari"]
    df[f"win_{name}"] = (df[col] < df["mae_ari"]).astype(int)

# =========================
# TABLE 1: OVERALL BY HORIZON (includes ETS/Prophet if present)
# =========================
rows=[]
for h in sorted(df["horizon"].unique()):
    sub = df[df["horizon"]==h]
    for name,col in models.items():
        if col not in sub.columns:
            continue
        x = sub[col].dropna()
        if len(x)==0:
            continue
        row = {
            "horizon":h,
            "model":name,
            "n":int(len(x)),
            "MAE_mean":float(x.mean()),
            "MAE_median":float(x.median()),
            "relMAE_mean":float(sub[f"rel_{name}"].dropna().mean()) if f"rel_{name}" in sub.columns else np.nan,
            "relMAE_median":float(sub[f"rel_{name}"].dropna().median()) if f"rel_{name}" in sub.columns else np.nan,
        }
        if name != "ARIMA" and f"win_{name}" in sub.columns:
            w = sub.loc[x.index, f"win_{name}"].dropna()
            row["wins_vs_ARIMA"] = int(w.sum()) if len(w) else np.nan
            row["win_rate_vs_ARIMA"] = float(w.mean()) if len(w) else np.nan
        rows.append(row)

table_overall = pd.DataFrame(rows).sort_values(["horizon","model"])
table_overall.to_csv(OUTDIR/"table_overall_by_horizon.csv", index=False)
print("Wrote:", OUTDIR/"table_overall_by_horizon.csv")

# =========================
# TABLE 2: REGIME P-VALUES (PINN vs ARIMA) + HOLM
# =========================
pv=[]
for h in sorted(df["horizon"].unique()):
    for reg in sorted(df["window"].unique()):
        sub = df[(df.horizon==h)&(df.window==reg)].dropna(subset=["mae_pin","mae_ari"])
        if len(sub) < 5:
            continue
        dif = (sub["mae_pin"] - sub["mae_ari"]).values
        try:
            p_w = float(wilcoxon(dif, alternative="two-sided").pvalue)
        except Exception:
            p_w = np.nan
        wins = int((sub["mae_pin"] < sub["mae_ari"]).sum())
        n = int(len(sub))
        p_s = float(binomtest(wins, n, 0.5, alternative="two-sided").pvalue)
        pv.append({
            "horizon":h, "regime":reg, "n":n,
            "wins_PINN":wins, "win_rate_PINN":wins/n,
            "mean_improvement_vs_ARIMA":float(((sub["mae_ari"]-sub["mae_pin"])/sub["mae_ari"]).mean()),
            "p_wilcoxon":p_w, "p_sign":p_s
        })

pv = pd.DataFrame(pv)
pv["p_wilcoxon_holm"] = np.nan
for h in pv["horizon"].unique():
    idx = pv.index[pv["horizon"]==h]
    pv.loc[idx, "p_wilcoxon_holm"] = holm_adj(pv.loc[idx, "p_wilcoxon"].values)

pv = pv.sort_values(["horizon","regime"])
pv.to_csv(OUTDIR/"table_regime_pvals_holm_PINN_vs_ARIMA.csv", index=False)
print("Wrote:", OUTDIR/"table_regime_pvals_holm_PINN_vs_ARIMA.csv")

# =========================
# FIGURE: IMPROVEMENT BOXPLOTS BY REGIME (PINN vs ARIMA)
# =========================
canonical = ["FirstWave","Winter20","Delta","Omicron","BA5_Waning"]

for h in sorted(df["horizon"].unique()):
    sub = df[df.horizon==h].dropna(subset=["mae_pin","mae_ari"]).copy()
    sub["imp"] = (sub["mae_ari"] - sub["mae_pin"]) / sub["mae_ari"]
    regimes = [r for r in canonical if r in sub["window"].unique()] + [r for r in sub["window"].unique() if r not in canonical]
    data = [sub.loc[sub.window==r, "imp"].dropna().values for r in regimes]

    plt.figure(figsize=(9,4))
    plt.boxplot(data, labels=regimes, showfliers=False)
    plt.axhline(0, linestyle="--")
    plt.ylabel("Improvement vs ARIMA (fraction)")
    plt.title(f"PINN improvement vs ARIMA by regime (h={h}d)")
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    outp = OUTDIR/f"fig_pinn_improvement_boxplot_{h}d.png"
    plt.savefig(outp, dpi=220)
    plt.close()
    print("Wrote:", outp)

# =========================
# FIGURE: CITY BARS (Figure-6-like) for 7-day horizon (PINN vs ARIMA)
# =========================
h = 7
sub = df[df.horizon==h].dropna(subset=["mae_pin","mae_ari"]).copy()
city_mean = sub.groupby("city")[["mae_pin","mae_ari"]].mean()
city_mean["imp"] = (city_mean["mae_ari"]-city_mean["mae_pin"])/city_mean["mae_ari"]
city_mean = city_mean.sort_values("imp", ascending=False)

plt.figure(figsize=(10,6))
x = np.arange(len(city_mean))
w = 0.38
plt.bar(x-w/2, city_mean["mae_pin"].values, w, label="PINN")
plt.bar(x+w/2, city_mean["mae_ari"].values, w, label="ARIMA")
plt.xticks(x, city_mean.index, rotation=45, ha="right")
plt.ylabel("MAE (cases/day)")
plt.title("City-level mean MAE across regimes (7-day horizon)")
plt.legend()
plt.tight_layout()
outp = OUTDIR/"fig_city_bars_pinn_vs_arima_7d.png"
plt.savefig(outp, dpi=220)
plt.close()
print("Wrote:", outp)

# Save enriched master (handy for later tables/plots)
df.to_csv(OUTDIR/"MASTER_with_derived.csv", index=False)
print("Wrote:", OUTDIR/"MASTER_with_derived.csv")

print("\nDone. Outputs in:", OUTDIR)

Wrote: C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\natcomm_master_analysis_outputs\table_overall_by_horizon.csv
Wrote: C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\natcomm_master_analysis_outputs\table_regime_pvals_holm_PINN_vs_ARIMA.csv


C:\Users\osmar\AppData\Local\Temp\ipykernel_1740\1383743618.py:139: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data, labels=regimes, showfliers=False)


Wrote: C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\natcomm_master_analysis_outputs\fig_pinn_improvement_boxplot_7d.png


C:\Users\osmar\AppData\Local\Temp\ipykernel_1740\1383743618.py:139: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data, labels=regimes, showfliers=False)


Wrote: C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\natcomm_master_analysis_outputs\fig_pinn_improvement_boxplot_14d.png
Wrote: C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\natcomm_master_analysis_outputs\fig_city_bars_pinn_vs_arima_7d.png
Wrote: C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\natcomm_master_analysis_outputs\MASTER_with_derived.csv

Done. Outputs in: C:\Users\osmar\OneDrive\Documents\PESQUISAS\COVID_PINN\natcomm_master_analysis_outputs
